In [ ]:
!nvidia-smi

Thu May  8 22:22:20 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   55C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p /content/RIPEMPHANTOM-T5/checkpoints
!cp -r /content/drive/MyDrive/RIPEMPHANTOM-T5/checkpoints/* /content/RIPEMPHANTOM-T5/checkpoints/
!cp /content/drive/MyDrive/RIPEMPHANTOM-T5/addresses.txt /content/RIPEMPHANTOM-T5/addresses.txt

cp: cannot stat '/content/drive/MyDrive/RIPEMPHANTOM-T5/addresses.txt': No such file or directory


In [ ]:
import torch
import cupy
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))
print(cupy.__version__, cupy.cuda.runtime.getDeviceCount(), cupy.cuda.runtime.getDeviceProperties(0)['name'])

2.0.1+cu118 True NVIDIA L4
13.3.0 1 b'NVIDIA L4'


In [ ]:
# === RIPEMPHANTOM-T5 ===
# Automated RIPEMD-160 Entropy Anomaly Exploit for Bitcoin Private Key Recovery
# Optimized for Google Colab with NVIDIA L4 GPU, Python 3.10, CUDA 12.x

# -- Automated Dependency Installation --
import subprocess
import sys
import os
import signal
import time
import random
import json
import hashlib
import base58
import math

def install_packages():
    """Install required packages"""
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
    except subprocess.CalledProcessError as e:
        print(f"Failed to upgrade pip: {e}")

    packages = [
        "base58", "pycryptodome", "ecdsa", "tqdm", "matplotlib",
        "scipy", "pandas", "seaborn", "statsmodels", "scikit-learn", "umap-learn",
        "numba", "cupy-cuda12x"
    ]
    for pkg in packages:
        try:
            __import__(pkg.replace("-cuda12x", ""))
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

install_packages()

# -- Core Imports --
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.mplot3d import Axes3D
from ecdsa import SigningKey, SECP256k1
from Crypto.Hash import RIPEMD160
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from statsmodels.stats.proportion import proportion_confint
from sklearn.cluster import DBSCAN
from sklearn.manifold import TSNE
import umap
from numba import jit
import cupy as cp
import logging
import threading
from concurrent.futures import ThreadPoolExecutor

# -- Prepare Workspace --
BASE_DIR = "/content/RIPEMPHANTOM-T5"
DRIVE_DIR = "/content/drive/MyDrive/RIPEMPHANTOM-T5"
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "logs"), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "figures"), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "reports"), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "datasets"), exist_ok=True)

# -- Setup Logging --
logging.basicConfig(
    filename=os.path.join(BASE_DIR, "logs", "recovery.log"),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# -- Detect GPU --
try:
    if torch.cuda.is_available():
        device = torch.device("cuda:0")  # Colab assigns L4 as cuda:0
        torch.cuda.set_device(0)
        logging.info(f"GPU detected: {torch.cuda.get_device_name(0)}, CUDA version: {torch.version.cuda}, "
                     f"CuPy devices: {cp.cuda.runtime.getDeviceCount()}, Selected device: cuda:0")
        print(f"Using GPU: {torch.cuda.get_device_name(0)}")
        !nvidia-smi
    else:
        device = torch.device("cpu")
        logging.info("No GPU detected, using CPU with numba JIT")
        print("No GPU detected, using CPU")
except Exception as e:
    device = torch.device("cpu")
    logging.warning(f"GPU detection failed: {e}, using CPU")
    print(f"GPU detection failed: {e}, using CPU")

# -- Constants --
MAX_K = 2**61
ADDR_FILE = os.path.join(BASE_DIR, "addresses.txt")
CHECKPOINT_PATH = os.path.join(BASE_DIR, "checkpoints")
DRIVE_CHECKPOINT_PATH = os.path.join(DRIVE_DIR, "checkpoints")
LOG_PATH = os.path.join(BASE_DIR, "logs", "telemetry.jsonl")
REPORT_PATH = os.path.join(BASE_DIR, "reports")
RECOVERED_KEYS_PATH = os.path.join(BASE_DIR, "recovered_keys.json")
DRIVE_RECOVERED_KEYS_PATH = os.path.join(DRIVE_DIR, "recovered_keys.json")
EXPECTED_PROB = 0.5
HAMMING_TARGET_RANGE = (41, 49)  # Anomaly range
BATCH_SIZE = 128 if device.type == "cuda" else 16  # Optimized for L4 GPU
PRIORITY_TARGETS = 100
CHECKPOINT_INTERVAL = 300  # 5 minutes
CHECKPOINT_STEPS = 500  # Every 500 steps
MAX_GOOD_SAMPLES = 50000  # Good keys (Hamming ≤50)
MAX_BAD_SAMPLES = 50000   # Bad keys (Hamming >80)

# -- Configuration --
RNG_SEED = int(time.time())
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)

# -- Save Config --
CONFIG = {
    "timestamp": time.strftime("%Y-%m-%d_%H-%M-%S"),
    "rng_seed": RNG_SEED,
    "max_k": MAX_K,
    "version": "RIPEMPHANTOM-T5",
    "hamming_target_range": HAMMING_TARGET_RANGE
}
with open(os.path.join(BASE_DIR, "experiment_config.json"), 'w') as f:
    json.dump(CONFIG, f, indent=2)

# -- Seed Keys --
SEED_KEYS = [
    0x006d816,  # Hamming 48 for 12ib7dAp...
    0x271000009982220  # Another low-Hamming key
]

# === Block 2: Key-to-Hash160 Pipeline ===
@jit(nopython=True)
def hamming_distance_and_flips(hash1, hash2):
    """Optimized Hamming distance and bit flip positions"""
    distance = 0
    flips = []
    for i in range(20):
        xor = hash1[i] ^ hash2[i]
        for j in range(8):
            if xor & (1 << j):
                bit_index = (i * 8) + (7 - j)
                flips.append(bit_index)
                distance += 1
    return distance, flips

def privkey_to_hash160(k: int, compressed: bool = True) -> bytes:
    """Generate RIPEMD160(SHA256(PubKey)) from private key"""
    try:
        if not (1 <= k < SECP256k1.order):
            return None
        sk = SigningKey.from_secret_exponent(k, curve=SECP256k1)
        vk = sk.get_verifying_key()
        if compressed:
            prefix = b'\x02' if vk.to_string()[32] % 2 == 0 else b'\x03'
            pubkey = prefix + vk.to_string()[:32]
        else:
            pubkey = b'\x04' + vk.to_string()
        sha = hashlib.sha256(pubkey).digest()
        ripemd = RIPEMD160.new(); ripemd.update(sha)
        return ripemd.digest()
    except Exception:
        return None

def key_to_vector(k: int) -> torch.Tensor:
    """Convert private key to 256-bit binary tensor"""
    return torch.tensor([(k >> i) & 1 for i in range(255, -1, -1)], dtype=torch.float32, device=device)

def hash160_to_vector(h: bytes) -> torch.Tensor:
    """Convert hash160 to 160-bit binary tensor"""
    return torch.tensor([(b >> i) & 1 for b in h for i in range(7, -1, -1)], dtype=torch.float32, device=device)

def vector_to_hash160(v: torch.Tensor) -> bytes:
    """Convert 160-bit vector to hash160 bytes"""
    hash_bytes = bytearray(20)
    v_cpu = v.cpu().numpy()
    for i in range(160):
        byte_idx = i // 8
        bit_idx = 7 - (i % 8)
        if v_cpu[i] > 0.5:
            hash_bytes[byte_idx] |= (1 << bit_idx)
    return bytes(hash_bytes)

# === Block 3: Neural Models ===
class HashPredictorNet(nn.Module):
    """Predict hash160 bit pattern from private key"""
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 160),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

class EntropyAnalysisNet(nn.Module):
    """Detect entropy anomalies in hash160 outputs"""
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(160, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2)
        )
        self.decoder = nn.Sequential(
            nn.Linear(64, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 160),
            nn.Sigmoid()
        )
        self.anomaly_detector = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        anomaly_score = self.anomaly_detector(encoded)
        return decoded, anomaly_score, encoded

# === Block 4: Adaptive Mutation Engine ===
bit_bias = defaultdict(float)
bad_bit_bias = defaultdict(float)  # For high-Hamming keys
xor_chain = []
mutation_log = []
recovered_keys = {}
mutation_bank = []

def update_bit_bias(flips: list, reward: float = 1.0, is_good: bool = True):
    """Reinforce bit positions for low (good) or high (bad) Hamming distances"""
    target = bit_bias if is_good else bad_bit_bias
    for bit in flips:
        target[bit] += reward

def add_xor_delta(prev_k: int, new_k: int, reward: float = 1.0):
    """Store successful XOR mutations"""
    delta = prev_k ^ new_k
    xor_chain.append((delta, reward))
    if len(xor_chain) > 100:
        xor_chain.pop(0)

def log_mutation_event(k_old, k_new, mutation_type, flips, delta_hamming):
    """Log mutation outcomes"""
    entry = {
        "time": time.time(),
        "type": mutation_type,
        "delta_hamming": delta_hamming,
        "flips": flips,
        "delta_k": k_old ^ k_new
    }
    mutation_log.append(entry)
    if len(mutation_log) > 5000:
        mutation_log.pop(0)

def record_mutation_result(mutation_vector: int, delta_hamming: int):
    """Track mutation effectiveness"""
    found = False
    for i, (vec, score) in enumerate(mutation_bank):
        if vec == mutation_vector:
            mutation_bank[i] = (vec, score + delta_hamming)
            found = True
            break
    if not found:
        mutation_bank.append((mutation_vector, delta_hamming))
    if len(mutation_bank) > 100:
        mutation_bank.sort(key=lambda x: -x[1])
        mutation_bank[:] = mutation_bank[:100]

def get_top_bit_bias(n=10, is_good: bool = True):
    """Get highest-scoring bit positions"""
    target = bit_bias if is_good else bad_bit_bias
    return sorted(target.items(), key=lambda x: -x[1])[:n]

def get_top_xor_deltas(n=10):
    """Get highest-reward XOR deltas"""
    return sorted(xor_chain, key=lambda x: -x[1])[:n]

def get_best_mutations(n=10):
    """Get top-performing mutation vectors"""
    return sorted(mutation_bank, key=lambda x: -x[1])[:n]

def mutate_key(k: int) -> int:
    """Adaptive key mutation with reinforcement learning"""
    new_k = k
    if random.random() < 0.5:
        new_k += random.randint(-1000000, 1000000)  # ±10^6 range
    for vec, _ in get_best_mutations(n=5):
        if random.random() < 0.3:
            new_k ^= vec
    for delta, _ in get_top_xor_deltas(n=5):
        if random.random() < 0.3:
            new_k ^= delta
    for bit, _ in get_top_bit_bias(n=5, is_good=True):
        if random.random() < 0.3:
            new_k ^= (1 << bit)
    for bit, _ in get_top_bit_bias(n=5, is_good=False):
        if random.random() < 0.1:
            new_k ^= (1 << bit)
    if random.random() < 0.2:
        new_k ^= random.getrandbits(8)
    return new_k if 1 <= new_k < MAX_K else random.randint(1, MAX_K - 1)

def generate_key(seed_k: int, transformer_model=None) -> int:
    """Generate key with neural guidance"""
    k = mutate_key(seed_k)
    if transformer_model:
        try:
            with torch.no_grad():
                k_tensor = key_to_vector(k).unsqueeze(0)
                prediction = transformer_model(k_tensor).squeeze(0)
                for i in range(160):
                    if prediction[i].item() > 0.95 and random.random() < 0.5:
                        k ^= (1 << (i % 160))
        except:
            pass
    return k if 1 <= k < MAX_K else random.randint(1, MAX_K - 1)

# === Block 5: Address and Target Management ===
def load_addresses() -> list:
    """Load and validate P2PKH addresses from file"""
    possible_paths = [ADDR_FILE] + [f"{ADDR_FILE[:-4]} ({i}).txt" for i in range(1, 6)]
    selected_path = next((p for p in possible_paths if os.path.exists(p)), None)

    if not selected_path:
        logging.error(f"No address file found at {ADDR_FILE} or alternatives.")
        raise FileNotFoundError("addresses.txt not found")

    targets = []
    with open(selected_path, 'r') as f:
        for line in f:
            addr = line.strip()
            try:
                decoded = base58.b58decode(addr)
                h160 = decoded[1:-4]
                if len(h160) == 20:
                    targets.append((addr, h160))
            except:
                logging.warning(f"Invalid address skipped: {addr}")
                continue

    if len(targets) == 0:
        logging.error("No valid P2PKH addresses found in file.")
        raise ValueError("No valid addresses loaded")

    logging.info(f"Loaded {len(targets)} valid P2PKH addresses from {selected_path}")
    return targets

target_stats = {}
recovered_keys = {}
mutation_bank = []

def init_target_stats(targets):
    """Initialize target statistics"""
    for addr, _ in targets:
        target_stats[addr] = {
            "best": 160,
            "best_k": -1,
            "scans": 0,
            "hamming_history": [],
            "last_improvement": time.time()
        }

def update_target_stat(addr: str, k: int, ham: int, compressed: bool):
    """Update target statistics and save recovered keys"""
    if len(target_stats[addr]["hamming_history"]) >= 1000:
        target_stats[addr]["hamming_history"].pop(0)
    target_stats[addr]["hamming_history"].append(ham)

    if ham < target_stats[addr]["best"]:
        target_stats[addr]["best"] = ham
        target_stats[addr]["best_k"] = k
        target_stats[addr]["last_improvement"] = time.time()
        logging.info(f"New best for {addr}: Hamming {ham}, Key {hex(k)}, Compressed: {compressed}")
        if ham == 0:
            recovered_keys[addr] = {"key": hex(k), "compressed": compressed}
            with open(RECOVERED_KEYS_PATH, 'w') as f:
                json.dump(recovered_keys, f, indent=2)
            with open(DRIVE_RECOVERED_KEYS_PATH, 'w') as f:
                json.dump(recovered_keys, f, indent=2)
            logging.info(f"Recovered key for {addr}: {hex(k)}, Compressed: {compressed}")

    target_stats[addr]["scans"] += 1

def get_priority_targets(targets, top_n=PRIORITY_TARGETS):
    """Prioritize targets with Hamming ≤50"""
    if not target_stats:
        return random.sample(targets, min(top_n, len(targets)))
    sorted_targets = sorted(
        target_stats.items(),
        key=lambda x: (x[1]["best"], -x[1]["last_improvement"])
    )
    top_addrs = set(addr for addr, _ in sorted_targets[:top_n] if target_stats[addr]["best"] <= 50)
    return [(addr, h) for addr, h in targets if addr in top_addrs]

# === Block 6: Statistical Analysis ===
def compute_bit_bias_confidence_intervals(is_good: bool = True):
    """Calculate confidence intervals for bit biases"""
    target = bit_bias if is_good else bad_bit_bias
    result = defaultdict(lambda: {"bias": 0, "proportion": 0, "ci_lower": 0, "ci_upper": 0, "significant": False})
    total_observations = sum(target.values()) or 1
    for bit, value in target.items():
        proportion = value / total_observations
        lower, upper = proportion_confint(value, total_observations, alpha=0.05, method='wilson')
        result[bit] = {
            "bias": value,
            "proportion": proportion,
            "ci_lower": lower,
            "ci_upper": upper,
            "significant": lower > EXPECTED_PROB or upper < EXPECTED_PROB
        }
    return result

def save_bit_bias_heatmap(is_good: bool = True):
    """Visualize bit bias distribution"""
    target = bit_bias if is_good else bad_bit_bias
    if not target:
        return
    bias_data = compute_bit_bias_confidence_intervals(is_good)
    bits = sorted(bias_data.keys())
    values = [bias_data[bit]["bias"] for bit in bits]
    lower_ci = [bias_data[bit]["ci_lower"] for bit in bits]
    upper_ci = [bias_data[bit]["ci_upper"] for bit in bits]

    plt.figure(figsize=(16, 6))
    plt.bar(bits, values, alpha=0.7)
    plt.errorbar(bits, values, yerr=[(v-l) for v, l in zip(values, lower_ci)],
                 fmt='none', ecolor='black', capsize=3)
    for bit in bits:
        if bias_data[bit]["significant"]:
            plt.plot(bit, bias_data[bit]["bias"], 'r*', markersize=10)
    plt.title(f"{'Good' if is_good else 'Bad'} Key Bit Bias Heatmap with 95% Confidence Intervals")
    plt.xlabel("Bit Position (0–159)")
    plt.ylabel("Reinforcement Score")
    plt.grid(True, alpha=0.3)
    plt.axhline(y=sum(values)/len(values) if values else 0, color='r', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, "figures", f"{'good' if is_good else 'bad'}_bit_bias_heatmap.png"), dpi=300)
    plt.savefig(os.path.join(DRIVE_DIR, "figures", f"{'good' if is_good else 'bad'}_bit_bias_heatmap.png"), dpi=300)
    plt.close()

def plot_loss_curve(losses, label: str):
    """Plot loss curve for neural models"""
    if not losses:
        return
    plt.figure(figsize=(10, 6))
    plt.plot(losses)
    if len(losses) > 10:
        window_size = min(len(losses) // 10, 20)
        smoothed = np.convolve(losses, np.ones(window_size)/window_size, mode='valid')
        plt.plot(range(window_size-1, len(losses)), smoothed, 'r-', linewidth=2)
    plt.title(f"{label} Loss Curve")
    plt.xlabel("Batch")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, "figures", f"{label.lower().replace(' ', '_')}_loss.png"), dpi=300)
    plt.savefig(os.path.join(DRIVE_DIR, "figures", f"{label.lower().replace(' ', '_')}_loss.png"), dpi=300)
    plt.close()

def create_bit_correlation_matrix(hash160_vectors, save_path=None):
    """Create correlation matrix for bit positions"""
    if not hash160_vectors or len(hash160_vectors) < 10:
        return None
    matrix = torch.stack(hash160_vectors).cpu().numpy()
    corr_matrix = np.corrcoef(matrix.T)
    plt.figure(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    cmap = sns.diverging_palette(230, 20, as_cmap=True)
    sns.heatmap(corr_matrix, mask=mask, cmap=cmap, vmax=0.3, vmin=-0.3,
                center=0, square=True, linewidths=.5)
    plt.title('Bit Position Correlation Matrix')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.savefig(save_path.replace(BASE_DIR, DRIVE_DIR), dpi=300)
        plt.close()
    else:
        plt.show()
    return corr_matrix

def visualize_hash_clusters(hash_vectors, save_path=None, method='tsne'):
    """Visualize hash clusters with dimensionality reduction"""
    if not hash_vectors or len(hash_vectors) < 10:
        return
    data = torch.stack(hash_vectors).cpu().numpy()
    if method.lower() == 'tsne':
        reducer = TSNE(n_components=2, random_state=RNG_SEED)
    elif method.lower() == 'umap':
        reducer = umap.UMAP(random_state=RNG_SEED)
    else:
        reducer = PCA(n_components=2, random_state=RNG_SEED)
    reduced_data = reducer.fit_transform(data)
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    clusters = dbscan.fit_predict(reduced_data)
    plt.figure(figsize=(12, 10))
    scatter = plt.scatter(reduced_data[:, 0], reduced_data[:, 1], c=clusters,
                         cmap='viridis', s=50, alpha=0.7)
    plt.legend(*scatter.legend_elements(), title="Clusters")
    plt.title(f'Hash160 Clusters ({method.upper()})')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.savefig(save_path.replace(BASE_DIR, DRIVE_DIR), dpi=300)
        plt.close()
    else:
        plt.show()
    return {'reduced_data': reduced_data, 'clusters': clusters}

def create_3d_bit_position_map(hash160_samples, save_path=None):
    """Create 3D visualization of bit position relationships"""
    if not hash160_samples or len(hash160_samples) < 100:
        return
    data = torch.stack(hash160_samples).cpu().numpy()
    bit_variance = np.var(data, axis=0)
    top_bits = np.argsort(bit_variance)[-3:]
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')
    x = data[:, top_bits[0]]
    y = data[:, top_bits[1]]
    z = data[:, top_bits[2]]
    colors = np.sum(data, axis=1)
    scatter = ax.scatter(x, y, z, c=colors, cmap='viridis', s=30, alpha=0.7)
    cbar = plt.colorbar(scatter)
    cbar.set_label('Bit Sum (Entropy Proxy)')
    ax.set_xlabel(f'Bit {top_bits[0]}')
    ax.set_ylabel(f'Bit {top_bits[1]}')
    ax.set_zlabel(f'Bit {top_bits[2]}')
    plt.title('3D Relationship Between High-Variance Bit Positions')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.savefig(save_path.replace(BASE_DIR, DRIVE_DIR), dpi=300)
        plt.close()
    else:
        plt.show()

def analyze_target_stats():
    """Analyze target performance with trend detection"""
    from scipy import stats
    if not target_stats:
        return {}

    analysis = {
        "global": {
            "best_hamming": min([s["best"] for s in target_stats.values()]),
            "avg_hamming": sum([s["best"] for s in target_stats.values()]) / len(target_stats),
            "total_scans": sum([s["scans"] for s in target_stats.values()]),
            "timestamp": time.time()
        },
        "targets": {}
    }

    for addr, stats in target_stats.items():
        if not stats["hamming_history"]:
            continue
        history = stats["hamming_history"]
        analysis["targets"][addr] = {
            "best": stats["best"],
            "mean": np.mean(history),
            "median": np.median(history),
            "std_dev": np.std(history),
            "scans": stats["scans"]
        }
        if len(history) > 10:
            x = np.arange(len(history))
            y = np.array(history)
            try:
                print(f"Debug: Analyzing {addr}, x type: {type(x)}, y type: {type(y)}, len(x): {len(x)}, len(y): {len(y)}")
                slope, _, r_value, p_value, _ = stats.linregress(x, y)
                analysis["targets"][addr].update({
                    "trend_slope": slope,
                    "trend_r_squared": r_value**2,
                    "trend_p_value": p_value
                })
            except Exception as e:
                print(f"Error in linregress for {addr}: {e}")
                analysis["targets"][addr].update({
                    "trend_slope": 0.0,
                    "trend_r_squared": 0.0,
                    "trend_p_value": 1.0
                })

    with open(os.path.join(REPORT_PATH, "target_analysis.json"), 'w') as f:
        json.dump(analysis, f, indent=2)
    with open(os.path.join(DRIVE_DIR, "reports", "target_analysis.json"), 'w') as f:
        json.dump(analysis, f, indent=2)
    return analysis

class NBISTTestSuite:
    """NIST-based statistical test suite for randomness"""
    def __init__(self):
        self.tests = {
            "frequency": self.frequency_test,
            "frequency_block": self.block_frequency_test,
            "runs": self.runs_test,
            "longest_run": self.longest_run_test,
            "serial": self.serial_test,
            "entropy": self.approximate_entropy_test
        }

    def frequency_test(self, bits):
        n = len(bits)
        s = sum([(2*bit-1) for bit in bits])
        s_obs = abs(s) / np.sqrt(n)
        p_value = math.erfc(s_obs / np.sqrt(2))
        return p_value

    def block_frequency_test(self, bits, block_size=10):
        n = len(bits)
        n_blocks = n // block_size
        if n_blocks == 0:
            return 1.0
        blocks = [bits[i:i+block_size] for i in range(0, n_blocks*block_size, block_size)]
        proportions = [sum(block)/block_size for block in blocks]
        chi_squared = 4.0 * block_size * sum([(prop-0.5)**2 for prop in proportions])
        p_value = stats.chi2.sf(chi_squared, n_blocks)
        return p_value

    def runs_test(self, bits):
        n = len(bits)
        if n < 100:
            return 1.0
        prop = sum(bits) / n
        tau = 2.0 / np.sqrt(n)
        if abs(prop - 0.5) >= tau:
            return 0.0
        runs = 1
        for i in range(1, n):
            if bits[i] != bits[i-1]:
                runs += 1
        p_value = math.erfc(abs(runs - 2*n*prop*(1-prop)) /
                          (2*np.sqrt(2*n)*prop*(1-prop)))
        return p_value

    def longest_run_test(self, bits):
        n = len(bits)
        if n < 128:
            return 1.0
        if n < 6272:
            k, m = 3, 8
            v = [1, 2, 3, 4]
            pi = [0.2148, 0.3672, 0.2305, 0.1875]
        else:
            k, m = 5, 128
            v = [4, 5, 6, 7, 8, 9]
            pi = [0.1174, 0.2430, 0.2493, 0.1752, 0.1027, 0.1124]
        num_blocks = n // m
        longest_runs = []
        for i in range(num_blocks):
            block = bits[i*m:(i+1)*m]
            max_run = 0
            current_run = 0
            for bit in block:
                if bit == 1:
                    current_run += 1
                    max_run = max(max_run, current_run)
                else:
                    current_run = 0
            longest_runs.append(max_run)
        frequencies = [0] * (k+1)
        for run in longest_runs:
            if run <= v[0]:
                frequencies[0] += 1
            elif run >= v[-1]:
                frequencies[-1] += 1
            else:
                for j in range(1, len(v)):
                    if v[j-1] < run <= v[j]:
                        frequencies[j] += 1
                        break
        chi_squared = sum([(frequencies[i] - num_blocks*pi[i])**2 / (num_blocks*pi[i])
                          for i in range(len(frequencies))])
        p_value = stats.chi2.sf(chi_squared, k)
        return p_value

    def serial_test(self, bits, m=3):
        n = len(bits)
        if n < 2**m:
            return 1.0
        extended_bits = bits + bits[:m-1]
        pattern_counts1 = defaultdict(int)
        for i in range(n):
            pattern = tuple(extended_bits[i:i+m])
            pattern_counts1[pattern] += 1
        pattern_counts2 = defaultdict(int)
        for i in range(n):
            pattern = tuple(extended_bits[i:i+m-1])
            pattern_counts2[pattern] += 1
        pattern_counts3 = defaultdict(int)
        for i in range(n):
            pattern = tuple(extended_bits[i:i+m-2])
            pattern_counts3[pattern] += 1
        psi_sq_m = sum([count**2 for count in pattern_counts1.values()]) * 2**m / n - n
        psi_sq_m1 = sum([count**2 for count in pattern_counts2.values()]) * 2**(m-1) / n - n
        psi_sq_m2 = sum([count**2 for count in pattern_counts3.values()]) * 2**(m-2) / n - n
        delta1 = psi_sq_m - psi_sq_m1
        delta2 = psi_sq_m - 2*psi_sq_m1 + psi_sq_m2
        p_value1 = stats.chi2.sf(delta1, 2**(m-1))
        p_value2 = stats.chi2.sf(delta2, 2**(m-2))
        return min(p_value1, p_value2)

    def approximate_entropy_test(self, bits, m=5):
        n = len(bits)
        if n < 100:
            return 1.0
        def phi(m_val):
            counts = defaultdict(int)
            for i in range(n):
                pattern = tuple(bits[(i+j) % n] for j in range(m_val))
                counts[pattern] += 1
            c_m = [count/n for count in counts.values()]
            return sum([p * np.log(p) for p in c_m if p > 0])
        apen = phi(m) - phi(m+1)
        chi_squared = 2 * n * (np.log(2) - apen)
        p_value = stats.chi2.sf(chi_squared, 2**m)
        return p_value

    def run_all_tests(self, bits):
        results = {}
        for test_name, test_func in self.tests.items():
            p_value = test_func(bits)
            passed = p_value >= 0.01
            conf_level = 0.95
            z = stats.norm.ppf(1 - (1 - conf_level) / 2)
            if passed:
                lower_ci = (1 + 1/(4*1) - z/(2*np.sqrt(1)))
                upper_ci = 1.0
            else:
                lower_ci = 0.0
                upper_ci = z/(2*np.sqrt(1))
            results[test_name] = {
                "p_value": p_value,
                "passed": passed,
                "confidence_lower": lower_ci,
                "confidence_upper": upper_ci
            }
        return results

def run_entropy_scan(sample_size: int = 100_000, reference_hashes: list = [],
                    statistical_rigor: bool = True):
    """Scan keyspace for entropy anomalies"""
    global entropy_counter, entropy_total, entropy_samples
    print(f"Running anomaly detection scan with {sample_size:,} keys...")
    entropy_counter = np.zeros(160, dtype=int)
    entropy_total = 0
    entropy_samples = 0
    hamming_distribution = []
    hash_samples = []
    bit_vectors = []

    for _ in tqdm(range(sample_size), desc="Entropy Scan"):
        k = random.randint(1, MAX_K)
        for compressed in [True, False]:
            h160 = privkey_to_hash160(k, compressed)
            if not h160:
                continue
            entropy_samples += 1
            for i in range(20):
                byte = h160[i]
                for j in range(8):
                    bit_index = (i * 8) + (7 - j)
                    if byte & (1 << j):
                        entropy_counter[bit_index] += 1
            entropy_total += 1
            hash_samples.append(h160)
            bit_vectors.append(hash160_to_vector(h160))
            for _, ref in reference_hashes:
                dist, _ = hamming_distance_and_flips(h160, ref)
                hamming_distribution.append(dist)

    plt.figure(figsize=(16, 6))
    proportions = entropy_counter / max(1, entropy_samples)
    confidence = 0.95
    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    lower_ci = []
    upper_ci = []
    for i in range(160):
        p = proportions[i]
        n = entropy_samples
        denominator = 1 + z**2/n
        centre_adjusted_prob = (p + z**2/(2*n))/denominator
        adjusted_standard_dev = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2))/denominator
        lower = max(0, centre_adjusted_prob - adjusted_standard_dev)
        upper = min(1, centre_adjusted_prob + adjusted_standard_dev)
        lower_ci.append(lower)
        upper_ci.append(upper)
    plt.bar(range(160), proportions, alpha=0.7)
    plt.errorbar(range(160), proportions,
                yerr=[proportions-lower_ci, upper_ci-proportions],
                fmt='none', ecolor='black', capsize=2, alpha=0.3)
    plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Expected (0.5)')
    for i in range(160):
        if lower_ci[i] > 0.5 or upper_ci[i] < 0.5:
            plt.plot(i, proportions[i], 'r*', markersize=8)
    plt.title(f"RIPEMD-160 Bitwise Distribution with {confidence*100:.0f}% Confidence Intervals")
    plt.xlabel("Bit Position (0–159)")
    plt.ylabel("Frequency of 1s")
    plt.ylim(0.45, 0.55)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, "figures", "hash160_entropy_profile.png"), dpi=300)
    plt.savefig(os.path.join(DRIVE_DIR, "figures", "hash160_entropy_profile.png"), dpi=300)
    plt.close()

    if hamming_distribution:
        plt.figure(figsize=(12, 8))
        hist, bin_edges = np.histogram(hamming_distribution, bins=40)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        plt.bar(bin_centers, hist, width=(bin_edges[1]-bin_edges[0])*0.8,
               color='steelblue', edgecolor='black', alpha=0.7)
        x = np.arange(40, 120)
        n = 160
        p = 0.5
        expected = stats.binom.pmf(x, n, p) * sum(hist) * (bin_edges[1]-bin_edges[0])
        plt.plot(x, expected, 'r-', linewidth=2,
                label='Expected (Binomial Distribution)')
        mean_ham = np.mean(hamming_distribution)
        std_ham = np.std(hamming_distribution)
        expected_mean = n * p
        expected_std = np.sqrt(n * p * (1-p))
        plt.axvline(x=mean_ham, color='blue', linestyle='--',
                   label=f'Observed Mean: {mean_ham:.2f}')
        plt.axvline(x=expected_mean, color='red', linestyle='--',
                   label=f'Expected Mean: {expected_mean:.2f}')
        ks_stat, p_value = stats.kstest(hamming_distribution, stats.binom.cdf, args=(n, p))
        plt.title(f'Hamming Distances to Reference Hash160s\nKS-test: D={ks_stat:.4f}, p={p_value:.6f}')
        plt.xlabel('Hamming Distance (bits)')
        plt.ylabel('Frequency')
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(BASE_DIR, "figures", "hamming_histogram.png"), dpi=300)
        plt.savefig(os.path.join(DRIVE_DIR, "figures", "hamming_histogram.png"), dpi=300)
        plt.close()

    if statistical_rigor and len(bit_vectors) > 1000:
        print("Performing advanced statistical analysis...")
        bit_array = np.stack([v.cpu().numpy() for v in bit_vectors])
        test_results = {}
        nist_suite = NBISTTestSuite()
        for bit_pos in tqdm(range(160), desc="Testing bit positions"):
            bits = bit_array[:, bit_pos].astype(int).tolist()
            test_results[bit_pos] = nist_suite.run_all_tests(bits)
        with open(os.path.join(BASE_DIR, "reports", "nist_test_results.json"), 'w') as f:
            json.dump(test_results, f, indent=2)
        with open(os.path.join(DRIVE_DIR, "reports", "nist_test_results.json"), 'w') as f:
            json.dump(test_results, f, indent=2)
        summary = {
            "failed_tests": 0,
            "total_tests": 0,
            "significant_bits": []
        }
        for bit_pos, results in test_results.items():
            for test_name, result in results.items():
                summary["total_tests"] += 1
                if not result["passed"]:
                    summary["failed_tests"] += 1
                    summary["significant_bits"].append({
                        "bit": bit_pos,
                        "test": test_name,
                        "p_value": result["p_value"]
                    })
        with open(os.path.join(BASE_DIR, "reports", "nist_summary.json"), 'w') as f:
            json.dump(summary, f, indent=2)
        with open(os.path.join(DRIVE_DIR, "reports", "nist_summary.json"), 'w') as f:
            json.dump(summary, f, indent=2)

    return {
        'hash_samples': hash_samples,
        'bit_vectors': bit_vectors,
        'hamming_distribution': hamming_distribution
    }

# === Block 7: GPU-Accelerated Key Processing ===
def process_key_batch(keys: list, targets: list) -> list:
    """GPU-accelerated key processing with parallelization, testing compressed/uncompressed"""
    results = []
    target_h160s = [h for _, h in targets]

    def process_key(k):
        batch_results = []
        for compressed in [True, False]:
            h160 = privkey_to_hash160(k, compressed)
            if h160:
                for addr, target_h160 in targets:
                    ham, flips = hamming_distance_and_flips(h160, target_h160)
                    batch_results.append((addr, k, h160, ham, flips, compressed))
        return batch_results

    if device.type == "cuda":
        try:
            keys_array = cp.array(keys, dtype=cp.uint64)
            target_h160s_array = cp.array(target_h160s, dtype=cp.uint8)
            for k in keys:
                for compressed in [True, False]:
                    h160 = privkey_to_hash160(k, compressed)
                    if h160:
                        h160_array = cp.array(list(h160), dtype=cp.uint8)
                        for addr, target_h160 in targets:
                            target_array = cp.array(list(target_h160), dtype=cp.uint8)
                            ham = cp.sum(h160_array != target_array).get()
                            _, flips = hamming_distance_and_flips(h160, target_h160)
                            results.append((addr, k, h160, ham, flips, compressed))
        except cp.cuda.memory.OutOfMemoryError:
            logging.warning("GPU memory error, falling back to CPU for this batch")
            with ThreadPoolExecutor() as executor:
                batch_results = executor.map(process_key, keys)
                for batch in batch_results:
                    results.extend(batch)
    else:
        with ThreadPoolExecutor() as executor:
            batch_results = executor.map(process_key, keys)
            for batch in batch_results:
                results.extend(batch)

    return results

# === Block 8: Checkpointing and Signal Handling ===
def save_checkpoint(models: dict, step_count: int):
    """Save models and state to local and Drive"""
    try:
        for model_name, model in models.items():
            torch.save(model.state_dict(), os.path.join(CHECKPOINT_PATH, f"{model_name}.pt"))
            torch.save(model.state_dict(), os.path.join(DRIVE_CHECKPOINT_PATH, f"{model_name}.pt"))

        state = {
            "bit_bias": dict(bit_bias),
            "bad_bit_bias": dict(bad_bit_bias),
            "xor_chain": xor_chain,
            "mutation_bank": [(hex(vec), score) for vec, score in mutation_bank],
            "step": step_count,
            "timestamp": time.time(),
            "config": CONFIG
        }
        torch.save(state, os.path.join(CHECKPOINT_PATH, "research_state.pt"))
        torch.save(state, os.path.join(DRIVE_CHECKPOINT_PATH, "research_state.pt"))
        with open(os.path.join(CHECKPOINT_PATH, "bit_bias.json"), 'w') as f:
            json.dump(dict(bit_bias), f)
        with open(os.path.join(DRIVE_CHECKPOINT_PATH, "bit_bias.json"), 'w') as f:
            json.dump(dict(bit_bias), f)
        with open(os.path.join(CHECKPOINT_PATH, "bad_bit_bias.json"), 'w') as f:
            json.dump(dict(bad_bit_bias), f)
        with open(os.path.join(DRIVE_CHECKPOINT_PATH, "bad_bit_bias.json"), 'w') as f:
            json.dump(dict(bad_bit_bias), f)
        with open(os.path.join(CHECKPOINT_PATH, "xor_chain.json"), 'w') as f:
            json.dump(xor_chain, f)
        with open(os.path.join(DRIVE_CHECKPOINT_PATH, "xor_chain.json"), 'w') as f:
            json.dump(xor_chain, f)
        with open(os.path.join(CHECKPOINT_PATH, "mutation_bank.json"), 'w') as f:
            json.dump([(hex(vec), score) for vec, score in mutation_bank], f)
        with open(os.path.join(DRIVE_CHECKPOINT_PATH, "mutation_bank.json"), 'w') as f:
            json.dump([(hex(vec), score) for vec, score in mutation_bank], f)
        with open(os.path.join(CHECKPOINT_PATH, "step.txt"), 'w') as f:
            f.write(str(step_count))
        with open(os.path.join(DRIVE_CHECKPOINT_PATH, "step.txt"), 'w') as f:
            f.write(str(step_count))

        log_event("checkpoint", {"step": step_count, "recovered_keys": len(recovered_keys)})
    except Exception as e:
        logging.error(f"Checkpoint save failed: {e}")

def load_checkpoint(models: dict) -> int:
    """Load prior state"""
    try:
        state_path = os.path.join(CHECKPOINT_PATH, "research_state.pt")
        step = 0
        if os.path.exists(state_path):
            state = torch.load(state_path, map_location=device)
            bit_bias.update(state.get("bit_bias", {}))
            bad_bit_bias.update(state.get("bad_bit_bias", {}))
            xor_chain.extend(state.get("xor_chain", []))
            mutation_bank.extend([(int(vec, 16), score) for vec, score in state.get("mutation_bank", [])])
            step = state.get("step", 0)
            for model_name, model in models.items():
                model_path = os.path.join(CHECKPOINT_PATH, f"{model_name}.pt")
                if os.path.exists(model_path):
                    model.load_state_dict(torch.load(model_path, map_location=device))
            logging.info(f"Loaded consolidated state from step {step}")
        return step
    except Exception as e:
        logging.warning(f"No checkpoint loaded: {e}")
        return 0

def log_event(event_type: str, payload: dict):
    """Log structured events"""
    try:
        entry = {
            "time": time.time(),
            "event": event_type,
            "data": payload
        }
        with open(LOG_PATH, 'a') as f:
            f.write(json.dumps(entry) + '\n')
        with open(os.path.join(DRIVE_DIR, "logs", "telemetry.jsonl"), 'a') as f:
            f.write(json.dumps(entry) + '\n')
    except Exception as e:
        print(f"Failed to log event: {e}")

# === Block 9: Signal Handler for Interruptions ===
def signal_handler(sig, frame, models, step):
    """Handle interruptions with immediate checkpointing"""
    print(f"Received interrupt signal {sig}, saving checkpoint at step {step}...")
    save_checkpoint(models, step)
    logging.info(f"Interrupted at step {step}, checkpoint saved")
    sys.exit(0)

# === Block 10: Main Recovery Loop ===
def run_phantom_t5(targets, max_steps=250_000):
    """Main recovery loop with adaptive learning of good and bad keys"""
    global bit_bias, bad_bit_bias, xor_chain, mutation_log, recovered_keys

    # Initialize models
    models = {
        'predictor_net': HashPredictorNet().to(device),
        'entropy_net': EntropyAnalysisNet().to(device)
    }
    optimizers = {
        'predictor_net': torch.optim.Adam(models['predictor_net'].parameters(), lr=0.001),
        'entropy_net': torch.optim.Adam(models['entropy_net'].parameters(), lr=0.001)
    }
    losses = {'predictor': [], 'entropy': []}

    # Load checkpoint
    start_step = load_checkpoint(models)

    # Initialize target stats
    init_target_stats(targets)

    # Tracking variables
    best_hamming = 160
    good_hash_samples = []  # Hamming ≤50
    bad_hash_samples = []   # Hamming >80
    good_hash_vectors = []
    bad_hash_vectors = []
    anomalies_found = 0
    last_checkpoint_time = time.time()
    start_time = time.time()
    keys_processed = 0

    # Set up signal handler
    signal.signal(signal.SIGINT, lambda sig, frame: signal_handler(sig, frame, models, start_step))
    signal.signal(signal.SIGTERM, lambda sig, frame: signal_handler(sig, frame, models, start_step))

    logging.info(f"Starting recovery with {len(targets)} targets, {max_steps} steps, device: {device}")
    print(f"Starting RIPEMPHANTOM-T5 with {len(targets)} targets on {device}...")

    try:
        for step in tqdm(range(start_step, max_steps), desc="Recovery Progress"):
            # Progress logging
            if step % 100 == 0:
                elapsed = time.time() - start_time
                steps_per_sec = (step - start_step) / max(1, elapsed)
                keys_per_sec = keys_processed / max(1, elapsed)
                eta = (max_steps - step) / max(1, steps_per_sec) / 3600
                print(f"Step {step}: Processed {len(good_hash_samples)} good samples, "
                      f"{len(bad_hash_samples)} bad samples, Best Hamming: {best_hamming}, "
                      f"Recovered: {len(recovered_keys)}, Keys/sec: {keys_per_sec:.2f}, ETA: {eta:.2f} hours")

            # Select priority targets
            current_targets = get_priority_targets(targets)
            if not current_targets:
                current_targets = random.sample(targets, min(5, len(targets)))

            # Generate keys
            seed_k = random.choice(SEED_KEYS + [random.randint(1, MAX_K)])
            keys = [generate_key(seed_k, models['predictor_net']) for _ in range(BATCH_SIZE)]

            # Process keys
            batch_results = process_key_batch(keys, current_targets)
            keys_processed += len(keys) * 2  # Compressed and uncompressed
            if not batch_results:
                continue

            # Update stats and train models
            good_key_vectors = []
            good_hash_vectors_batch = []
            bad_key_vectors = []
            bad_hash_vectors_batch = []
            for addr, k, h160, ham, flips, compressed in batch_results:
                update_target_stat(addr, k, ham, compressed)
                best_hamming = min(best_hamming, ham)
                if ham <= 50 or ham == 0:
                    anomalies_found += 1
                    reward = 1.0 / max(1, ham)  # Higher reward for lower Hamming
                    update_bit_bias(flips, reward, is_good=True)
                    add_xor_delta(seed_k, k, reward)
                    record_mutation_result(k ^ seed_k, 160 - ham)
                    log_mutation_event(seed_k, k, "targeted", flips, ham - target_stats[addr]["best"])
                    if len(good_hash_samples) < MAX_GOOD_SAMPLES:
                        good_hash_samples.append(h160)
                        good_hash_vectors.append(hash160_to_vector(h160))
                        good_key_vectors.append(key_to_vector(k))
                        good_hash_vectors_batch.append(hash160_to_vector(h160))
                elif ham > 80:
                    reward = 1.0 / max(1, 160 - ham)  # Reward for avoiding high Hamming
                    update_bit_bias(flips, reward, is_good=False)
                    if len(bad_hash_samples) < MAX_BAD_SAMPLES:
                        bad_hash_samples.append(h160)
                        bad_hash_vectors.append(hash160_to_vector(h160))
                        bad_key_vectors.append(key_to_vector(k))
                        bad_hash_vectors_batch.append(hash160_to_vector(h160))

            # Train models on balanced good/bad data
            if good_hash_vectors_batch or bad_hash_vectors_batch:
                all_key_vectors = good_key_vectors + bad_key_vectors[:len(good_key_vectors)]  # Balance dataset
                all_hash_vectors = good_hash_vectors_batch + bad_hash_vectors_batch[:len(good_hash_vectors_batch)]
                if all_key_vectors:
                    key_vectors = torch.stack(all_key_vectors)
                    true_hash = torch.stack(all_hash_vectors)

                    optimizers['predictor_net'].zero_grad()
                    pred_hash = models['predictor_net'](key_vectors)
                    loss_pred = F.binary_cross_entropy(pred_hash, true_hash)
                    loss_pred.backward()
                    optimizers['predictor_net'].step()
                    losses['predictor'].append(loss_pred.item())

                    optimizers['entropy_net'].zero_grad()
                    decoded, anomaly_score, _ = models['entropy_net'](true_hash)
                    recon_loss = F.binary_cross_entropy(decoded, true_hash)
                    anomaly_loss = torch.mean(anomaly_score)
                    loss_entropy = recon_loss + 0.1 * anomaly_loss
                    loss_entropy.backward()
                    optimizers['entropy_net'].step()
                    losses['entropy'].append(loss_entropy.item())

            # Periodic checkpointing and analysis
            current_time = time.time()
            if (step + 1) % CHECKPOINT_STEPS == 0 or (current_time - last_checkpoint_time) > CHECKPOINT_INTERVAL:
                save_bit_bias_heatmap(is_good=True)
                save_bit_bias_heatmap(is_good=False)
                plot_loss_curve(losses['predictor'], "Hash Predictor")
                plot_loss_curve(losses['entropy'], "Entropy Analysis")
                if len(good_hash_vectors) >= 1000:
                    create_bit_correlation_matrix(good_hash_vectors[-1000:],
                                                os.path.join(BASE_DIR, "figures", f"good_correlation_matrix_step_{step+1}.png"))
                    visualize_hash_clusters(good_hash_vectors[-1000:],
                                          os.path.join(BASE_DIR, "figures", f"good_hash_clusters_step_{step+1}.png"))
                    create_3d_bit_position_map(good_hash_vectors[-1000:],
                                              os.path.join(BASE_DIR, "figures", f"good_3d_bit_map_step_{step+1}.png"))
                if len(bad_hash_vectors) >= 1000:
                    create_bit_correlation_matrix(bad_hash_vectors[-1000:],
                                                os.path.join(BASE_DIR, "figures", f"bad_correlation_matrix_step_{step+1}.png"))
                    visualize_hash_clusters(bad_hash_vectors[-1000:],
                                          os.path.join(BASE_DIR, "figures", f"bad_hash_clusters_step_{step+1}.png"))
                    create_3d_bit_position_map(bad_hash_vectors[-1000:],
                                              os.path.join(BASE_DIR, "figures", f"bad_3d_bit_map_step_{step+1}.png"))
                analyze_target_stats()
                threading.Thread(target=save_checkpoint, args=(models, step + 1)).start()
                last_checkpoint_time = current_time

            # Check if all keys recovered
            if len(recovered_keys) == len(targets):
                print("All keys recovered!")
                break

    except KeyboardInterrupt:
        print(f"Interrupted at step {step}. Saving checkpoint...")
        save_checkpoint(models, step)
        logging.info(f"Interrupted at step {step}, checkpoint saved")

    # Finalize
    save_bit_bias_heatmap(is_good=True)
    save_bit_bias_heatmap(is_good=False)
    plot_loss_curve(losses['predictor'], "Hash Predictor")
    plot_loss_curve(losses['entropy'], "Entropy Analysis")
    if len(good_hash_vectors) >= 1000:
        create_bit_correlation_matrix(good_hash_vectors, os.path.join(BASE_DIR, "figures", "final_good_correlation_matrix.png"))
        visualize_hash_clusters(good_hash_vectors, os.path.join(BASE_DIR, "figures", "final_good_hash_clusters.png"))
        create_3d_bit_position_map(good_hash_vectors, os.path.join(BASE_DIR, "figures", "final_good_3d_bit_map.png"))
    if len(bad_hash_vectors) >= 1000:
        create_bit_correlation_matrix(bad_hash_vectors, os.path.join(BASE_DIR, "figures", "final_bad_correlation_matrix.png"))
        visualize_hash_clusters(bad_hash_vectors, os.path.join(BASE_DIR, "figures", "final_bad_hash_clusters.png"))
        create_3d_bit_position_map(bad_hash_vectors, os.path.join(BASE_DIR, "figures", "final_bad_3d_bit_map.png"))
    analyze_target_stats()
    save_checkpoint(models, max_steps)
    with open(os.path.join(REPORT_PATH, "final_report.json"), 'w') as f:
        json.dump({
            "best_hamming": best_hamming,
            "good_samples": len(good_hash_samples),
            "bad_samples": len(bad_hash_samples),
            "anomalies_found": anomalies_found,
            "recovered_keys": recovered_keys
        }, f, indent=2)
    with open(os.path.join(DRIVE_DIR, "reports", "final_report.json"), 'w') as f:
        json.dump({
            "best_hamming": best_hamming,
            "good_samples": len(good_hash_samples),
            "bad_samples": len(bad_hash_samples),
            "anomalies_found": anomalies_found,
            "recovered_keys": recovered_keys
        }, f, indent=2)

    print(f"Recovery complete! Recovered {len(recovered_keys)} keys. Check {RECOVERED_KEYS_PATH} for results.")
    return {
        'best_hamming': best_hamming,
        'good_samples': len(good_hash_samples),
        'bad_samples': len(bad_hash_samples),
        'anomalies_found': anomalies_found,
        'recovered_keys': recovered_keys
    }

# === Block 11: Entry Point ===
if __name__ == "__main__":
    try:
        targets = load_addresses()
        if not targets:
            raise ValueError("No valid addresses loaded")
        run_phantom_t5(targets)
    except Exception as e:
        logging.error(f"Execution failed: {e}")
        print(f"Error: {e}. Check {os.path.join(BASE_DIR, 'logs', 'recovery.log')} for details.")

Using GPU: NVIDIA L4
Fri May  9 03:55:48 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   46C    P8             16W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------------

Recovery Progress:   0%|          | 0/250000 [00:00<?, ?it/s]

Step 0: Processed 0 good samples, 0 bad samples, Best Hamming: 160, Recovered: 0, Keys/sec: 0.00, ETA: 69.44 hours


Recovery Progress:   0%|          | 0/250000 [00:00<?, ?it/s]
ERROR:root:Execution failed: invalid literal for int() with base 10: b'I\x0b\x9dn\x80\x9f\xad\xb6\xa0\xae\x98\xfd\xa1+\xcd}\xd6uR:'


Error: invalid literal for int() with base 10: b'I\x0b\x9dn\x80\x9f\xad\xb6\xa0\xae\x98\xfd\xa1+\xcd}\xd6uR:'. Check /content/RIPEMPHANTOM-T5/logs/recovery.log for details.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# === RIPEMPHANTOM-T5 ===
# Automated RIPEMD-160 Entropy Anomaly Exploit for Bitcoin Private Key Recovery
# Optimized for Google Colab with NVIDIA L4 GPU, Python 3.10, CUDA 12.x

# -- Automated Dependency Installation --
import subprocess
import sys
import os
import signal
import time
import random
import json
import hashlib
import base58
import math

def install_packages():
    """Install required packages"""
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
    except subprocess.CalledProcessError as e:
        print(f"Failed to upgrade pip: {e}")

    packages = [
        "base58", "pycryptodome", "ecdsa", "tqdm", "matplotlib",
        "scipy", "pandas", "seaborn", "statsmodels", "scikit-learn", "umap-learn",
        "numba", "cupy-cuda12x"
    ]
    for pkg in packages:
        try:
            __import__(pkg.replace("-cuda12x", ""))
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

install_packages()

# -- Core Imports --
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.mplot3d import Axes3D
from ecdsa import SigningKey, SECP256k1
from Crypto.Hash import RIPEMD160
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from statsmodels.stats.proportion import proportion_confint
from sklearn.cluster import DBSCAN
from sklearn.manifold import TSNE
import umap
from numba import jit
import cupy as cp
import logging
import threading
from concurrent.futures import ThreadPoolExecutor

# -- Prepare Workspace --
BASE_DIR = "/content/RIPEMPHANTOM-T5"
DRIVE_DIR = "/content/drive/MyDrive/RIPEMPHANTOM-T5"
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "logs"), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "figures"), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "reports"), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "datasets"), exist_ok=True)

# -- Setup Logging --
logging.basicConfig(
    filename=os.path.join(BASE_DIR, "logs", "recovery.log"),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# -- Detect GPU --
try:
    if torch.cuda.is_available():
        device = torch.device("cuda:0")  # Colab assigns L4 as cuda:0
        torch.cuda.set_device(0)
        logging.info(f"GPU detected: {torch.cuda.get_device_name(0)}, CUDA version: {torch.version.cuda}, "
                     f"CuPy devices: {cp.cuda.runtime.getDeviceCount()}, Selected device: cuda:0")
        print(f"Using GPU: {torch.cuda.get_device_name(0)}")
        !nvidia-smi
    else:
        device = torch.device("cpu")
        logging.info("No GPU detected, using CPU with numba JIT")
        print("No GPU detected, using CPU")
except Exception as e:
    device = torch.device("cpu")
    logging.warning(f"GPU detection failed: {e}, using CPU")
    print(f"GPU detection failed: {e}, using CPU")

# -- Constants --
MAX_K = 2**61
ADDR_FILE = os.path.join(BASE_DIR, "addresses.txt")
CHECKPOINT_PATH = os.path.join(BASE_DIR, "checkpoints")
DRIVE_CHECKPOINT_PATH = os.path.join(DRIVE_DIR, "checkpoints")
LOG_PATH = os.path.join(BASE_DIR, "logs", "telemetry.jsonl")
REPORT_PATH = os.path.join(BASE_DIR, "reports")
RECOVERED_KEYS_PATH = os.path.join(BASE_DIR, "recovered_keys.json")
DRIVE_RECOVERED_KEYS_PATH = os.path.join(DRIVE_DIR, "recovered_keys.json")
EXPECTED_PROB = 0.5
HAMMING_TARGET_RANGE = (41, 49)  # Anomaly range
BATCH_SIZE = 128 if device.type == "cuda" else 16  # Optimized for L4 GPU
PRIORITY_TARGETS = 100
CHECKPOINT_INTERVAL = 300  # 5 minutes
CHECKPOINT_STEPS = 500  # Every 500 steps
MAX_GOOD_SAMPLES = 50000  # Good keys (Hamming ≤50)
MAX_BAD_SAMPLES = 50000   # Bad keys (Hamming >80)

# -- Configuration --
RNG_SEED = int(time.time())
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)

# -- Save Config --
CONFIG = {
    "timestamp": time.strftime("%Y-%m-%d_%H-%M-%S"),
    "rng_seed": RNG_SEED,
    "max_k": MAX_K,
    "version": "RIPEMPHANTOM-T5",
    "hamming_target_range": HAMMING_TARGET_RANGE
}
with open(os.path.join(BASE_DIR, "experiment_config.json"), 'w') as f:
    json.dump(CONFIG, f, indent=2)

# -- Seed Keys --
SEED_KEYS = [
    0x006d816,  # Hamming 48 for 12ib7dAp...
    0x271000009982220  # Another low-Hamming key
]

# === Block 2: Key-to-Hash160 Pipeline ===
@jit(nopython=True)
def hamming_distance_and_flips(hash1, hash2):
    """Optimized Hamming distance and bit flip positions"""
    distance = 0
    flips = []
    for i in range(20):
        xor = hash1[i] ^ hash2[i]
        for j in range(8):
            if xor & (1 << j):
                bit_index = (i * 8) + (7 - j)
                flips.append(bit_index)
                distance += 1
    return distance, flips

def privkey_to_hash160(k: int, compressed: bool = True) -> bytes:
    """Generate RIPEMD160(SHA256(PubKey)) from private key"""
    try:
        if not (1 <= k < SECP256k1.order):
            return None
        sk = SigningKey.from_secret_exponent(k, curve=SECP256k1)
        vk = sk.get_verifying_key()
        if compressed:
            prefix = b'\x02' if vk.to_string()[32] % 2 == 0 else b'\x03'
            pubkey = prefix + vk.to_string()[:32]
        else:
            pubkey = b'\x04' + vk.to_string()
        sha = hashlib.sha256(pubkey).digest()
        ripemd = RIPEMD160.new(); ripemd.update(sha)
        return ripemd.digest()
    except Exception:
        return None

def key_to_vector(k: int) -> torch.Tensor:
    """Convert private key to 256-bit binary tensor"""
    return torch.tensor([(k >> i) & 1 for i in range(255, -1, -1)], dtype=torch.float32, device=device)

def hash160_to_vector(h: bytes) -> torch.Tensor:
    """Convert hash160 to 160-bit binary tensor"""
    return torch.tensor([(b >> i) & 1 for b in h for i in range(7, -1, -1)], dtype=torch.float32, device=device)

def vector_to_hash160(v: torch.Tensor) -> bytes:
    """Convert 160-bit vector to hash160 bytes"""
    hash_bytes = bytearray(20)
    v_cpu = v.cpu().numpy()
    for i in range(160):
        byte_idx = i // 8
        bit_idx = 7 - (i % 8)
        if v_cpu[i] > 0.5:
            hash_bytes[byte_idx] |= (1 << bit_idx)
    return bytes(hash_bytes)

# === Block 3: Neural Models ===
class HashPredictorNet(nn.Module):
    """Predict hash160 bit pattern from private key"""
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 160),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

class EntropyAnalysisNet(nn.Module):
    """Detect entropy anomalies in hash160 outputs"""
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(160, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2)
        )
        self.decoder = nn.Sequential(
            nn.Linear(64, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 160),
            nn.Sigmoid()
        )
        self.anomaly_detector = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        anomaly_score = self.anomaly_detector(encoded)
        return decoded, anomaly_score, encoded

# === Block 4: Adaptive Mutation Engine ===
bit_bias = defaultdict(float)
bad_bit_bias = defaultdict(float)  # For high-Hamming keys
xor_chain = []
mutation_log = []
recovered_keys = {}
mutation_bank = []

def update_bit_bias(flips: list, reward: float = 1.0, is_good: bool = True):
    """Reinforce bit positions for low (good) or high (bad) Hamming distances"""
    target = bit_bias if is_good else bad_bit_bias
    for bit in flips:
        target[bit] += reward

def add_xor_delta(prev_k: int, new_k: int, reward: float = 1.0):
    """Store successful XOR mutations"""
    delta = prev_k ^ new_k
    xor_chain.append((delta, reward))
    if len(xor_chain) > 100:
        xor_chain.pop(0)

def log_mutation_event(k_old, k_new, mutation_type, flips, delta_hamming):
    """Log mutation outcomes"""
    entry = {
        "time": time.time(),
        "type": mutation_type,
        "delta_hamming": delta_hamming,
        "flips": flips,
        "delta_k": k_old ^ k_new
    }
    mutation_log.append(entry)
    if len(mutation_log) > 5000:
        mutation_log.pop(0)

def record_mutation_result(mutation_vector: int, delta_hamming: int):
    """Track mutation effectiveness"""
    found = False
    for i, (vec, score) in enumerate(mutation_bank):
        if vec == mutation_vector:
            mutation_bank[i] = (vec, score + delta_hamming)
            found = True
            break
    if not found:
        mutation_bank.append((mutation_vector, delta_hamming))
    if len(mutation_bank) > 100:
        mutation_bank.sort(key=lambda x: -x[1])
        mutation_bank[:] = mutation_bank[:100]

def get_top_bit_bias(n=10, is_good: bool = True):
    """Get highest-scoring bit positions"""
    target = bit_bias if is_good else bad_bit_bias
    return sorted(target.items(), key=lambda x: -x[1])[:n]

def get_top_xor_deltas(n=10):
    """Get highest-reward XOR deltas"""
    return sorted(xor_chain, key=lambda x: -x[1])[:n]

def get_best_mutations(n=10):
    """Get top-performing mutation vectors"""
    return sorted(mutation_bank, key=lambda x: -x[1])[:n]

def mutate_key(k: int) -> int:
    """Adaptive key mutation with reinforcement learning"""
    new_k = k
    if random.random() < 0.5:
        new_k += random.randint(-1000000, 1000000)  # ±10^6 range
    for vec, _ in get_best_mutations(n=5):
        if random.random() < 0.3:
            new_k ^= vec
    for delta, _ in get_top_xor_deltas(n=5):
        if random.random() < 0.3:
            new_k ^= delta
    for bit, _ in get_top_bit_bias(n=5, is_good=True):
        if random.random() < 0.3:
            new_k ^= (1 << bit)
    for bit, _ in get_top_bit_bias(n=5, is_good=False):
        if random.random() < 0.1:
            new_k ^= (1 << bit)
    if random.random() < 0.2:
        new_k ^= random.getrandbits(8)
    return new_k if 1 <= new_k < MAX_K else random.randint(1, MAX_K - 1)

def generate_key(seed_k: int, transformer_model=None) -> int:
    """Generate key with neural guidance"""
    k = mutate_key(seed_k)
    if transformer_model:
        try:
            with torch.no_grad():
                k_tensor = key_to_vector(k).unsqueeze(0)
                prediction = transformer_model(k_tensor).squeeze(0)
                for i in range(160):
                    if prediction[i].item() > 0.95 and random.random() < 0.5:
                        k ^= (1 << (i % 160))
        except:
            pass
    return k if 1 <= k < MAX_K else random.randint(1, MAX_K - 1)

# === Block 5: Address and Target Management ===
def load_addresses() -> list:
    """Load and validate P2PKH addresses from file with improved error handling"""
    possible_paths = [ADDR_FILE] + [f"{ADDR_FILE[:-4]} ({i}).txt" for i in range(1, 6)]
    selected_path = next((p for p in possible_paths if os.path.exists(p)), None)

    if not selected_path:
        logging.error(f"No address file found at {ADDR_FILE} or alternatives.")
        raise FileNotFoundError("addresses.txt not found")

    targets = []
    with open(selected_path, 'r') as f:
        for line in f:
            addr = line.strip()
            if not addr:  # Skip empty lines
                continue
            try:
                decoded = base58.b58decode(addr)
                if len(decoded) < 25:  # Version (1) + hash160 (20) + checksum (4)
                    logging.warning(f"Decoded address too short: {addr}, length: {len(decoded)}")
                    continue
                h160 = decoded[1:-4]  # Skip version and checksum
                if len(h160) != 20:
                    logging.warning(f"Invalid hash160 length for {addr}: {len(h160)}")
                    continue
                targets.append((addr, h160))
                logging.info(f"Successfully loaded address: {addr}")
            except Exception as e:
                logging.warning(f"Invalid address skipped: {addr}, error: {e}")
                continue

    if len(targets) == 0:
        logging.error("No valid P2PKH addresses found in file.")
        raise ValueError("No valid addresses loaded")

    logging.info(f"Loaded {len(targets)} valid P2PKH addresses from {selected_path}")
    return targets

target_stats = {}
recovered_keys = {}
mutation_bank = []

def init_target_stats(targets):
    """Initialize target statistics"""
    for addr, _ in targets:
        target_stats[addr] = {
            "best": 160,
            "best_k": -1,
            "scans": 0,
            "hamming_history": [],
            "last_improvement": time.time()
        }

def update_target_stat(addr: str, k: int, ham: int, compressed: bool):
    """Update target statistics and save recovered keys"""
    if len(target_stats[addr]["hamming_history"]) >= 1000:
        target_stats[addr]["hamming_history"].pop(0)
    target_stats[addr]["hamming_history"].append(ham)

    if ham < target_stats[addr]["best"]:
        target_stats[addr]["best"] = ham
        target_stats[addr]["best_k"] = k
        target_stats[addr]["last_improvement"] = time.time()
        logging.info(f"New best for {addr}: Hamming {ham}, Key {hex(k)}, Compressed: {compressed}")
        if ham == 0:
            recovered_keys[addr] = {"key": hex(k), "compressed": compressed}
            with open(RECOVERED_KEYS_PATH, 'w') as f:
                json.dump(recovered_keys, f, indent=2)
            with open(DRIVE_RECOVERED_KEYS_PATH, 'w') as f:
                json.dump(recovered_keys, f, indent=2)
            logging.info(f"Recovered key for {addr}: {hex(k)}, Compressed: {compressed}")

    target_stats[addr]["scans"] += 1

def get_priority_targets(targets, top_n=PRIORITY_TARGETS):
    """Prioritize targets with Hamming ≤50"""
    if not target_stats:
        return random.sample(targets, min(top_n, len(targets)))
    sorted_targets = sorted(
        target_stats.items(),
        key=lambda x: (x[1]["best"], -x[1]["last_improvement"])
    )
    top_addrs = set(addr for addr, _ in sorted_targets[:top_n] if target_stats[addr]["best"] <= 50)
    return [(addr, h) for addr, h in targets if addr in top_addrs]

# === Block 6: Statistical Analysis ===
def compute_bit_bias_confidence_intervals(is_good: bool = True):
    """Calculate confidence intervals for bit biases"""
    target = bit_bias if is_good else bad_bit_bias
    result = defaultdict(lambda: {"bias": 0, "proportion": 0, "ci_lower": 0, "ci_upper": 0, "significant": False})
    total_observations = sum(target.values()) or 1
    for bit, value in target.items():
        proportion = value / total_observations
        lower, upper = proportion_confint(value, total_observations, alpha=0.05, method='wilson')
        result[bit] = {
            "bias": value,
            "proportion": proportion,
            "ci_lower": lower,
            "ci_upper": upper,
            "significant": lower > EXPECTED_PROB or upper < EXPECTED_PROB
        }
    return result

def save_bit_bias_heatmap(is_good: bool = True):
    """Visualize bit bias distribution"""
    target = bit_bias if is_good else bad_bit_bias
    if not target:
        return
    bias_data = compute_bit_bias_confidence_intervals(is_good)
    bits = sorted(bias_data.keys())
    values = [bias_data[bit]["bias"] for bit in bits]
    lower_ci = [bias_data[bit]["ci_lower"] for bit in bits]
    upper_ci = [bias_data[bit]["ci_upper"] for bit in bits]

    plt.figure(figsize=(16, 6))
    plt.bar(bits, values, alpha=0.7)
    plt.errorbar(bits, values, yerr=[(v-l) for v, l in zip(values, lower_ci)],
                 fmt='none', ecolor='black', capsize=3)
    for bit in bits:
        if bias_data[bit]["significant"]:
            plt.plot(bit, bias_data[bit]["bias"], 'r*', markersize=10)
    plt.title(f"{'Good' if is_good else 'Bad'} Key Bit Bias Heatmap with 95% Confidence Intervals")
    plt.xlabel("Bit Position (0–159)")
    plt.ylabel("Reinforcement Score")
    plt.grid(True, alpha=0.3)
    plt.axhline(y=sum(values)/len(values) if values else 0, color='r', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, "figures", f"{'good' if is_good else 'bad'}_bit_bias_heatmap.png"), dpi=300)
    plt.savefig(os.path.join(DRIVE_DIR, "figures", f"{'good' if is_good else 'bad'}_bit_bias_heatmap.png"), dpi=300)
    plt.close()

def plot_loss_curve(losses, label: str):
    """Plot loss curve for neural models"""
    if not losses:
        return
    plt.figure(figsize=(10, 6))
    plt.plot(losses)
    if len(losses) > 10:
        window_size = min(len(losses) // 10, 20)
        smoothed = np.convolve(losses, np.ones(window_size)/window_size, mode='valid')
        plt.plot(range(window_size-1, len(losses)), smoothed, 'r-', linewidth=2)
    plt.title(f"{label} Loss Curve")
    plt.xlabel("Batch")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, "figures", f"{label.lower().replace(' ', '_')}_loss.png"), dpi=300)
    plt.savefig(os.path.join(DRIVE_DIR, "figures", f"{label.lower().replace(' ', '_')}_loss.png"), dpi=300)
    plt.close()

def create_bit_correlation_matrix(hash160_vectors, save_path=None):
    """Create correlation matrix for bit positions"""
    if not hash160_vectors or len(hash160_vectors) < 10:
        return None
    matrix = torch.stack(hash160_vectors).cpu().numpy()
    corr_matrix = np.corrcoef(matrix.T)
    plt.figure(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    cmap = sns.diverging_palette(230, 20, as_cmap=True)
    sns.heatmap(corr_matrix, mask=mask, cmap=cmap, vmax=0.3, vmin=-0.3,
                center=0, square=True, linewidths=.5)
    plt.title('Bit Position Correlation Matrix')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.savefig(save_path.replace(BASE_DIR, DRIVE_DIR), dpi=300)
        plt.close()
    else:
        plt.show()
    return corr_matrix

def visualize_hash_clusters(hash_vectors, save_path=None, method='tsne'):
    """Visualize hash clusters with dimensionality reduction"""
    if not hash_vectors or len(hash_vectors) < 10:
        return
    data = torch.stack(hash_vectors).cpu().numpy()
    if method.lower() == 'tsne':
        reducer = TSNE(n_components=2, random_state=RNG_SEED)
    elif method.lower() == 'umap':
        reducer = umap.UMAP(random_state=RNG_SEED)
    else:
        reducer = PCA(n_components=2, random_state=RNG_SEED)
    reduced_data = reducer.fit_transform(data)
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    clusters = dbscan.fit_predict(reduced_data)
    plt.figure(figsize=(12, 10))
    scatter = plt.scatter(reduced_data[:, 0], reduced_data[:, 1], c=clusters,
                         cmap='viridis', s=50, alpha=0.7)
    plt.legend(*scatter.legend_elements(), title="Clusters")
    plt.title(f'Hash160 Clusters ({method.upper()})')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.savefig(save_path.replace(BASE_DIR, DRIVE_DIR), dpi=300)
        plt.close()
    else:
        plt.show()
    return {'reduced_data': reduced_data, 'clusters': clusters}

def create_3d_bit_position_map(hash160_samples, save_path=None):
    """Create 3D visualization of bit position relationships"""
    if not hash160_samples or len(hash160_samples) < 100:
        return
    data = torch.stack(hash160_samples).cpu().numpy()
    bit_variance = np.var(data, axis=0)
    top_bits = np.argsort(bit_variance)[-3:]
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')
    x = data[:, top_bits[0]]
    y = data[:, top_bits[1]]
    z = data[:, top_bits[2]]
    colors = np.sum(data, axis=1)
    scatter = ax.scatter(x, y, z, c=colors, cmap='viridis', s=30, alpha=0.7)
    cbar = plt.colorbar(scatter)
    cbar.set_label('Bit Sum (Entropy Proxy)')
    ax.set_xlabel(f'Bit {top_bits[0]}')
    ax.set_ylabel(f'Bit {top_bits[1]}')
    ax.set_zlabel(f'Bit {top_bits[2]}')
    plt.title('3D Relationship Between High-Variance Bit Positions')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.savefig(save_path.replace(BASE_DIR, DRIVE_DIR), dpi=300)
        plt.close()
    else:
        plt.show()

def analyze_target_stats():
    """Analyze target performance with trend detection"""
    from scipy import stats
    if not target_stats:
        return {}

    analysis = {
        "global": {
            "best_hamming": min([s["best"] for s in target_stats.values()]),
            "avg_hamming": sum([s["best"] for s in target_stats.values()]) / len(target_stats),
            "total_scans": sum([s["scans"] for s in target_stats.values()]),
            "timestamp": time.time()
        },
        "targets": {}
    }

    for addr, stats in target_stats.items():
        if not stats["hamming_history"]:
            continue
        history = stats["hamming_history"]
        analysis["targets"][addr] = {
            "best": stats["best"],
            "mean": np.mean(history),
            "median": np.median(history),
            "std_dev": np.std(history),
            "scans": stats["scans"]
        }
        if len(history) > 10:
            x = np.arange(len(history))
            y = np.array(history)
            try:
                print(f"Debug: Analyzing {addr}, x type: {type(x)}, y type: {type(y)}, len(x): {len(x)}, len(y): {len(y)}")
                slope, _, r_value, p_value, _ = stats.linregress(x, y)
                analysis["targets"][addr].update({
                    "trend_slope": slope,
                    "trend_r_squared": r_value**2,
                    "trend_p_value": p_value
                })
            except Exception as e:
                print(f"Error in linregress for {addr}: {e}")
                analysis["targets"][addr].update({
                    "trend_slope": 0.0,
                    "trend_r_squared": 0.0,
                    "trend_p_value": 1.0
                })

    with open(os.path.join(REPORT_PATH, "target_analysis.json"), 'w') as f:
        json.dump(analysis, f, indent=2)
    with open(os.path.join(DRIVE_DIR, "reports", "target_analysis.json"), 'w') as f:
        json.dump(analysis, f, indent=2)
    return analysis

class NBISTTestSuite:
    """NIST-based statistical test suite for randomness"""
    def __init__(self):
        self.tests = {
            "frequency": self.frequency_test,
            "frequency_block": self.block_frequency_test,
            "runs": self.runs_test,
            "longest_run": self.longest_run_test,
            "serial": self.serial_test,
            "entropy": self.approximate_entropy_test
        }

    def frequency_test(self, bits):
        n = len(bits)
        s = sum([(2*bit-1) for bit in bits])
        s_obs = abs(s) / np.sqrt(n)
        p_value = math.erfc(s_obs / np.sqrt(2))
        return p_value

    def block_frequency_test(self, bits, block_size=10):
        n = len(bits)
        n_blocks = n // block_size
        if n_blocks == 0:
            return 1.0
        blocks = [bits[i:i+block_size] for i in range(0, n_blocks*block_size, block_size)]
        proportions = [sum(block)/block_size for block in blocks]
        chi_squared = 4.0 * block_size * sum([(prop-0.5)**2 for prop in proportions])
        p_value = stats.chi2.sf(chi_squared, n_blocks)
        return p_value

    def runs_test(self, bits):
        n = len(bits)
        if n < 100:
            return 1.0
        prop = sum(bits) / n
        tau = 2.0 / np.sqrt(n)
        if abs(prop - 0.5) >= tau:
            return 0.0
        runs = 1
        for i in range(1, n):
            if bits[i] != bits[i-1]:
                runs += 1
        p_value = math.erfc(abs(runs - 2*n*prop*(1-prop)) /
                          (2*np.sqrt(2*n)*prop*(1-prop)))
        return p_value

    def longest_run_test(self, bits):
        n = len(bits)
        if n < 128:
            return 1.0
        if n < 6272:
            k, m = 3, 8
            v = [1, 2, 3, 4]
            pi = [0.2148, 0.3672, 0.2305, 0.1875]
        else:
            k, m = 5, 128
            v = [4, 5, 6, 7, 8, 9]
            pi = [0.1174, 0.2430, 0.2493, 0.1752, 0.1027, 0.1124]
        num_blocks = n // m
        longest_runs = []
        for i in range(num_blocks):
            block = bits[i*m:(i+1)*m]
            max_run = 0
            current_run = 0
            for bit in block:
                if bit == 1:
                    current_run += 1
                    max_run = max(max_run, current_run)
                else:
                    current_run = 0
            longest_runs.append(max_run)
        frequencies = [0] * (k+1)
        for run in longest_runs:
            if run <= v[0]:
                frequencies[0] += 1
            elif run >= v[-1]:
                frequencies[-1] += 1
            else:
                for j in range(1, len(v)):
                    if v[j-1] < run <= v[j]:
                        frequencies[j] += 1
                        break
        chi_squared = sum([(frequencies[i] - num_blocks*pi[i])**2 / (num_blocks*pi[i])
                          for i in range(len(frequencies))])
        p_value = stats.chi2.sf(chi_squared, k)
        return p_value

    def serial_test(self, bits, m=3):
        n = len(bits)
        if n < 2**m:
            return 1.0
        extended_bits = bits + bits[:m-1]
        pattern_counts1 = defaultdict(int)
        for i in range(n):
            pattern = tuple(extended_bits[i:i+m])
            pattern_counts1[pattern] += 1
        pattern_counts2 = defaultdict(int)
        for i in range(n):
            pattern = tuple(extended_bits[i:i+m-1])
            pattern_counts2[pattern] += 1
        pattern_counts3 = defaultdict(int)
        for i in range(n):
            pattern = tuple(extended_bits[i:i+m-2])
            pattern_counts3[pattern] += 1
        psi_sq_m = sum([count**2 for count in pattern_counts1.values()]) * 2**m / n - n
        psi_sq_m1 = sum([count**2 for count in pattern_counts2.values()]) * 2**(m-1) / n - n
        psi_sq_m2 = sum([count**2 for count in pattern_counts3.values()]) * 2**(m-2) / n - n
        delta1 = psi_sq_m - psi_sq_m1
        delta2 = psi_sq_m - 2*psi_sq_m1 + psi_sq_m2
        p_value1 = stats.chi2.sf(delta1, 2**(m-1))
        p_value2 = stats.chi2.sf(delta2, 2**(m-2))
        return min(p_value1, p_value2)

    def approximate_entropy_test(self, bits, m=5):
        n = len(bits)
        if n < 100:
            return 1.0
        def phi(m_val):
            counts = defaultdict(int)
            for i in range(n):
                pattern = tuple(bits[(i+j) % n] for j in range(m_val))
                counts[pattern] += 1
            c_m = [count/n for count in counts.values()]
            return sum([p * np.log(p) for p in c_m if p > 0])
        apen = phi(m) - phi(m+1)
        chi_squared = 2 * n * (np.log(2) - apen)
        p_value = stats.chi2.sf(chi_squared, 2**m)
        return p_value

    def run_all_tests(self, bits):
        results = {}
        for test_name, test_func in self.tests.items():
            p_value = test_func(bits)
            passed = p_value >= 0.01
            conf_level = 0.95
            z = stats.norm.ppf(1 - (1 - conf_level) / 2)
            if passed:
                lower_ci = (1 + 1/(4*1) - z/(2*np.sqrt(1)))
                upper_ci = 1.0
            else:
                lower_ci = 0.0
                upper_ci = z/(2*np.sqrt(1))
            results[test_name] = {
                "p_value": p_value,
                "passed": passed,
                "confidence_lower": lower_ci,
                "confidence_upper": upper_ci
            }
        return results

def run_entropy_scan(sample_size: int = 100_000, reference_hashes: list = [],
                    statistical_rigor: bool = True):
    """Scan keyspace for entropy anomalies"""
    global entropy_counter, entropy_total, entropy_samples
    print(f"Running anomaly detection scan with {sample_size:,} keys...")
    entropy_counter = np.zeros(160, dtype=int)
    entropy_total = 0
    entropy_samples = 0
    hamming_distribution = []
    hash_samples = []
    bit_vectors = []

    for _ in tqdm(range(sample_size), desc="Entropy Scan"):
        k = random.randint(1, MAX_K)
        for compressed in [True, False]:
            h160 = privkey_to_hash160(k, compressed)
            if not h160:
                continue
            entropy_samples += 1
            for i in range(20):
                byte = h160[i]
                for j in range(8):
                    bit_index = (i * 8) + (7 - j)
                    if byte & (1 << j):
                        entropy_counter[bit_index] += 1
            entropy_total += 1
            hash_samples.append(h160)
            bit_vectors.append(hash160_to_vector(h160))
            for _, ref in reference_hashes:
                dist, _ = hamming_distance_and_flips(h160, ref)
                hamming_distribution.append(dist)

    plt.figure(figsize=(16, 6))
    proportions = entropy_counter / max(1, entropy_samples)
    confidence = 0.95
    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    lower_ci = []
    upper_ci = []
    for i in range(160):
        p = proportions[i]
        n = entropy_samples
        denominator = 1 + z**2/n
        centre_adjusted_prob = (p + z**2/(2*n))/denominator
        adjusted_standard_dev = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2))/denominator
        lower = max(0, centre_adjusted_prob - adjusted_standard_dev)
        upper = min(1, centre_adjusted_prob + adjusted_standard_dev)
        lower_ci.append(lower)
        upper_ci.append(upper)
    plt.bar(range(160), proportions, alpha=0.7)
    plt.errorbar(range(160), proportions,
                yerr=[proportions-lower_ci, upper_ci-proportions],
                fmt='none', ecolor='black', capsize=2, alpha=0.3)
    plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Expected (0.5)')
    for i in range(160):
        if lower_ci[i] > 0.5 or upper_ci[i] < 0.5:
            plt.plot(i, proportions[i], 'r*', markersize=8)
    plt.title(f"RIPEMD-160 Bitwise Distribution with {confidence*100:.0f}% Confidence Intervals")
    plt.xlabel("Bit Position (0–159)")
    plt.ylabel("Frequency of 1s")
    plt.ylim(0.45, 0.55)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, "figures", "hash160_entropy_profile.png"), dpi=300)
    plt.savefig(os.path.join(DRIVE_DIR, "figures", "hash160_entropy_profile.png"), dpi=300)
    plt.close()

    if hamming_distribution:
        plt.figure(figsize=(12, 8))
        hist, bin_edges = np.histogram(hamming_distribution, bins=40)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        plt.bar(bin_centers, hist, width=(bin_edges[1]-bin_edges[0])*0.8,
               color='steelblue', edgecolor='black', alpha=0.7)
        x = np.arange(40, 120)
        n = 160
        p = 0.5
        expected = stats.binom.pmf(x, n, p) * sum(hist) * (bin_edges[1]-bin_edges[0])
        plt.plot(x, expected, 'r-', linewidth=2,
                label='Expected (Binomial Distribution)')
        mean_ham = np.mean(hamming_distribution)
        std_ham = np.std(hamming_distribution)
        expected_mean = n * p
        expected_std = np.sqrt(n * p * (1-p))
        plt.axvline(x=mean_ham, color='blue', linestyle='--',
                   label=f'Observed Mean: {mean_ham:.2f}')
        plt.axvline(x=expected_mean, color='red', linestyle='--',
                   label=f'Expected Mean: {expected_mean:.2f}')
        ks_stat, p_value = stats.kstest(hamming_distribution, stats.binom.cdf, args=(n, p))
        plt.title(f'Hamming Distances to Reference Hash160s\nKS-test: D={ks_stat:.4f}, p={p_value:.6f}')
        plt.xlabel('Hamming Distance (bits)')
        plt.ylabel('Frequency')
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(BASE_DIR, "figures", "hamming_histogram.png"), dpi=300)
        plt.savefig(os.path.join(DRIVE_DIR, "figures", "hamming_histogram.png"), dpi=300)
        plt.close()

    if statistical_rigor and len(bit_vectors) > 1000:
        print("Performing advanced statistical analysis...")
        bit_array = np.stack([v.cpu().numpy() for v in bit_vectors])
        test_results = {}
        nist_suite = NBISTTestSuite()
        for bit_pos in tqdm(range(160), desc="Testing bit positions"):
            bits = bit_array[:, bit_pos].astype(int).tolist()
            test_results[bit_pos] = nist_suite.run_all_tests(bits)
        with open(os.path.join(BASE_DIR, "reports", "nist_test_results.json"), 'w') as f:
            json.dump(test_results, f, indent=2)
        with open(os.path.join(DRIVE_DIR, "reports", "nist_test_results.json"), 'w') as f:
            json.dump(test_results, f, indent=2)
        summary = {
            "failed_tests": 0,
            "total_tests": 0,
            "significant_bits": []
        }
        for bit_pos, results in test_results.items():
            for test_name, result in results.items():
                summary["total_tests"] += 1
                if not result["passed"]:
                    summary["failed_tests"] += 1
                    summary["significant_bits"].append({
                        "bit": bit_pos,
                        "test": test_name,
                        "p_value": result["p_value"]
                    })
        with open(os.path.join(BASE_DIR, "reports", "nist_summary.json"), 'w') as f:
            json.dump(summary, f, indent=2)
        with open(os.path.join(DRIVE_DIR, "reports", "nist_summary.json"), 'w') as f:
            json.dump(summary, f, indent=2)

    return {
        'hash_samples': hash_samples,
        'bit_vectors': bit_vectors,
        'hamming_distribution': hamming_distribution
    }

# === Block 7: GPU-Accelerated Key Processing ===
def process_key_batch(keys: list, targets: list) -> list:
    """GPU-accelerated key processing with parallelization, testing compressed/uncompressed"""
    results = []
    target_h160s = [h for _, h in targets]

    def process_key(k):
        batch_results = []
        for compressed in [True, False]:
            h160 = privkey_to_hash160(k, compressed)
            if h160:
                for addr, target_h160 in targets:
                    ham, flips = hamming_distance_and_flips(h160, target_h160)
                    batch_results.append((addr, k, h160, ham, flips, compressed))
        return batch_results

    if device.type == "cuda":
        try:
            keys_array = cp.array(keys, dtype=cp.uint64)
            target_h160s_array = cp.array(target_h160s, dtype=cp.uint8)
            for k in keys:
                for compressed in [True, False]:
                    h160 = privkey_to_hash160(k, compressed)
                    if h160:
                        h160_array = cp.array(list(h160), dtype=cp.uint8)
                        for addr, target_h160 in targets:
                            target_array = cp.array(list(target_h160), dtype=cp.uint8)
                            ham = cp.sum(h160_array != target_array).get()
                            _, flips = hamming_distance_and_flips(h160, target_h160)
                            results.append((addr, k, h160, ham, flips, compressed))
        except cp.cuda.memory.OutOfMemoryError:
            logging.warning("GPU memory error, falling back to CPU for this batch")
            with ThreadPoolExecutor() as executor:
                batch_results = executor.map(process_key, keys)
                for batch in batch_results:
                    results.extend(batch)
    else:
        with ThreadPoolExecutor() as executor:
            batch_results = executor.map(process_key, keys)
            for batch in batch_results:
                results.extend(batch)

    return results

# === Block 8: Checkpointing and Signal Handling ===
def save_checkpoint(models: dict, step_count: int):
    """Save models and state to local and Drive"""
    try:
        for model_name, model in models.items():
            torch.save(model.state_dict(), os.path.join(CHECKPOINT_PATH, f"{model_name}.pt"))
            torch.save(model.state_dict(), os.path.join(DRIVE_CHECKPOINT_PATH, f"{model_name}.pt"))

        state = {
            "bit_bias": dict(bit_bias),
            "bad_bit_bias": dict(bad_bit_bias),
            "xor_chain": xor_chain,
            "mutation_bank": [(hex(vec), score) for vec, score in mutation_bank],
            "step": step_count,
            "timestamp": time.time(),
            "config": CONFIG
        }
        torch.save(state, os.path.join(CHECKPOINT_PATH, "research_state.pt"))
        torch.save(state, os.path.join(DRIVE_CHECKPOINT_PATH, "research_state.pt"))
        with open(os.path.join(CHECKPOINT_PATH, "bit_bias.json"), 'w') as f:
            json.dump(dict(bit_bias), f)
        with open(os.path.join(DRIVE_CHECKPOINT_PATH, "bit_bias.json"), 'w') as f:
            json.dump(dict(bit_bias), f)
        with open(os.path.join(CHECKPOINT_PATH, "bad_bit_bias.json"), 'w') as f:
            json.dump(dict(bad_bit_bias), f)
        with open(os.path.join(DRIVE_CHECKPOINT_PATH, "bad_bit_bias.json"), 'w') as f:
            json.dump(dict(bad_bit_bias), f)
        with open(os.path.join(CHECKPOINT_PATH, "xor_chain.json"), 'w') as f:
            json.dump(xor_chain, f)
        with open(os.path.join(DRIVE_CHECKPOINT_PATH, "xor_chain.json"), 'w') as f:
            json.dump(xor_chain, f)
        with open(os.path.join(CHECKPOINT_PATH, "mutation_bank.json"), 'w') as f:
            json.dump([(hex(vec), score) for vec, score in mutation_bank], f)
        with open(os.path.join(DRIVE_CHECKPOINT_PATH, "mutation_bank.json"), 'w') as f:
            json.dump([(hex(vec), score) for vec, score in mutation_bank], f)
        with open(os.path.join(CHECKPOINT_PATH, "step.txt"), 'w') as f:
            f.write(str(step_count))
        with open(os.path.join(DRIVE_CHECKPOINT_PATH, "step.txt"), 'w') as f:
            f.write(str(step_count))

        log_event("checkpoint", {"step": step_count, "recovered_keys": len(recovered_keys)})
    except Exception as e:
        logging.error(f"Checkpoint save failed: {e}")

def load_checkpoint(models: dict) -> int:
    """Load prior state"""
    try:
        state_path = os.path.join(CHECKPOINT_PATH, "research_state.pt")
        step = 0
        if os.path.exists(state_path):
            state = torch.load(state_path, map_location=device)
            bit_bias.update(state.get("bit_bias", {}))
            bad_bit_bias.update(state.get("bad_bit_bias", {}))
            xor_chain.extend(state.get("xor_chain", []))
            mutation_bank.extend([(int(vec, 16), score) for vec, score in state.get("mutation_bank", [])])
            step = state.get("step", 0)
            for model_name, model in models.items():
                model_path = os.path.join(CHECKPOINT_PATH, f"{model_name}.pt")
                if os.path.exists(model_path):
                    model.load_state_dict(torch.load(model_path, map_location=device))
            logging.info(f"Loaded consolidated state from step {step}")
        return step
    except Exception as e:
        logging.warning(f"No checkpoint loaded: {e}")
        return 0

def log_event(event_type: str, payload: dict):
    """Log structured events"""
    try:
        entry = {
            "time": time.time(),
            "event": event_type,
            "data": payload
        }
        with open(LOG_PATH, 'a') as f:
            f.write(json.dumps(entry) + '\n')
        with open(os.path.join(DRIVE_DIR, "logs", "telemetry.jsonl"), 'a') as f:
            f.write(json.dumps(entry) + '\n')
    except Exception as e:
        print(f"Failed to log event: {e}")

# === Block 9: Signal Handler for Interruptions ===
def signal_handler(sig, frame, models, step):
    """Handle interruptions with immediate checkpointing"""
    print(f"Received interrupt signal {sig}, saving checkpoint at step {step}...")
    save_checkpoint(models, step)
    logging.info(f"Interrupted at step {step}, checkpoint saved")
    sys.exit(0)

# === Block 10: Main Recovery Loop ===
def run_phantom_t5(targets, max_steps=250_000):
    """Main recovery loop with adaptive learning of good and bad keys"""
    global bit_bias, bad_bit_bias, xor_chain, mutation_log, recovered_keys

    # Initialize models
    models = {
        'predictor_net': HashPredictorNet().to(device),
        'entropy_net': EntropyAnalysisNet().to(device)
    }
    optimizers = {
        'predictor_net': torch.optim.Adam(models['predictor_net'].parameters(), lr=0.001),
        'entropy_net': torch.optim.Adam(models['entropy_net'].parameters(), lr=0.001)
    }
    losses = {'predictor': [], 'entropy': []}

    # Load checkpoint
    start_step = load_checkpoint(models)

    # Initialize target stats
    init_target_stats(targets)

    # Tracking variables
    best_hamming = 160
    good_hash_samples = []  # Hamming ≤50
    bad_hash_samples = []   # Hamming >80
    good_hash_vectors = []
    bad_hash_vectors = []
    anomalies_found = 0
    last_checkpoint_time = time.time()
    start_time = time.time()
    keys_processed = 0

    # Set up signal handler
    signal.signal(signal.SIGINT, lambda sig, frame: signal_handler(sig, frame, models, start_step))
    signal.signal(signal.SIGTERM, lambda sig, frame: signal_handler(sig, frame, models, start_step))

    logging.info(f"Starting recovery with {len(targets)} targets, {max_steps} steps, device: {device}")
    print(f"Starting RIPEMPHANTOM-T5 with {len(targets)} targets on {device}...")

    try:
        for step in tqdm(range(start_step, max_steps), desc="Recovery Progress"):
            # Progress logging
            if step % 100 == 0:
                elapsed = time.time() - start_time
                steps_per_sec = (step - start_step) / max(1, elapsed)
                keys_per_sec = keys_processed / max(1, elapsed)
                eta = (max_steps - step) / max(1, steps_per_sec) / 3600
                print(f"Step {step}: Processed {len(good_hash_samples)} good samples, "
                      f"{len(bad_hash_samples)} bad samples, Best Hamming: {best_hamming}, "
                      f"Recovered: {len(recovered_keys)}, Keys/sec: {keys_per_sec:.2f}, ETA: {eta:.2f} hours")

            # Select priority targets
            current_targets = get_priority_targets(targets)
            if not current_targets:
                current_targets = random.sample(targets, min(5, len(targets)))

            # Generate keys
            seed_k = random.choice(SEED_KEYS + [random.randint(1, MAX_K)])
            keys = [generate_key(seed_k, models['predictor_net']) for _ in range(BATCH_SIZE)]

            # Process keys
            batch_results = process_key_batch(keys, current_targets)
            keys_processed += len(keys) * 2  # Compressed and uncompressed
            if not batch_results:
                continue

            # Update stats and train models
            good_key_vectors = []
            good_hash_vectors_batch = []
            bad_key_vectors = []
            bad_hash_vectors_batch = []
            for addr, k, h160, ham, flips, compressed in batch_results:
                update_target_stat(addr, k, ham, compressed)
                best_hamming = min(best_hamming, ham)
                if ham <= 50 or ham == 0:
                    anomalies_found += 1
                    reward = 1.0 / max(1, ham)  # Higher reward for lower Hamming
                    update_bit_bias(flips, reward, is_good=True)
                    add_xor_delta(seed_k, k, reward)
                    record_mutation_result(k ^ seed_k, 160 - ham)
                    log_mutation_event(seed_k, k, "targeted", flips, ham - target_stats[addr]["best"])
                    if len(good_hash_samples) < MAX_GOOD_SAMPLES:
                        good_hash_samples.append(h160)
                        good_hash_vectors.append(hash160_to_vector(h160))
                        good_key_vectors.append(key_to_vector(k))
                        good_hash_vectors_batch.append(hash160_to_vector(h160))
                elif ham > 80:
                    reward = 1.0 / max(1, 160 - ham)  # Reward for avoiding high Hamming
                    update_bit_bias(flips, reward, is_good=False)
                    if len(bad_hash_samples) < MAX_BAD_SAMPLES:
                        bad_hash_samples.append(h160)
                        bad_hash_vectors.append(hash160_to_vector(h160))
                        bad_key_vectors.append(key_to_vector(k))
                        bad_hash_vectors_batch.append(hash160_to_vector(h160))

            # Train models on balanced good/bad data
            if good_hash_vectors_batch or bad_hash_vectors_batch:
                all_key_vectors = good_key_vectors + bad_key_vectors[:len(good_key_vectors)]  # Balance dataset
                all_hash_vectors = good_hash_vectors_batch + bad_hash_vectors_batch[:len(good_hash_vectors_batch)]
                if all_key_vectors:
                    key_vectors = torch.stack(all_key_vectors)
                    true_hash = torch.stack(all_hash_vectors)

                    optimizers['predictor_net'].zero_grad()
                    pred_hash = models['predictor_net'](key_vectors)
                    loss_pred = F.binary_cross_entropy(pred_hash, true_hash)
                    loss_pred.backward()
                    optimizers['predictor_net'].step()
                    losses['predictor'].append(loss_pred.item())

                    optimizers['entropy_net'].zero_grad()
                    decoded, anomaly_score, _ = models['entropy_net'](true_hash)
                    recon_loss = F.binary_cross_entropy(decoded, true_hash)
                    anomaly_loss = torch.mean(anomaly_score)
                    loss_entropy = recon_loss + 0.1 * anomaly_loss
                    loss_entropy.backward()
                    optimizers['entropy_net'].step()
                    losses['entropy'].append(loss_entropy.item())

            # Periodic checkpointing and analysis
            current_time = time.time()
            if (step + 1) % CHECKPOINT_STEPS == 0 or (current_time - last_checkpoint_time) > CHECKPOINT_INTERVAL:
                save_bit_bias_heatmap(is_good=True)
                save_bit_bias_heatmap(is_good=False)
                plot_loss_curve(losses['predictor'], "Hash Predictor")
                plot_loss_curve(losses['entropy'], "Entropy Analysis")
                if len(good_hash_vectors) >= 1000:
                    create_bit_correlation_matrix(good_hash_vectors[-1000:],
                                                os.path.join(BASE_DIR, "figures", f"good_correlation_matrix_step_{step+1}.png"))
                    visualize_hash_clusters(good_hash_vectors[-1000:],
                                          os.path.join(BASE_DIR, "figures", f"good_hash_clusters_step_{step+1}.png"))
                    create_3d_bit_position_map(good_hash_vectors[-1000:],
                                              os.path.join(BASE_DIR, "figures", f"good_3d_bit_map_step_{step+1}.png"))
                if len(bad_hash_vectors) >= 1000:
                    create_bit_correlation_matrix(bad_hash_vectors[-1000:],
                                                os.path.join(BASE_DIR, "figures", f"bad_correlation_matrix_step_{step+1}.png"))
                    visualize_hash_clusters(bad_hash_vectors[-1000:],
                                          os.path.join(BASE_DIR, "figures", f"bad_hash_clusters_step_{step+1}.png"))
                    create_3d_bit_position_map(bad_hash_vectors[-1000:],
                                              os.path.join(BASE_DIR, "figures", f"bad_3d_bit_map_step_{step+1}.png"))
                analyze_target_stats()
                threading.Thread(target=save_checkpoint, args=(models, step + 1)).start()
                last_checkpoint_time = current_time

            # Check if all keys recovered
            if len(recovered_keys) == len(targets):
                print("All keys recovered!")
                break

    except KeyboardInterrupt:
        print(f"Interrupted at step {step}. Saving checkpoint...")
        save_checkpoint(models, step)
        logging.info(f"Interrupted at step {step}, checkpoint saved")

    # Finalize
    save_bit_bias_heatmap(is_good=True)
    save_bit_bias_heatmap(is_good=False)
    plot_loss_curve(losses['predictor'], "Hash Predictor")
    plot_loss_curve(losses['entropy'], "Entropy Analysis")
    if len(good_hash_vectors) >= 1000:
        create_bit_correlation_matrix(good_hash_vectors, os.path.join(BASE_DIR, "figures", "final_good_correlation_matrix.png"))
        visualize_hash_clusters(good_hash_vectors, os.path.join(BASE_DIR, "figures", "final_good_hash_clusters.png"))
        create_3d_bit_position_map(good_hash_vectors, os.path.join(BASE_DIR, "figures", "final_good_3d_bit_map.png"))
    if len(bad_hash_vectors) >= 1000:
        create_bit_correlation_matrix(bad_hash_vectors, os.path.join(BASE_DIR, "figures", "final_bad_correlation_matrix.png"))
        visualize_hash_clusters(bad_hash_vectors, os.path.join(BASE_DIR, "figures", "final_bad_hash_clusters.png"))
        create_3d_bit_position_map(bad_hash_vectors, os.path.join(BASE_DIR, "figures", "final_bad_3d_bit_map.png"))
    analyze_target_stats()
    save_checkpoint(models, max_steps)
    with open(os.path.join(REPORT_PATH, "final_report.json"), 'w') as f:
        json.dump({
            "best_hamming": best_hamming,
            "good_samples": len(good_hash_samples),
            "bad_samples": len(bad_hash_samples),
            "anomalies_found": anomalies_found,
            "recovered_keys": recovered_keys
        }, f, indent=2)
    with open(os.path.join(DRIVE_DIR, "reports", "final_report.json"), 'w') as f:
        json.dump({
            "best_hamming": best_hamming,
            "good_samples": len(good_hash_samples),
            "bad_samples": len(bad_hash_samples),
            "anomalies_found": anomalies_found,
            "recovered_keys": recovered_keys
        }, f, indent=2)

    print(f"Recovery complete! Recovered {len(recovered_keys)} keys. Check {RECOVERED_KEYS_PATH} for results.")
# === Block 11: Entry Point ===
if __name__ == "__main__":
    try:
        targets = load_addresses()
        if not targets:
            raise ValueError("No valid addresses loaded")
        run_phantom_t5(targets)
    except Exception as e:
        logging.error(f"Execution failed: {e}")
        print(f"Error: {e}. Check {os.path.join(BASE_DIR, 'logs', 'recovery.log')} for details.")

Using GPU: NVIDIA L4
Fri May  9 03:59:20 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   58C    P0             29W /   72W |     253MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------------

Recovery Progress:   0%|          | 0/250000 [00:00<?, ?it/s]

Step 0: Processed 0 good samples, 0 bad samples, Best Hamming: 160, Recovered: 0, Keys/sec: 0.00, ETA: 69.44 hours


Recovery Progress:   0%|          | 0/250000 [00:00<?, ?it/s]
ERROR:root:Execution failed: invalid literal for int() with base 10: b'\x98#\xac\x02\xc2\x96\xf3\xfc/\xdb\xd6*\xa2&\xd8\xf5\x89\xf4\x89\x10'


Error: invalid literal for int() with base 10: b'\x98#\xac\x02\xc2\x96\xf3\xfc/\xdb\xd6*\xa2&\xd8\xf5\x89\xf4\x89\x10'. Check /content/RIPEMPHANTOM-T5/logs/recovery.log for details.


In [ ]:
import subprocess
import sys
import os
import signal
import time
import random
import json
import hashlib
import base58
import math
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.mplot3d import Axes3D
from ecdsa import SigningKey, SECP256k1
from Crypto.Hash import RIPEMD160
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from statsmodels.stats.proportion import proportion_confint
from sklearn.cluster import DBSCAN
from sklearn.manifold import TSNE
import umap
from numba import jit
import cupy as cp
import logging
import threading
from concurrent.futures import ThreadPoolExecutor
from google.colab import drive, files

# Downgrade NumPy to avoid compatibility issues
subprocess.run(["pip", "install", "-q", "numpy<2"], check=True)

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Prepare Workspace
BASE_DIR = "/content/RIPEMPHANTOM-T5"
DRIVE_DIR = "/content/drive/MyDrive/RIPEMPHANTOM-T5"
for d in [BASE_DIR, DRIVE_DIR]:
    os.makedirs(os.path.join(d, "checkpoints"), exist_ok=True)
    os.makedirs(os.path.join(d, "logs"), exist_ok=True)
    os.makedirs(os.path.join(d, "figures"), exist_ok=True)
    os.makedirs(os.path.join(d, "reports"), exist_ok=True)
    os.makedirs(os.path.join(d, "datasets"), exist_ok=True)

# Setup Logging
logging.basicConfig(
    filename=os.path.join(BASE_DIR, "logs", "recovery.log"),
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# Detect GPU
try:
    if torch.cuda.is_available():
        device = torch.device("cuda:0")
        torch.cuda.set_device(0)
        logging.info(f"GPU: {torch.cuda.get_device_name(0)}, CUDA: {torch.version.cuda}, CuPy devices: {cp.cuda.runtime.getDeviceCount()}")
        print(f"Using GPU: {torch.cuda.get_device_name(0)}")
        subprocess.run(["nvidia-smi"])
    else:
        device = torch.device("cpu")
        logging.info("No GPU, using CPU with numba JIT")
        print("No GPU, using CPU")
except Exception as e:
    device = torch.device("cpu")
    logging.warning(f"GPU detection failed: {e}, using CPU")
    print(f"GPU detection failed: {e}, using CPU")

# Constants
MAX_K = 2**61
ADDR_FILE = os.path.join(BASE_DIR, "addresses.txt")
CSV_FILE = os.path.join(BASE_DIR, "thekeymaster.csv")
CHECKPOINT_PATH = os.path.join(BASE_DIR, "checkpoints")
DRIVE_CHECKPOINT_PATH = os.path.join(DRIVE_DIR, "checkpoints")
LOG_PATH = os.path.join(BASE_DIR, "logs", "telemetry.jsonl")
REPORT_PATH = os.path.join(BASE_DIR, "reports")
RECOVERED_KEYS_PATH = os.path.join(BASE_DIR, "recovered_keys.json")
DRIVE_RECOVERED_KEYS_PATH = os.path.join(DRIVE_DIR, "recovered_keys.json")
EXPECTED_PROB = 0.5
HAMMING_TARGET_RANGE = (41, 49)
BATCH_SIZE = 128 if device.type == "cuda" else 16
PRIORITY_TARGETS = 100
CHECKPOINT_INTERVAL = 300
CHECKPOINT_STEPS = 500
MAX_GOOD_SAMPLES = 50000
MAX_BAD_SAMPLES = 50000

# Configuration
RNG_SEED = int(time.time())
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)
CONFIG = {
    "timestamp": time.strftime("%Y-%m-%d_%H-%M-%S"),
    "rng_seed": RNG_SEED,
    "max_k": MAX_K,
    "version": "RIPEMPHANTOM-T5",
    "hamming_target_range": HAMMING_TARGET_RANGE
}
with open(os.path.join(BASE_DIR, "experiment_config.json"), 'w') as f:
    json.dump(CONFIG, f, indent=2)
with open(os.path.join(DRIVE_DIR, "experiment_config.json"), 'w') as f:
    json.dump(CONFIG, f, indent=2)

# File Upload Prompt
print("Upload addresses.txt (required, 999 P2PKH addresses) and thekeymaster.csv (optional)")
uploaded = files.upload()
if "addresses.txt" not in uploaded:
    logging.error("addresses.txt not uploaded")
    raise FileNotFoundError("addresses.txt is required")
with open(ADDR_FILE, "wb") as f:
    f.write(uploaded["addresses.txt"])
if "thekeymaster.csv" in uploaded:
    with open(CSV_FILE, "wb") as f:
        f.write(uploaded["thekeymaster.csv"])
    logging.info("thekeymaster.csv uploaded")
else:
    logging.info("No thekeymaster.csv uploaded, proceeding without it")

# Block 2: Key-to-Hash160 Pipeline
@jit(nopython=True)
def hamming_distance_and_flips(hash1, hash2):
    distance = 0
    flips = []
    for i in range(20):
        xor = hash1[i] ^ hash2[i]
        for j in range(8):
            if xor & (1 << j):
                bit_index = (i * 8) + (7 - j)
                flips.append(bit_index)
                distance += 1
    return distance, flips

def privkey_to_hash160(k: int, compressed: bool = True) -> bytes:
    if not isinstance(k, int):
        logging.error(f"Invalid private key type: {type(k)}, value: {k}")
        raise ValueError(f"Private key must be an integer, got {type(k)}")
    try:
        if not (1 <= k < SECP256k1.order):
            logging.warning(f"Private key out of range: {k}")
            return None
        sk = SigningKey.from_secret_exponent(k, curve=SECP256k1)
        vk = sk.get_verifying_key()
        if compressed:
            prefix = b'\x02' if vk.to_string()[32] % 2 == 0 else b'\x03'
            pubkey = prefix + vk.to_string()[:32]
        else:
            pubkey = b'\x04' + vk.to_string()
        sha = hashlib.sha256(pubkey).digest()
        ripemd = RIPEMD160.new(); ripemd.update(sha)
        return ripemd.digest()
    except Exception as e:
        logging.error(f"Error in privkey_to_hash160 for key {k}: {e}")
        return None

def key_to_vector(k: int) -> torch.Tensor:
    if not isinstance(k, int):
        logging.error(f"key_to_vector received non-integer: {k}")
        return torch.zeros(256, dtype=torch.float32, device=device)
    return torch.tensor([(k >> i) & 1 for i in range(255, -1, -1)], dtype=torch.float32, device=device)

def hash160_to_vector(h: bytes) -> torch.Tensor:
    if not isinstance(h, bytes) or len(h) != 20:
        logging.error(f"Invalid hash160: {h}")
        return torch.zeros(160, dtype=torch.float32, device=device)
    return torch.tensor([(b >> i) & 1 for b in h for i in range(7, -1, -1)], dtype=torch.float32, device=device)

def vector_to_hash160(v: torch.Tensor) -> bytes:
    hash_bytes = bytearray(20)
    v_cpu = v.cpu().numpy()
    for i in range(160):
        byte_idx = i // 8
        bit_idx = 7 - (i % 8)
        if v_cpu[i] > 0.5:
            hash_bytes[byte_idx] |= (1 << bit_idx)
    return bytes(hash_bytes)

# Block 3: Neural Models
class HashPredictorNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 160),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.model(x)

class EntropyAnalysisNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(160, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2)
        )
        self.decoder = nn.Sequential(
            nn.Linear(64, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 160),
            nn.Sigmoid()
        )
        self.anomaly_detector = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        anomaly_score = self.anomaly_detector(encoded)
        return decoded, anomaly_score, encoded

# Block 4: Adaptive Mutation Engine
bit_bias = defaultdict(float)
bad_bit_bias = defaultdict(float)
xor_chain = []
mutation_log = []
recovered_keys = {}
mutation_bank = []

def update_bit_bias(flips: list, reward: float = 1.0, is_good: bool = True):
    target = bit_bias if is_good else bad_bit_bias
    for bit in flips:
        target[bit] += reward

def add_xor_delta(prev_k: int, new_k: int, reward: float = 1.0):
    if not (isinstance(prev_k, int) and isinstance(new_k, int)):
        logging.error(f"Invalid XOR delta inputs: prev_k={type(prev_k)}, new_k={type(new_k)}")
        return
    delta = prev_k ^ new_k
    xor_chain.append((delta, reward))
    if len(xor_chain) > 100:
        xor_chain.pop(0)

def log_mutation_event(k_old, k_new, mutation_type, flips, delta_hamming):
    entry = {
        "time": time.time(),
        "type": mutation_type,
        "delta_hamming": delta_hamming,
        "flips": flips,
        "delta_k": k_old ^ k_new if isinstance(k_old, int) and isinstance(k_new, int) else None
    }
    mutation_log.append(entry)
    if len(mutation_log) > 5000:
        mutation_log.pop(0)

def record_mutation_result(mutation_vector: int, delta_hamming: int):
    if not isinstance(mutation_vector, int):
        logging.error(f"Invalid mutation vector type: {type(mutation_vector)}, value: {mutation_vector}")
        return
    found = False
    for i, (vec, score) in enumerate(mutation_bank):
        if vec == mutation_vector:
            mutation_bank[i] = (vec, score + delta_hamming)
            found = True
            break
    if not found:
        mutation_bank.append((mutation_vector, delta_hamming))
    if len(mutation_bank) > 100:
        mutation_bank.sort(key=lambda x: -x[1])
        mutation_bank[:] = mutation_bank[:100]
    logging.debug(f"mutation_bank size: {len(mutation_bank)}, sample: {mutation_bank[:5]}")

def get_top_bit_bias(n=10, is_good: bool = True):
    target = bit_bias if is_good else bad_bit_bias
    return sorted(target.items(), key=lambda x: -x[1])[:n]

def get_top_xor_deltas(n=10):
    return sorted(xor_chain, key=lambda x: -x[1])[:n]

def get_best_mutations(n=10):
    valid_mutations = [(vec, score) for vec, score in mutation_bank if isinstance(vec, int)]
    logging.debug(f"Valid mutations: {len(valid_mutations)}, sample: {valid_mutations[:5]}")
    return sorted(valid_mutations, key=lambda x: -x[1])[:n]

def mutate_key(k: int) -> int:
    if not isinstance(k, int):
        logging.error(f"mutate_key received non-integer: {k}")
        return random.randint(1, MAX_K - 1)
    new_k = k
    if random.random() < 0.5:
        new_k += random.randint(-1000000, 1000000)
    for vec, _ in get_best_mutations(n=5):
        if random.random() < 0.3 and isinstance(vec, int):
            new_k ^= vec
        else:
            logging.debug(f"Skipped non-integer or low-probability mutation vector: {vec}")
    for delta, _ in get_top_xor_deltas(n=5):
        if random.random() < 0.3 and isinstance(delta, int):
            new_k ^= delta
    for bit, _ in get_top_bit_bias(n=5, is_good=True):
        if random.random() < 0.3 and isinstance(bit, int):
            new_k ^= (1 << bit)
    for bit, _ in get_top_bit_bias(n=5, is_good=False):
        if random.random() < 0.1 and isinstance(bit, int):
            new_k ^= (1 << bit)
    if random.random() < 0.2:
        new_k ^= random.getrandbits(8)
    if not isinstance(new_k, int):
        logging.error(f"mutate_key produced non-integer: {new_k}")
        return random.randint(1, MAX_K - 1)
    return new_k if 1 <= new_k < MAX_K else random.randint(1, MAX_K - 1)

def generate_key(seed_k: int, transformer_model=None) -> int:
    if not isinstance(seed_k, int):
        logging.error(f"generate_key received non-integer: {seed_k}")
        return random.randint(1, MAX_K - 1)
    k = mutate_key(seed_k)
    if transformer_model:
        try:
            with torch.no_grad():
                k_tensor = key_to_vector(k).unsqueeze(0)
                prediction = transformer_model(k_tensor).squeeze(0)
                for i in range(160):
                    if prediction[i].item() > 0.95 and random.random() < 0.5:
                        k ^= (1 << (i % 160))
        except Exception as e:
            logging.warning(f"Transformer model error: {e}")
    if not isinstance(k, int):
        logging.error(f"generate_key produced non-integer: {k}")
        return random.randint(1, MAX_K - 1)
    logging.debug(f"Generated key: {k}, type: {type(k)}")
    return k if 1 <= k < MAX_K else random.randint(1, MAX_K - 1)

# Block 5: Address and Target Management
def load_addresses() -> list:
    targets = []
    with open(ADDR_FILE, 'r') as f:
        for line in f:
            addr = line.strip()
            if not addr:
                continue
            try:
                decoded = base58.b58decode(addr)
                if len(decoded) < 25 or decoded[0] != 0x00 or len(decoded[1:-4]) != 20:
                    logging.warning(f"Invalid address: {addr}")
                    continue
                h160 = decoded[1:-4]
                targets.append((addr, h160))
            except Exception as e:
                logging.warning(f"Error decoding {addr}: {e}")
                continue
    if not targets:
        logging.error("No valid P2PKH addresses found")
        raise ValueError("No valid addresses loaded")
    logging.info(f"Loaded {len(targets)} valid P2PKH addresses")
    return targets

def load_keymaster_csv():
    if os.path.exists(CSV_FILE):
        try:
            df = pd.read_csv(CSV_FILE)
            logging.info(f"Loaded thekeymaster.csv with {len(df)} entries")
            return df
        except Exception as e:
            logging.warning(f"Error loading thekeymaster.csv: {e}")
    return None

target_stats = {}
recovered_keys = {}
mutation_bank = []

def init_target_stats(targets):
    for addr, _ in targets:
        target_stats[addr] = {
            "best": 160,
            "best_k": -1,
            "scans": 0,
            "hamming_history": [],
            "last_improvement": time.time()
        }

def update_target_stat(addr: str, k: int, ham: int, compressed: bool):
    if not isinstance(k, int):
        logging.error(f"update_target_stat received non-integer key: {k}")
        return
    if len(target_stats[addr]["hamming_history"]) >= 1000:
        target_stats[addr]["hamming_history"].pop(0)
    target_stats[addr]["hamming_history"].append(ham)
    if ham < target_stats[addr]["best"]:
        target_stats[addr]["best"] = ham
        target_stats[addr]["best_k"] = k
        target_stats[addr]["last_improvement"] = time.time()
        logging.info(f"New best for {addr}: Hamming {ham}, Key {hex(k)}, Compressed: {compressed}")
        if ham == 0:
            recovered_keys[addr] = {"key": hex(k), "compressed": compressed}
            with open(RECOVERED_KEYS_PATH, 'w') as f:
                json.dump(recovered_keys, f, indent=2)
            with open(DRIVE_RECOVERED_KEYS_PATH, 'w') as f:
                json.dump(recovered_keys, f, indent=2)
            logging.info(f"Recovered key for {addr}: {hex(k)}, Compressed: {compressed}")
    target_stats[addr]["scans"] += 1

def get_priority_targets(targets, top_n=PRIORITY_TARGETS):
    if not target_stats:
        return random.sample(targets, min(top_n, len(targets)))
    sorted_targets = sorted(
        target_stats.items(),
        key=lambda x: (x[1]["best"], -x[1]["last_improvement"])
    )
    top_addrs = set(addr for addr, _ in sorted_targets[:top_n] if target_stats[addr]["best"] <= 50)
    return [(addr, h) for addr, h in targets if addr in top_addrs]

# Block 6: Statistical Analysis
def compute_bit_bias_confidence_intervals(is_good: bool = True):
    target = bit_bias if is_good else bad_bit_bias
    result = defaultdict(lambda: {"bias": 0, "proportion": 0, "ci_lower": 0, "ci_upper": 0, "significant": False})
    total_observations = sum(target.values()) or 1
    for bit, value in target.items():
        proportion = value / total_observations
        lower, upper = proportion_confint(value, total_observations, alpha=0.05, method='wilson')
        result[bit] = {
            "bias": value,
            "proportion": proportion,
            "ci_lower": lower,
            "ci_upper": upper,
            "significant": lower > EXPECTED_PROB or upper < EXPECTED_PROB
        }
    return result

def save_bit_bias_heatmap(is_good: bool = True):
    target = bit_bias if is_good else bad_bit_bias
    if not target:
        return
    bias_data = compute_bit_bias_confidence_intervals(is_good)
    bits = sorted(bias_data.keys())
    values = [bias_data[bit]["bias"] for bit in bits]
    lower_ci = [bias_data[bit]["ci_lower"] for bit in bits]
    upper_ci = [bias_data[bit]["ci_upper"] for bit in bits]
    plt.figure(figsize=(16, 6))
    plt.bar(bits, values, alpha=0.7)
    plt.errorbar(bits, values, yerr=[(v-l) for v, l in zip(values, lower_ci)],
                 fmt='none', ecolor='black', capsize=3)
    for bit in bits:
        if bias_data[bit]["significant"]:
            plt.plot(bit, bias_data[bit]["bias"], 'r*', markersize=10)
    plt.title(f"{'Good' if is_good else 'Bad'} Key Bit Bias Heatmap with 95% Confidence Intervals")
    plt.xlabel("Bit Position (0–159)")
    plt.ylabel("Reinforcement Score")
    plt.grid(True, alpha=0.3)
    plt.axhline(y=sum(values)/len(values) if values else 0, color='r', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, "figures", f"{'good' if is_good else 'bad'}_bit_bias_heatmap.png"), dpi=300)
    plt.savefig(os.path.join(DRIVE_DIR, "figures", f"{'good' if is_good else 'bad'}_bit_bias_heatmap.png"), dpi=300)
    plt.close()

def plot_loss_curve(losses, label: str):
    if not losses:
        return
    plt.figure(figsize=(10, 6))
    plt.plot(losses)
    if len(losses) > 10:
        window_size = min(len(losses) // 10, 20)
        smoothed = np.convolve(losses, np.ones(window_size)/window_size, mode='valid')
        plt.plot(range(window_size-1, len(losses)), smoothed, 'r-', linewidth=2)
    plt.title(f"{label} Loss Curve")
    plt.xlabel("Batch")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, "figures", f"{label.lower().replace(' ', '_')}_loss.png"), dpi=300)
    plt.savefig(os.path.join(DRIVE_DIR, "figures", f"{label.lower().replace(' ', '_')}_loss.png"), dpi=300)
    plt.close()

def create_bit_correlation_matrix(hash160_vectors, save_path=None):
    if not hash160_vectors or len(hash160_vectors) < 10:
        return None
    matrix = torch.stack(hash160_vectors).cpu().numpy()
    corr_matrix = np.corrcoef(matrix.T)
    plt.figure(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    cmap = sns.diverging_palette(230, 20, as_cmap=True)
    sns.heatmap(corr_matrix, mask=mask, cmap=cmap, vmax=0.3, vmin=-0.3,
                center=0, square=True, linewidths=.5)
    plt.title('Bit Position Correlation Matrix')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.savefig(save_path.replace(BASE_DIR, DRIVE_DIR), dpi=300)
        plt.close()
    return corr_matrix

def visualize_hash_clusters(hash_vectors, save_path=None, method='tsne'):
    if not hash_vectors or len(hash_vectors) < 10:
        return
    data = torch.stack(hash_vectors).cpu().numpy()
    if method.lower() == 'tsne':
        reducer = TSNE(n_components=2, random_state=RNG_SEED)
    elif method.lower() == 'umap':
        reducer = umap.UMAP(random_state=RNG_SEED)
    else:
        reducer = PCA(n_components=2, random_state=RNG_SEED)
    reduced_data = reducer.fit_transform(data)
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    clusters = dbscan.fit_predict(reduced_data)
    plt.figure(figsize=(12, 10))
    scatter = plt.scatter(reduced_data[:, 0], reduced_data[:, 1], c=clusters,
                         cmap='viridis', s=50, alpha=0.7)
    plt.legend(*scatter.legend_elements(), title="Clusters")
    plt.title(f'Hash160 Clusters ({method.upper()})')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.savefig(save_path.replace(BASE_DIR, DRIVE_DIR), dpi=300)
        plt.close()
    return {'reduced_data': reduced_data, 'clusters': clusters}

def create_3d_bit_position_map(hash160_samples, save_path=None):
    if not hash160_samples or len(hash160_samples) < 100:
        return
    data = torch.stack(hash160_samples).cpu().numpy()
    bit_variance = np.var(data, axis=0)
    top_bits = np.argsort(bit_variance)[-3:]
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')
    x = data[:, top_bits[0]]
    y = data[:, top_bits[1]]
    z = data[:, top_bits[2]]
    colors = np.sum(data, axis=1)
    scatter = ax.scatter(x, y, z, c=colors, cmap='viridis', s=30, alpha=0.7)
    cbar = plt.colorbar(scatter)
    cbar.set_label('Bit Sum (Entropy Proxy)')
    ax.set_xlabel(f'Bit {top_bits[0]}')
    ax.set_ylabel(f'Bit {top_bits[1]}')
    ax.set_zlabel(f'Bit {top_bits[2]}')
    plt.title('3D Relationship Between High-Variance Bit Positions')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.savefig(save_path.replace(BASE_DIR, DRIVE_DIR), dpi=300)
        plt.close()

def analyze_target_stats():
    from scipy import stats
    if not target_stats:
        return {}
    analysis = {
        "global": {
            "best_hamming": min([s["best"] for s in target_stats.values()]),
            "avg_hamming": sum([s["best"] for s in target_stats.values()]) / len(target_stats),
            "total_scans": sum([s["scans"] for s in target_stats.values()]),
            "timestamp": time.time()
        },
        "targets": {}
    }
    for addr, stats in target_stats.items():
        if not stats["hamming_history"]:
            continue
        history = stats["hamming_history"]
        analysis["targets"][addr] = {
            "best": stats["best"],
            "mean": np.mean(history),
            "median": np.median(history),
            "std_dev": np.std(history),
            "scans": stats["scans"]
        }
        if len(history) > 10:
            x = np.arange(len(history))
            y = np.array(history)
            try:
                slope, _, r_value, p_value, _ = stats.linregress(x, y)
                analysis["targets"][addr].update({
                    "trend_slope": slope,
                    "trend_r_squared": r_value**2,
                    "trend_p_value": p_value
                })
            except Exception as e:
                analysis["targets"][addr].update({
                    "trend_slope": 0.0,
                    "trend_r_squared": 0.0,
                    "trend_p_value": 1.0
                })
    with open(os.path.join(REPORT_PATH, "target_analysis.json"), 'w') as f:
        json.dump(analysis, f, indent=2)
    with open(os.path.join(DRIVE_DIR, "reports", "target_analysis.json"), 'w') as f:
        json.dump(analysis, f, indent=2)
    return analysis

class NBISTTestSuite:
    def __init__(self):
        self.tests = {
            "frequency": self.frequency_test,
            "frequency_block": self.block_frequency_test,
            "runs": self.runs_test,
            "longest_run": self.longest_run_test,
            "serial": self.serial_test,
            "entropy": self.approximate_entropy_test
        }
    def frequency_test(self, bits):
        n = len(bits)
        s = sum([(2*bit-1) for bit in bits])
        s_obs = abs(s) / np.sqrt(n)
        p_value = math.erfc(s_obs / np.sqrt(2))
        return p_value
    def block_frequency_test(self, bits, block_size=10):
        n = len(bits)
        n_blocks = n // block_size
        if n_blocks == 0:
            return 1.0
        blocks = [bits[i:i+block_size] for i in range(0, n_blocks*block_size, block_size)]
        proportions = [sum(block)/block_size for block in blocks]
        chi_squared = 4.0 * block_size * sum([(prop-0.5)**2 for prop in proportions])
        p_value = stats.chi2.sf(chi_squared, n_blocks)
        return p_value
    def runs_test(self, bits):
        n = len(bits)
        if n < 100:
            return 1.0
        prop = sum(bits) / n
        tau = 2.0 / np.sqrt(n)
        if abs(prop - 0.5) >= tau:
            return 0.0
        runs = 1
        for i in range(1, n):
            if bits[i] != bits[i-1]:
                runs += 1
        p_value = math.erfc(abs(runs - 2*n*prop*(1-prop)) /
                          (2*np.sqrt(2*n)*prop*(1-prop)))
        return p_value
    def longest_run_test(self, bits):
        n = len(bits)
        if n < 128:
            return 1.0
        if n < 6272:
            k, m = 3, 8
            v = [1, 2, 3, 4]
            pi = [0.2148, 0.3672, 0.2305, 0.1875]
        else:
            k, m = 5, 128
            v = [4, 5, 6, 7, 8, 9]
            pi = [0.1174, 0.2430, 0.2493, 0.1752, 0.1027, 0.1124]
        num_blocks = n // m
        longest_runs = []
        for i in range(num_blocks):
            block = bits[i*m:(i+1)*m]
            max_run = 0
            current_run = 0
            for bit in block:
                if bit == 1:
                    current_run += 1
                    max_run = max(max_run, current_run)
                else:
                    current_run = 0
            longest_runs.append(max_run)
        frequencies = [0] * (k+1)
        for run in longest_runs:
            if run <= v[0]:
                frequencies[0] += 1
            elif run >= v[-1]:
                frequencies[-1] += 1
            else:
                for j in range(1, len(v)):
                    if v[j-1] < run <= v[j]:
                        frequencies[j] += 1
                        break
        chi_squared = sum([(frequencies[i] - num_blocks*pi[i])**2 / (num_blocks*pi[i])
                          for i in range(len(frequencies))])
        p_value = stats.chi2.sf(chi_squared, k)
        return p_value
    def serial_test(self, bits, m=3):
        n = len(bits)
        if n < 2**m:
            return 1.0
        extended_bits = bits + bits[:m-1]
        pattern_counts1 = defaultdict(int)
        for i in range(n):
            pattern = tuple(extended_bits[i:i+m])
            pattern_counts1[pattern] += 1
        pattern_counts2 = defaultdict(int)
        for i in range(n):
            pattern = tuple(extended_bits[i:i+m-1])
            pattern_counts2[pattern] += 1
        pattern_counts3 = defaultdict(int)
        for i in range(n):
            pattern = tuple(extended_bits[i:i+m-2])
            pattern_counts3[pattern] += 1
        psi_sq_m = sum([count**2 for count in pattern_counts1.values()]) * 2**m / n - n
        psi_sq_m1 = sum([count**2 for count in pattern_counts2.values()]) * 2**(m-1) / n - n
        psi_sq_m2 = sum([count**2 for count in pattern_counts3.values()]) * 2**(m-2) / n - n
        delta1 = psi_sq_m - psi_sq_m1
        delta2 = psi_sq_m - 2*psi_sq_m1 + psi_sq_m2
        p_value1 = stats.chi2.sf(delta1, 2**(m-1))
        p_value2 = stats.chi2.sf(delta2, 2**(m-2))
        return min(p_value1, p_value2)
    def approximate_entropy_test(self, bits, m=5):
        n = len(bits)
        if n < 100:
            return 1.0
        def phi(m_val):
            counts = defaultdict(int)
            for i in range(n):
                pattern = tuple(bits[(i+j) % n] for j in range(m_val))
                counts[pattern] += 1
            c_m = [count/n for count in counts.values()]
            return sum([p * np.log(p) for p in c_m if p > 0])
        apen = phi(m) - phi(m+1)
        chi_squared = 2 * n * (np.log(2) - apen)
        p_value = stats.chi2.sf(chi_squared, 2**m)
        return p_value
    def run_all_tests(self, bits):
        results = {}
        for test_name, test_func in self.tests.items():
            p_value = test_func(bits)
            passed = p_value >= 0.01
            conf_level = 0.95
            z = stats.norm.ppf(1 - (1 - conf_level) / 2)
            if passed:
                lower_ci = (1 + 1/(4*1) - z/(2*np.sqrt(1)))
                upper_ci = 1.0
            else:
                lower_ci = 0.0
                upper_ci = z/(2*np.sqrt(1))
            results[test_name] = {
                "p_value": p_value,
                "passed": passed,
                "confidence_lower": lower_ci,
                "confidence_upper": upper_ci
            }
        return results

def run_entropy_scan(sample_size: int = 100_000, reference_hashes: list = [],
                    statistical_rigor: bool = True):
    print(f"Running anomaly detection scan with {sample_size:,} keys...")
    entropy_counter = np.zeros(160, dtype=int)
    entropy_total = 0
    entropy_samples = 0
    hamming_distribution = []
    hash_samples = []
    bit_vectors = []
    for _ in tqdm(range(sample_size), desc="Entropy Scan"):
        k = random.randint(1, MAX_K)
        for compressed in [True, False]:
            h160 = privkey_to_hash160(k, compressed)
            if not h160:
                continue
            entropy_samples += 1
            for i in range(20):
                byte = h160[i]
                for j in range(8):
                    bit_index = (i * 8) + (7 - j)
                    if byte & (1 << j):
                        entropy_counter[bit_index] += 1
            entropy_total += 1
            hash_samples.append(h160)
            bit_vectors.append(hash160_to_vector(h160))
            for _, ref in reference_hashes:
                dist, _ = hamming_distance_and_flips(h160, ref)
                hamming_distribution.append(dist)
    plt.figure(figsize=(16, 6))
    proportions = entropy_counter / max(1, entropy_samples)
    confidence = 0.95
    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    lower_ci = []
    upper_ci = []
    for i in range(160):
        p = proportions[i]
        n = entropy_samples
        denominator = 1 + z**2/n
        centre_adjusted_prob = (p + z**2/(2*n))/denominator
        adjusted_standard_dev = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2))/denominator
        lower = max(0, centre_adjusted_prob - adjusted_standard_dev)
        upper = min(1, centre_adjusted_prob + adjusted_standard_dev)
        lower_ci.append(lower)
        upper_ci.append(upper)
    plt.bar(range(160), proportions, alpha=0.7)
    plt.errorbar(range(160), proportions,
                yerr=[proportions-lower_ci, upper_ci-proportions],
                fmt='none', ecolor='black', capsize=2, alpha=0.3)
    plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Expected (0.5)')
    for i in range(160):
        if lower_ci[i] > 0.5 or upper_ci[i] < 0.5:
            plt.plot(i, proportions[i], 'r*', markersize=8)
    plt.title(f"RIPEMD-160 Bitwise Distribution with {confidence*100:.0f}% Confidence Intervals")
    plt.xlabel("Bit Position (0–159)")
    plt.ylabel("Frequency of 1s")
    plt.ylim(0.45, 0.55)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, "figures", "hash160_entropy_profile.png"), dpi=300)
    plt.savefig(os.path.join(DRIVE_DIR, "figures", "hash160_entropy_profile.png"), dpi=300)
    plt.close()
    if hamming_distribution:
        plt.figure(figsize=(12, 8))
        hist, bin_edges = np.histogram(hamming_distribution, bins=40)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        plt.bar(bin_centers, hist, width=(bin_edges[1]-bin_edges[0])*0.8,
               color='steelblue', edgecolor='black', alpha=0.7)
        x = np.arange(40, 120)
        n = 160
        p = 0.5
        expected = stats.binom.pmf(x, n, p) * sum(hist) * (bin_edges[1]-bin_edges[0])
        plt.plot(x, expected, 'r-', linewidth=2,
                label='Expected (Binomial Distribution)')
        mean_ham = np.mean(hamming_distribution)
        std_ham = np.std(hamming_distribution)
        expected_mean = n * p
        expected_std = np.sqrt(n * p * (1 - p))
        plt.axvline(x=mean_ham, color='blue', linestyle='--',
                   label=f'Observed Mean: {mean_ham:.2f}')
        plt.axvline(x=expected_mean, color='red', linestyle='--',
                   label=f'Expected Mean: {expected_mean:.2f}')
        ks_stat, p_value = stats.kstest(hamming_distribution, stats.binom.cdf, args=(n, p))
        plt.title(f'Hamming Distances to Reference Hash160s\nKS-test: D={ks_stat:.4f}, p={p_value:.6f}')
        plt.xlabel('Hamming Distance (bits)')
        plt.ylabel('Frequency')
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(BASE_DIR, "figures", "hamming_histogram.png"), dpi=300)
        plt.savefig(os.path.join(DRIVE_DIR, "figures", "hamming_histogram.png"), dpi=300)
        plt.close()
    if statistical_rigor and len(bit_vectors) > 1000:
        print("Performing advanced statistical analysis...")
        bit_array = np.stack([v.cpu().numpy() for v in bit_vectors])
        test_results = {}
        nist_suite = NBISTTestSuite()
        for bit_pos in tqdm(range(160), desc="Testing bit positions"):
            bits = bit_array[:, bit_pos].astype(int).tolist()
            test_results[bit_pos] = nist_suite.run_all_tests(bits)
        with open(os.path.join(BASE_DIR, "reports", "nist_test_results.json"), 'w') as f:
            json.dump(test_results, f, indent=2)
        with open(os.path.join(DRIVE_DIR, "reports", "nist_test_results.json"), 'w') as f:
            json.dump(test_results, f, indent=2)
        summary = {
            "failed_tests": 0,
            "total_tests": 0,
            "significant_bits": []
        }
        for bit_pos, results in test_results.items():
            for test_name, result in results.items():
                summary["total_tests"] += 1
                if not result["passed"]:
                    summary["failed_tests"] += 1
                    summary["significant_bits"].append({
                        "bit": bit_pos,
                        "test": test_name,
                        "p_value": result["p_value"]
                    })
        with open(os.path.join(BASE_DIR, "reports", "nist_summary.json"), 'w') as f:
            json.dump(summary, f, indent=2)
        with open(os.path.join(DRIVE_DIR, "reports", "nist_summary.json"), 'w') as f:
            json.dump(summary, f, indent=2)
    return {
        'hash_samples': hash_samples,
        'bit_vectors': bit_vectors,
        'hamming_distribution': hamming_distribution
    }

# Block 7: GPU-Accelerated Key Processing
def process_key_batch(keys: list, targets: list) -> list:
    results = []
    target_h160s = [h for _, h in targets]
    def process_key(k):
        if not isinstance(k, int):
            logging.error(f"process_key received non-integer: {k}")
            return []
        batch_results = []
        for compressed in [True, False]:
            h160 = privkey_to_hash160(k, compressed)
            if h160:
                for addr, target_h160 in targets:
                    ham, flips = hamming_distance_and_flips(h160, target_h160)
                    batch_results.append((addr, k, h160, ham, flips, compressed))
        return batch_results
    valid_keys = [k for k in keys if isinstance(k, int)]
    if len(valid_keys) < len(keys):
        logging.warning(f"Filtered {len(keys) - len(valid_keys)} non-integer keys")
    if device.type == "cuda":
        try:
            keys_array = cp.array(valid_keys, dtype=cp.uint64)
            target_h160s_array = cp.array(target_h160s, dtype=cp.uint8)
            for k in valid_keys:
                for compressed in [True, False]:
                    h160 = privkey_to_hash160(k, compressed)
                    if h160:
                        h160_array = cp.array(list(h160), dtype=cp.uint8)
                        for addr, target_h160 in targets:
                            target_array = cp.array(list(target_h160), dtype=cp.uint8)
                            ham = cp.sum(h160_array != target_array).get()
                            _, flips = hamming_distance_and_flips(h160, target_h160)
                            results.append((addr, k, h160, ham, flips, compressed))
        except cp.cuda.memory.OutOfMemoryError:
            logging.warning("GPU memory error, falling back to CPU")
            with ThreadPoolExecutor() as executor:
                batch_results = executor.map(process_key, valid_keys)
                for batch in batch_results:
                    results.extend(batch)
    else:
        with ThreadPoolExecutor() as executor:
            batch_results = executor.map(process_key, valid_keys)
            for batch in batch_results:
                results.extend(batch)
    logging.debug(f"Batch processed, results size: {len(results)}")
    return results

# Block 8: Checkpointing and Signal Handling
def save_checkpoint(models: dict, step_count: int):
    try:
        for model_name, model in models.items():
            torch.save(model.state_dict(), os.path.join(CHECKPOINT_PATH, f"{model_name}.pt"))
            torch.save(model.state_dict(), os.path.join(DRIVE_CHECKPOINT_PATH, f"{model_name}.pt"))
        state = {
            "bit_bias": dict(bit_bias),
            "bad_bit_bias": dict(bad_bit_bias),
            "xor_chain": xor_chain,
            "mutation_bank": [(hex(vec), score) for vec, score in mutation_bank if isinstance(vec, int)],
            "step": step_count,
            "timestamp": time.time(),
            "config": CONFIG
        }
        torch.save(state, os.path.join(CHECKPOINT_PATH, "research_state.pt"))
        torch.save(state, os.path.join(DRIVE_CHECKPOINT_PATH, "research_state.pt"))
        with open(os.path.join(CHECKPOINT_PATH, "bit_bias.json"), 'w') as f:
            json.dump(dict(bit_bias), f)
        with open(os.path.join(DRIVE_CHECKPOINT_PATH, "bit_bias.json"), 'w') as f:
            json.dump(dict(bit_bias), f)
        with open(os.path.join(CHECKPOINT_PATH, "bad_bit_bias.json"), 'w') as f:
            json.dump(dict(bad_bit_bias), f)
        with open(os.path.join(DRIVE_CHECKPOINT_PATH, "bad_bit_bias.json"), 'w') as f:
            json.dump(dict(bad_bit_bias), f)
        with open(os.path.join(CHECKPOINT_PATH, "xor_chain.json"), 'w') as f:
            json.dump(xor_chain, f)
        with open(os.path.join(DRIVE_CHECKPOINT_PATH, "xor_chain.json"), 'w') as f:
            json.dump(xor_chain, f)
        with open(os.path.join(CHECKPOINT_PATH, "mutation_bank.json"), 'w') as f:
            json.dump([(hex(vec), score) for vec, score in mutation_bank if isinstance(vec, int)], f)
        with open(os.path.join(DRIVE_CHECKPOINT_PATH, "mutation_bank.json"), 'w') as f:
            json.dump([(hex(vec), score) for vec, score in mutation_bank if isinstance(vec, int)], f)
        with open(os.path.join(CHECKPOINT_PATH, "step.txt"), 'w') as f:
            f.write(str(step_count))
        with open(os.path.join(DRIVE_CHECKPOINT_PATH, "step.txt"), 'w') as f:
            f.write(str(step_count))
        log_event("checkpoint", {"step": step_count, "recovered_keys": len(recovered_keys)})
    except Exception as e:
        logging.error(f"Checkpoint save failed: {e}")

def load_checkpoint(models: dict) -> int:
    try:
        state_path = os.path.join(CHECKPOINT_PATH, "research_state.pt")
        step = 0
        if os.path.exists(state_path):
            state = torch.load(state_path, map_location=device)
            bit_bias.update(state.get("bit_bias", {}))
            bad_bit_bias.update(state.get("bad_bit_bias", {}))
            xor_chain.extend(state.get("xor_chain", []))
            # Only load integer mutation vectors
            mutation_bank.extend([(int(vec, 16), score) for vec, score in state.get("mutation_bank", []) if isinstance(vec, str)])
            step = state.get("step", 0)
            for model_name, model in models.items():
                model_path = os.path.join(CHECKPOINT_PATH, f"{model_name}.pt")
                if os.path.exists(model_path):
                    model.load_state_dict(torch.load(model_path, map_location=device))
            logging.info(f"Loaded state from step {step}")
        return step
    except Exception as e:
        logging.warning(f"No checkpoint loaded: {e}")
        return 0

def log_event(event_type: str, payload: dict):
    try:
        entry = {
            "time": time.time(),
            "event": event_type,
            "data": payload
        }
        with open(LOG_PATH, 'a') as f:
            f.write(json.dumps(entry) + '\n')
        with open(os.path.join(DRIVE_DIR, "logs", "telemetry.jsonl"), 'a') as f:
            f.write(json.dumps(entry) + '\n')
    except Exception as e:
        print(f"Failed to log event: {e}")

def signal_handler(sig, frame, models, step):
    print(f"Received interrupt signal {sig}, saving checkpoint at step {step}...")
    save_checkpoint(models, step)
    logging.info(f"Interrupted at step {step}, checkpoint saved")
    sys.exit(0)

# Block 10: Main Recovery Loop
def run_phantom_t5(targets, max_steps=250_000):
    global bit_bias, bad_bit_bias, xor_chain, mutation_log, recovered_keys, mutation_bank
    models = {
        'predictor_net': HashPredictorNet().to(device),
        'entropy_net': EntropyAnalysisNet().to(device)
    }
    optimizers = {
        'predictor_net': torch.optim.Adam(models['predictor_net'].parameters(), lr=0.001),
        'entropy_net': torch.optim.Adam(models['entropy_net'].parameters(), lr=0.001)
    }
    losses = {'predictor': [], 'entropy': []}
    # Start fresh to avoid corrupted state
    start_step = 0
    init_target_stats(targets)
    best_hamming = 160
    good_hash_samples = []
    bad_hash_samples = []
    good_hash_vectors = []
    bad_hash_vectors = []
    anomalies_found = 0
    last_checkpoint_time = time.time()
    start_time = time.time()
    keys_processed = 0
    signal.signal(signal.SIGINT, lambda sig, frame: signal_handler(sig, frame, models, start_step))
    signal.signal(signal.SIGTERM, lambda sig, frame: signal_handler(sig, frame, models, start_step))
    logging.info(f"Starting recovery with {len(targets)} targets, {max_steps} steps, device: {device}")
    print(f"Starting RIPEMPHANTOM-T5 with {len(targets)} targets on {device}...")
    try:
        for step in tqdm(range(start_step, max_steps), desc="Recovery Progress"):
            if step % 100 == 0:
                elapsed = time.time() - start_time
                steps_per_sec = (step - start_step) / max(1, elapsed)
                keys_per_sec = keys_processed / max(1, elapsed)
                eta = (max_steps - step) / max(1, steps_per_sec) / 3600
                print(f"Step {step}: Processed {len(good_hash_samples)} good samples, "
                      f"{len(bad_hash_samples)} bad samples, Best Hamming: {best_hamming}, "
                      f"Recovered: {len(recovered_keys)}, Keys/sec: {keys_per_sec:.2f}, ETA: {eta:.2f} hours")
            # Reset mutation_bank before each batch
            mutation_bank[:] = [(vec, score) for vec, score in mutation_bank if isinstance(vec, int)]
            logging.debug(f"Cleared mutation_bank at step {step}, size: {len(mutation_bank)}")
            current_targets = get_priority_targets(targets)
            if not current_targets:
                current_targets = random.sample(targets, min(5, len(targets)))
            seed_k = random.randint(1, MAX_K)
            keys = [generate_key(seed_k, models['predictor_net']) for _ in range(BATCH_SIZE)]
            logging.debug(f"Step {step}: Generated keys types: {[type(k) for k in keys]}")
            batch_results = process_key_batch(keys, current_targets)
            keys_processed += len(keys) * 2
            if not batch_results:
                continue
            good_key_vectors = []
            good_hash_vectors_batch = []
            bad_key_vectors = []
            bad_hash_vectors_batch = []
            for addr, k, h160, ham, flips, compressed in batch_results:
                update_target_stat(addr, k, ham, compressed)
                best_hamming = min(best_hamming, ham)
                if ham <= 50 or ham == 0:
                    anomalies_found += 1
                    reward = 1.0 / max(1, ham)
                    update_bit_bias(flips, reward, is_good=True)
                    add_xor_delta(seed_k, k, reward)
                    mutation_vector = k ^ seed_k
                    if isinstance(mutation_vector, int):
                        record_mutation_result(mutation_vector, 160 - ham)
                    else:
                        logging.error(f"Invalid mutation vector: {mutation_vector}")
                    log_mutation_event(seed_k, k, "targeted", flips, ham - target_stats[addr]["best"])
                    if len(good_hash_samples) < MAX_GOOD_SAMPLES:
                        good_hash_samples.append(h160)
                        good_hash_vectors.append(hash160_to_vector(h160))
                        good_key_vectors.append(key_to_vector(k))
                        good_hash_vectors_batch.append(hash160_to_vector(h160))
                elif ham > 80:
                    reward = 1.0 / max(1, 160 - ham)
                    update_bit_bias(flips, reward, is_good=False)
                    if len(bad_hash_samples) < MAX_BAD_SAMPLES:
                        bad_hash_samples.append(h160)
                        bad_hash_vectors.append(hash160_to_vector(h160))
                        bad_key_vectors.append(key_to_vector(k))
                        bad_hash_vectors_batch.append(hash160_to_vector(h160))
            if good_hash_vectors_batch or bad_hash_vectors_batch:
                all_key_vectors = good_key_vectors + bad_key_vectors[:len(good_key_vectors)]
                all_hash_vectors = good_hash_vectors_batch + bad_hash_vectors_batch[:len(good_hash_vectors_batch)]
                if all_key_vectors:
                    key_vectors = torch.stack(all_key_vectors)
                    true_hash = torch.stack(all_hash_vectors)
                    optimizers['predictor_net'].zero_grad()
                    pred_hash = models['predictor_net'](key_vectors)
                    loss_pred = F.binary_cross_entropy(pred_hash, true_hash)
                    loss_pred.backward()
                    optimizers['predictor_net'].step()
                    losses['predictor'].append(loss_pred.item())
                    optimizers['entropy_net'].zero_grad()
                    decoded, anomaly_score, _ = models['entropy_net'](true_hash)
                    recon_loss = F.binary_cross_entropy(decoded, true_hash)
                    anomaly_loss = torch.mean(anomaly_score)
                    loss_entropy = recon_loss + 0.1 * anomaly_loss
                    loss_entropy.backward()
                    optimizers['entropy_net'].step()
                    losses['entropy'].append(loss_entropy.item())
            current_time = time.time()
            if (step + 1) % CHECKPOINT_STEPS == 0 or (current_time - last_checkpoint_time) > CHECKPOINT_INTERVAL:
                save_bit_bias_heatmap(is_good=True)
                save_bit_bias_heatmap(is_good=False)
                plot_loss_curve(losses['predictor'], "Hash Predictor")
                plot_loss_curve(losses['entropy'], "Entropy Analysis")
                if len(good_hash_vectors) >= 1000:
                    create_bit_correlation_matrix(good_hash_vectors[-1000:],
                                                os.path.join(BASE_DIR, "figures", f"good_correlation_matrix_step_{step+1}.png"))
                    visualize_hash_clusters(good_hash_vectors[-1000:],
                                          os.path.join(BASE_DIR, "figures", f"good_hash_clusters_step_{step+1}.png"))
                    create_3d_bit_position_map(good_hash_vectors[-1000:],
                                              os.path.join(BASE_DIR, "figures", f"good_3d_bit_map_step_{step+1}.png"))
                if len(bad_hash_vectors) >= 1000:
                    create_bit_correlation_matrix(bad_hash_vectors[-1000:],
                                                os.path.join(BASE_DIR, "figures", f"bad_correlation_matrix_step_{step+1}.png"))
                    visualize_hash_clusters(bad_hash_vectors[-1000:],
                                          os.path.join(BASE_DIR, "figures", f"bad_hash_clusters_step_{step+1}.png"))
                    create_3d_bit_position_map(bad_hash_vectors[-1000:],
                                              os.path.join(BASE_DIR, "figures", f"bad_3d_bit_map_step_{step+1}.png"))
                analyze_target_stats()
                threading.Thread(target=save_checkpoint, args=(models, step + 1)).start()
                last_checkpoint_time = current_time
            if len(recovered_keys) == len(targets):
                print("All keys recovered!")
                break
    except KeyboardInterrupt:
        print(f"Interrupted at step {step}. Saving checkpoint...")
        save_checkpoint(models, step)
        logging.info(f"Interrupted at step {step}, checkpoint saved")
    save_bit_bias_heatmap(is_good=True)
    save_bit_bias_heatmap(is_good=False)
    plot_loss_curve(losses['predictor'], "Hash Predictor")
    plot_loss_curve(losses['entropy'], "Entropy Analysis")
    if len(good_hash_vectors) >= 1000:
        create_bit_correlation_matrix(good_hash_vectors, os.path.join(BASE_DIR, "figures", "final_good_correlation_matrix.png"))
        visualize_hash_clusters(good_hash_vectors, os.path.join(BASE_DIR, "figures", "final_good_hash_clusters.png"))
        create_3d_bit_position_map(good_hash_vectors, os.path.join(BASE_DIR, "figures", "final_good_3d_bit_map.png"))
    if len(bad_hash_vectors) >= 1000:
        create_bit_correlation_matrix(bad_hash_vectors, os.path.join(BASE_DIR, "figures", "final_bad_correlation_matrix.png"))
        visualize_hash_clusters(bad_hash_vectors, os.path.join(BASE_DIR, "figures", "final_bad_hash_clusters.png"))
        create_3d_bit_position_map(bad_hash_vectors, os.path.join(BASE_DIR, "figures", "final_bad_3d_bit_map.png"))
    analyze_target_stats()
    save_checkpoint(models, max_steps)
    with open(os.path.join(REPORT_PATH, "final_report.json"), 'w') as f:
        json.dump({
            "best_hamming": best_hamming,
            "good_samples": len(good_hash_samples),
            "bad_samples": len(bad_hash_samples),
            "anomalies_found": anomalies_found,
            "recovered_keys": recovered_keys
        }, f, indent=2)
    with open(os.path.join(DRIVE_DIR, "reports", "final_report.json"), 'w') as f:
        json.dump({
            "best_hamming": best_hamming,
            "good_samples": len(good_hash_samples),
            "bad_samples": len(bad_hash_samples),
            "anomalies_found": anomalies_found,
            "recovered_keys": recovered_keys
        }, f, indent=2)
    print(f"Recovery complete! Recovered {len(recovered_keys)} keys. Check {RECOVERED_KEYS_PATH} for results.")

# Block 11: Entry Point
if __name__ == "__main__":
    try:
        targets = load_addresses()
        keymaster_df = load_keymaster_csv()
        if not targets:
            raise ValueError("No valid addresses loaded")
        run_phantom_t5(targets)
    except Exception as e:
        logging.error(f"Execution failed: {e}")
        print(f"Error: {e}. Check {os.path.join(BASE_DIR, 'logs', 'recovery.log')} for details.")

Mounted at /content/drive
Using GPU: NVIDIA L4
Upload addresses.txt (required, 999 P2PKH addresses) and thekeymaster.csv (optional)


Saving thekeymaster.csv to thekeymaster.csv
Saving addresses.txt to addresses.txt
Starting RIPEMPHANTOM-T5 with 999 targets on cuda:0...


Recovery Progress:   0%|          | 0/250000 [00:00<?, ?it/s]

Step 0: Processed 0 good samples, 0 bad samples, Best Hamming: 160, Recovered: 0, Keys/sec: 0.00, ETA: 69.44 hours


Recovery Progress:   0%|          | 0/250000 [00:00<?, ?it/s]
ERROR:root:Execution failed: invalid literal for int() with base 10: b"\x18&\x14\xaf\xf6\xc1\xfdn$\xbc\xd4\xe4\xe2V8@'\xf1m\x85"


Error: invalid literal for int() with base 10: b"\x18&\x14\xaf\xf6\xc1\xfdn$\xbc\xd4\xe4\xe2V8@'\xf1m\x85". Check /content/RIPEMPHANTOM-T5/logs/recovery.log for details.


In [ ]:


# Required imports (all dependencies will be installed at runtime)
import subprocess
import sys
import os
import signal
import time
import random
import json
import hashlib
import math
import warnings
warnings.filterwarnings('ignore')
import traceback

# Print banner immediately
BANNER = """
██████╗ ██╗██████╗ ███████╗███╗   ███╗██████╗ ██╗  ██╗ █████╗ ███╗   ██╗████████╗ ██████╗ ███╗   ███╗
██╔══██╗██║██╔══██╗██╔════╝████╗ ████║██╔══██╗██║  ██║██╔══██╗████╗  ██║╚══██╔══╝██╔═══██╗████╗ ████║
██████╔╝██║██████╔╝█████╗  ██╔████╔██║██████╔╝███████║███████║██╔██╗ ██║   ██║   ██║   ██║██╔████╔██║
██╔══██╗██║██╔═══╝ ██╔══╝  ██║╚██╔╝██║██╔═══╝ ██╔══██║██╔══██║██║╚██╗██║   ██║   ██║   ██║██║╚██╔╝██║
██║  ██║██║██║     ███████╗██║ ╚═╝ ██║██║     ██║  ██║██║  ██║██║ ╚████║   ██║   ╚██████╔╝██║ ╚═╝ ██║
╚═╝  ╚═╝╚═╝╚═╝     ╚══════╝╚═╝     ╚═╝╚═╝     ╚═╝  ╚═╝╚═╝  ╚═╝╚═╝  ╚═══╝   ╚═╝    ╚═════╝ ╚═╝     ╚═╝
                    Bitcoin Private Key Recovery Tool - RIPEMD-160 Edition T5
                             *** For Educational Purposes Only ***
"""
print(BANNER)

# Install required packages - Run this first and immediately
def setup_environment():
    """Install all necessary packages and prepare the environment."""
    print("Setting up environment...")
    packages = [
        "ecdsa==0.18.0",
        "pycryptodome==3.18.0",
        "base58==2.1.1",
        "numba==0.57.1",
        "scikit-learn==1.2.2",
        "statsmodels==0.14.0",
        "umap-learn==0.5.3",
        "matplotlib==3.7.2",
        "seaborn==0.12.2",
        "pandas==2.0.3",
        "tqdm==4.65.0",
        "torch==2.0.1",
    ]

    # Try to install CuPy for GPU acceleration
    try:
        # Check if CUDA is available
        subprocess.check_call(["nvidia-smi"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        # Determine CUDA version for compatible CuPy installation
        cuda_output = subprocess.check_output(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"]).decode('utf-8').strip()
        cuda_version = cuda_output.split(".")[0]

        print(f"CUDA detected version: {cuda_version}")
        # Select appropriate CuPy version
        if int(cuda_version) >= 11:
            packages.append("cupy-cuda11x")
        elif int(cuda_version) >= 10:
            packages.append("cupy-cuda10x")
        else:
            packages.append("cupy")
    except Exception as e:
        print(f"Could not determine CUDA version: {e}. Will use CPU fallback.")

    # Install all packages
    for package in packages:
        try:
            print(f"Installing {package}...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
        except Exception as e:
            print(f"Warning: Failed to install {package}: {e}")

    print("Environment setup complete.")

# Call setup at the beginning
setup_environment()

# Now import everything we need - after setup is complete
print("Importing libraries...")
try:
    import numpy as np
    import pandas as pd
    from tqdm import tqdm
    from collections import defaultdict
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import matplotlib.pyplot as plt
    import seaborn as sns
    from matplotlib.colors import LinearSegmentedColormap
    from mpl_toolkits.mplot3d import Axes3D
    from scipy import stats
    from scipy.spatial.distance import pdist, squareform
    from statsmodels.stats.proportion import proportion_confint
    from sklearn.cluster import DBSCAN
    from sklearn.decomposition import PCA
    from ecdsa import SigningKey, SECP256k1
    from Crypto.Hash import RIPEMD160
    import base58
    from numba import jit, cuda
    import logging
    import threading
    from concurrent.futures import ThreadPoolExecutor
    from google.colab import drive, files
except ImportError as e:
    print(f"Error importing libraries: {e}")
    print("Trying to continue despite import errors...")

# Try to import CuPy, with fallback to NumPy if not available
try:
    import cupy as cp
    print("CuPy successfully imported for GPU acceleration.")
except ImportError:
    # Create a wrapper around NumPy to mimic CuPy
    print("CuPy not available, using NumPy fallback.")
    class CpFallback:
        def __getattr__(self, name):
            return getattr(np, name)
        cuda = type('obj', (object,), {
            'runtime': type('obj', (object,), {
                'getDeviceCount': lambda: 0
            })
        })
    cp = CpFallback()

# Improved constants for better performance and flexibility
class Config:
    """Centralized configuration class for all parameters."""
    # Paths
    BASE_DIR = "/content/RIPEMPHANTOM-T5"
    DRIVE_DIR = "/content/drive/MyDrive/RIPEMPHANTOM-T5"
    ADDR_FILE = os.path.join(BASE_DIR, "addresses.txt")
    CSV_FILE = os.path.join(BASE_DIR, "thekeymaster.csv")
    RECOVERED_KEYS_PATH = os.path.join(BASE_DIR, "recovered_keys.json")
    DRIVE_RECOVERED_KEYS_PATH = os.path.join(DRIVE_DIR, "recovered_keys.json")

    # Algorithm parameters
    MAX_K = 2**61  # Upper bound for private keys
    EXPECTED_PROB = 0.5  # Expected bit probability
    HAMMING_TARGET_RANGE = (41, 49)  # Target range for interesting Hamming distances
    PRIORITY_TARGETS = 100  # Number of addresses to prioritize
    MAX_GOOD_SAMPLES = 50000  # Maximum number of promising samples to keep
    MAX_BAD_SAMPLES = 50000  # Maximum number of unpromising samples to keep

    # Performance parameters
    BATCH_SIZE = 128  # Will be adjusted based on available GPU
    CHECKPOINT_INTERVAL = 300  # Seconds between checkpoints
    CHECKPOINT_STEPS = 500  # Steps between checkpoint saves
    MAX_STEPS = 250_000  # Maximum steps for main loop

    # Analysis parameters
    RUN_SEQUENTIAL_ANALYSIS = True  # Whether to run sequential key analysis
    RUN_ENTROPY_ANALYSIS = True     # Whether to run entropy analysis
    SEQUENTIAL_KEYS_COUNT = 10000   # Number of sequential keys to analyze
    ENTROPY_SAMPLE_SIZE = 20000     # Number of samples for entropy analysis

    # Runtime variables (will be set during initialization)
    device = None
    rng_seed = None
    timestamp = None
    use_gpu = False

    @classmethod
    def initialize(cls):
        """Initialize runtime configuration variables."""
        cls.timestamp = time.strftime("%Y-%m-%d_%H-%M-%S")
        cls.rng_seed = int(time.time())

        # Create necessary directories
        for d in [cls.BASE_DIR, cls.DRIVE_DIR]:
            os.makedirs(os.path.join(d, "checkpoints"), exist_ok=True)
            os.makedirs(os.path.join(d, "logs"), exist_ok=True)
            os.makedirs(os.path.join(d, "figures"), exist_ok=True)
            os.makedirs(os.path.join(d, "reports"), exist_ok=True)
            os.makedirs(os.path.join(d, "datasets"), exist_ok=True)

        # Setup device (GPU/CPU)
        try:
            if torch.cuda.is_available():
                cls.device = torch.device("cuda:0")
                torch.cuda.set_device(0)
                cls.use_gpu = True
                # Adjust batch size based on available GPU memory
                mem_info = torch.cuda.get_device_properties(0).total_memory
                if mem_info > 16e9:  # >16GB
                    cls.BATCH_SIZE = 256
                elif mem_info > 8e9:  # >8GB
                    cls.BATCH_SIZE = 128
                else:  # <8GB
                    cls.BATCH_SIZE = 64
                print(f"Using GPU: {torch.cuda.get_device_name(0)}")
                print(f"Batch size set to {cls.BATCH_SIZE} based on GPU memory")
            else:
                cls.device = torch.device("cpu")
                cls.BATCH_SIZE = 16
                print("No GPU detected, using CPU with reduced batch size")
        except Exception as e:
            cls.device = torch.device("cpu")
            cls.BATCH_SIZE = 16
            print(f"GPU detection failed: {e}, falling back to CPU")

        # Set random seeds
        random.seed(cls.rng_seed)
        np.random.seed(cls.rng_seed)
        torch.manual_seed(cls.rng_seed)
        if cls.use_gpu:
            torch.cuda.manual_seed_all(cls.rng_seed)

        return cls

# Initialize logging
def setup_logging():
    """Configure logging with appropriate handlers and format."""
    log_file = os.path.join(Config.BASE_DIR, "logs", "recovery.log")
    logging.basicConfig(
        filename=log_file,
        level=logging.DEBUG,
        format="%(asctime)s - %(levelname)s - %(message)s"
    )

    # Add console handler for warnings and errors
    console = logging.StreamHandler()
    console.setLevel(logging.WARNING)
    formatter = logging.Formatter('%(levelname)s: %(message)s')
    console.setFormatter(formatter)
    logging.getLogger('').addHandler(console)

    # Log system information
    logging.info("=== RIPEMPHANTOM-T5 Session Started ===")
    if Config.use_gpu:
        logging.info(f"GPU: {torch.cuda.get_device_name(0)}, CUDA: {torch.version.cuda}")
        try:
            logging.info(f"CuPy devices: {cp.cuda.runtime.getDeviceCount()}")
        except:
            logging.info("CuPy not available or failed to initialize")
    else:
        logging.info("Running on CPU")

    return log_file

# Mount Google Drive and prepare workspace
def prepare_workspace():
    """Mount Google Drive and prepare the working directory."""
    try:
        drive.mount('/content/drive', force_remount=True)
        print("Google Drive mounted successfully")
    except Exception as e:
        print(f"Warning: Failed to mount Google Drive: {e}")
        print("Will continue without Drive backup capability")

    # Save configuration
    config_data = {
        "timestamp": Config.timestamp,
        "rng_seed": Config.rng_seed,
        "max_k": Config.MAX_K,
        "version": "RIPEMPHANTOM-T5",
        "hamming_target_range": Config.HAMMING_TARGET_RANGE,
        "batch_size": Config.BATCH_SIZE,
        "device": str(Config.device),
        "use_gpu": Config.use_gpu
    }

    with open(os.path.join(Config.BASE_DIR, "experiment_config.json"), 'w') as f:
        json.dump(config_data, f, indent=2)
    try:
        with open(os.path.join(Config.DRIVE_DIR, "experiment_config.json"), 'w') as f:
            json.dump(config_data, f, indent=2)
    except:
        logging.warning("Could not save config to Drive")

# File management and upload handling
def handle_file_uploads():
    """Prompt for file uploads and handle them appropriately."""
    print("Upload addresses.txt (required, P2PKH addresses) and thekeymaster.csv (optional)")
    uploaded = files.upload()

    if "addresses.txt" not in uploaded:
        logging.error("addresses.txt not uploaded")
        raise FileNotFoundError("addresses.txt is required")

    with open(Config.ADDR_FILE, "wb") as f:
        f.write(uploaded["addresses.txt"])

    if "thekeymaster.csv" in uploaded:
        with open(Config.CSV_FILE, "wb") as f:
            f.write(uploaded["thekeymaster.csv"])
        logging.info("thekeymaster.csv uploaded")
    else:
        logging.info("No thekeymaster.csv uploaded, proceeding without it")

    if "best_matches.csv" in uploaded:
        with open(os.path.join(Config.BASE_DIR, "best_matches.csv"), "wb") as f:
            f.write(uploaded["best_matches.csv"])
        logging.info("best_matches.csv uploaded")

    return uploaded

# Enhanced Key-to-Hash160 Pipeline with better error handling
@jit(nopython=True)
def hamming_distance_and_flips(hash1, hash2):
    """
    Calculate Hamming distance and identify flipped bits between two hash160 values.

    Args:
        hash1: First hash160 byte array
        hash2: Second hash160 byte array

    Returns:
        Tuple of (distance, list of bit positions that differ)
    """
    distance = 0
    flips = []
    for i in range(20):
        xor = hash1[i] ^ hash2[i]
        for j in range(8):
            if xor & (1 << j):
                bit_index = (i * 8) + (7 - j)
                flips.append(bit_index)
                distance += 1
    return distance, flips

def privkey_to_hash160(k, compressed=True):
    """
    Convert a private key to a hash160 (RIPEMD160(SHA256(pubkey))).

    Args:
        k: Private key as integer
        compressed: Whether to use compressed public key format

    Returns:
        hash160 bytes or None if error
    """
    # Proper type checking
    if not isinstance(k, int):
        logging.error(f"Invalid private key type: {type(k)}, value: {k}")
        return None

    try:
        # Validate key range
        if not (1 <= k < SECP256k1.order):
            logging.warning(f"Private key out of range: {k}")
            return None

        # Generate public key
        sk = SigningKey.from_secret_exponent(k, curve=SECP256k1)
        vk = sk.get_verifying_key()

        # Format public key (compressed or uncompressed)
        if compressed:
            prefix = b'\x02' if vk.to_string()[32] % 2 == 0 else b'\x03'
            pubkey = prefix + vk.to_string()[:32]
        else:
            pubkey = b'\x04' + vk.to_string()

        # Calculate hash160
        sha = hashlib.sha256(pubkey).digest()
        ripemd = RIPEMD160.new()
        ripemd.update(sha)
        return ripemd.digest()
    except Exception as e:
        if k is not None:  # Only log if k isn't None (to reduce noise)
            logging.error(f"Error in privkey_to_hash160 for key {k}: {e}")
        return None

def key_to_vector(k):
    """
    Convert a private key to a binary vector representation.

    Args:
        k: Private key as integer

    Returns:
        256-bit binary vector as torch.Tensor
    """
    if not isinstance(k, int):
        logging.error(f"key_to_vector received non-integer: {k}, type: {type(k)}")
        return torch.zeros(256, dtype=torch.float32, device=Config.device)

    return torch.tensor(
        [(k >> i) & 1 for i in range(255, -1, -1)],
        dtype=torch.float32,
        device=Config.device
    )

def hash160_to_vector(h):
    """
    Convert a hash160 bytes to a binary vector representation.

    Args:
        h: hash160 bytes

    Returns:
        160-bit binary vector as torch.Tensor
    """
    if not isinstance(h, bytes) or len(h) != 20:
        logging.error(f"Invalid hash160: {h}, type: {type(h)}")
        return torch.zeros(160, dtype=torch.float32, device=Config.device)

    return torch.tensor(
        [(b >> i) & 1 for b in h for i in range(7, -1, -1)],
        dtype=torch.float32,
        device=Config.device
    )

def vector_to_hash160(v):
    """
    Convert a binary vector back to hash160 bytes.

    Args:
        v: 160-bit binary vector as torch.Tensor

    Returns:
        hash160 bytes
    """
    hash_bytes = bytearray(20)
    v_cpu = v.cpu().numpy()

    for i in range(160):
        byte_idx = i // 8
        bit_idx = 7 - (i % 8)
        if v_cpu[i] > 0.5:
            hash_bytes[byte_idx] |= (1 << bit_idx)

    return bytes(hash_bytes)

def hash160_to_address(h160, testnet=False):
    """
    Convert a hash160 to a Bitcoin address.

    Args:
        h160: hash160 bytes
        testnet: Whether to use testnet format

    Returns:
        Bitcoin address as string
    """
    version = b'\x6f' if testnet else b'\x00'
    data = version + h160
    checksum = hashlib.sha256(hashlib.sha256(data).digest()).digest()[:4]
    return base58.b58encode(data + checksum).decode('utf-8')

# Neural Models with improved architecture
class HashPredictorNet(nn.Module):
    """Neural network to predict hash160 from private key."""
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Dropout(0.2),  # Added dropout for regularization
            nn.Linear(512, 512),  # Added extra layer for more capacity
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 160),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

class EntropyAnalysisNet(nn.Module):
    """Autoencoder for anomaly detection in hash160 distributions."""
    def __init__(self):
        super().__init__()
        # Encoder with batch normalization for better training
        self.encoder = nn.Sequential(
            nn.Linear(160, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.2)
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(64, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 160),
            nn.Sigmoid()
        )

        # Anomaly detector
        self.anomaly_detector = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        anomaly_score = self.anomaly_detector(encoded)
        return decoded, anomaly_score, encoded

# Adaptive Mutation Engine - for finding optimal bit-flipping strategies
class AdaptiveMutationEngine:
    """Engine for generating and evolving private key mutations."""
    def __init__(self):
        self.bit_bias = defaultdict(float)
        self.bad_bit_bias = defaultdict(float)
        self.xor_chain = []
        self.mutation_log = []
        self.mutation_bank = []
        self.max_mutation_bank_size = 100
        self.max_xor_chain_size = 100

    def update_bit_bias(self, flips, reward=1.0, is_good=True):
        """Update bit bias statistics based on results."""
        target = self.bit_bias if is_good else self.bad_bit_bias
        for bit in flips:
            if isinstance(bit, int):  # Ensure bit is an integer
                target[bit] += reward

    def add_xor_delta(self, prev_k, new_k, reward=1.0):
        """Add a successful XOR delta to the chain."""
        if not (isinstance(prev_k, int) and isinstance(new_k, int)):
            logging.error(f"Invalid XOR delta inputs: prev_k={type(prev_k)}, new_k={type(new_k)}")
            return

        delta = prev_k ^ new_k
        self.xor_chain.append((delta, reward))

        # Maintain maximum size
        if len(self.xor_chain) > self.max_xor_chain_size:
            self.xor_chain.pop(0)

    def log_mutation_event(self, k_old, k_new, mutation_type, flips, delta_hamming):
        """Log a mutation event for analysis."""
        try:
            # Only log if we have valid integers
            if not (isinstance(k_old, int) and isinstance(k_new, int)):
                return

            entry = {
                "time": time.time(),
                "type": mutation_type,
                "delta_hamming": delta_hamming,
                "flips": flips,
                "delta_k": k_old ^ k_new
            }
            self.mutation_log.append(entry)

            # Keep log size manageable
            if len(self.mutation_log) > 5000:
                self.mutation_log.pop(0)
        except Exception as e:
            logging.error(f"Error in log_mutation_event: {e}")

    def record_mutation_result(self, mutation_vector, delta_hamming):
        """Record the effect of a mutation strategy."""
        if not isinstance(mutation_vector, int):
            logging.error(f"Invalid mutation vector type: {type(mutation_vector)}")
            return

        # Update existing entry or add new one
        found = False
        for i, (vec, score) in enumerate(self.mutation_bank):
            if vec == mutation_vector:
                self.mutation_bank[i] = (vec, score + delta_hamming)
                found = True
                break

        if not found:
            self.mutation_bank.append((mutation_vector, delta_hamming))

        # Keep bank size limited and sorted by effectiveness
        if len(self.mutation_bank) > self.max_mutation_bank_size:
            self.mutation_bank.sort(key=lambda x: -x[1])
            self.mutation_bank = self.mutation_bank[:self.max_mutation_bank_size]

    def get_top_bit_bias(self, n=10, is_good=True):
        """Get the most effective bit positions to flip."""
        target = self.bit_bias if is_good else self.bad_bit_bias
        return sorted(target.items(), key=lambda x: -x[1])[:n]

    def get_top_xor_deltas(self, n=10):
        """Get the most effective XOR patterns."""
        return sorted(self.xor_chain, key=lambda x: -x[1])[:n]

    def get_best_mutations(self, n=10):
        """Get the best mutation strategies."""
        # Filter for valid integers
        valid_mutations = [(vec, score) for vec, score in self.mutation_bank if isinstance(vec, int)]
        return sorted(valid_mutations, key=lambda x: -x[1])[:n]

    def mutate_key(self, k):
        """Apply intelligent mutations to a private key."""
        if not isinstance(k, int):
            logging.error(f"mutate_key received non-integer: {k}, type: {type(k)}")
            return random.randint(1, Config.MAX_K - 1)

        # Start with the original key
        new_k = k

        # Apply different mutation strategies with probabilities

        # 1. Simple random offset (50% chance)
        if random.random() < 0.5:
            new_k += random.randint(-1000000, 1000000)

        # 2. Apply best mutation vectors from history (30% chance each)
        for vec, _ in self.get_best_mutations(n=5):
            if random.random() < 0.3 and isinstance(vec, int):
                new_k ^= vec

        # 3. Apply successful XOR deltas from history (30% chance each)
        for delta, _ in self.get_top_xor_deltas(n=5):
            if random.random() < 0.3 and isinstance(delta, int):
                new_k ^= delta

        # 4. Flip bits with high success bias (30% chance each)
        for bit, _ in self.get_top_bit_bias(n=5, is_good=True):
            if random.random() < 0.3 and isinstance(bit, int):
                new_k ^= (1 << bit)

        # 5. Avoid bits with negative bias (10% chance each)
        for bit, _ in self.get_top_bit_bias(n=5, is_good=False):
            if random.random() < 0.1 and isinstance(bit, int):
                new_k ^= (1 << bit)

        # 6. Random byte mutation (20% chance)
        if random.random() < 0.2:
            new_k ^= random.getrandbits(8)

        # Ensure result is valid and in range
        if not isinstance(new_k, int):
            logging.error(f"mutate_key produced non-integer: {new_k}")
            return random.randint(1, Config.MAX_K - 1)

        return new_k if 1 <= new_k < Config.MAX_K else random.randint(1, Config.MAX_K - 1)

    def generate_key(self, seed_k, transformer_model=None):
        """Generate a new private key using mutation and optional ML guidance."""
        if not isinstance(seed_k, int):
            logging.error(f"generate_key received non-integer: {seed_k}, type: {type(seed_k)}")
            return random.randint(1, Config.MAX_K - 1)

        # First apply mutations
        k = self.mutate_key(seed_k)

        # Then apply transformer model guidance if available
        if transformer_model:
            try:
                with torch.no_grad():
                    k_tensor = key_to_vector(k).unsqueeze(0)
                    prediction = transformer_model(k_tensor).squeeze(0)

                    # Use high-confidence predictions to guide bit flipping
                    for i in range(160):
                        if prediction[i].item() > 0.95 and random.random() < 0.5:
                            k ^= (1 << (i % 160))
            except Exception as e:
                logging.warning(f"Transformer model error: {e}")

        # Final validation
        if not isinstance(k, int):
            logging.error(f"generate_key produced non-integer: {k}")
            return random.randint(1, Config.MAX_K - 1)

        logging.debug(f"Generated key: {k}")
        return k if 1 <= k < Config.MAX_K else random.randint(1, Config.MAX_K - 1)

    def reset(self):
        """Reset the mutation engine state."""
        self.bit_bias = defaultdict(float)
        self.bad_bit_bias = defaultdict(float)
        self.xor_chain = []
        self.mutation_log = []
        self.mutation_bank = []

    def to_dict(self):
        """Convert state to dictionary for saving."""
        return {
            "bit_bias": dict(self.bit_bias),
            "bad_bit_bias": dict(self.bad_bit_bias),
            "xor_chain": self.xor_chain,
            "mutation_bank": [(hex(vec), score) for vec, score in self.mutation_bank if isinstance(vec, int)]
        }

    def from_dict(self, state_dict):
        """Restore state from dictionary."""
        self.bit_bias = defaultdict(float, state_dict.get("bit_bias", {}))
        self.bad_bit_bias = defaultdict(float, state_dict.get("bad_bit_bias", {}))
        self.xor_chain = state_dict.get("xor_chain", [])

        # Carefully restore mutation bank
        self.mutation_bank = []
        for vec_hex, score in state_dict.get("mutation_bank", []):
            try:
                if isinstance(vec_hex, str) and vec_hex.startswith('0x'):
                    self.mutation_bank.append((int(vec_hex, 16), score))
            except (ValueError, TypeError) as e:
                logging.warning(f"Skipping invalid mutation vector: {vec_hex}, error: {e}")

# Address and Target Management
class TargetManager:
    """Manage Bitcoin addresses and track best matches."""
    def __init__(self):
        self.targets = []  # List of (address, hash160) tuples
        self.target_stats = {}  # Statistics for each target
        self.recovered_keys = {}  # Successfully recovered keys
        self.priority_targets = Config.PRIORITY_TARGETS  # Number of addresses to prioritize

    def load_addresses(self):
        """Load Bitcoin addresses from file."""
        targets = []
        with open(Config.ADDR_FILE, 'r') as f:
            for line in f:
                addr = line.strip()
                if not addr:
                    continue

                try:
                    # Decode and validate P2PKH address
                    decoded = base58.b58decode(addr)
                    if len(decoded) < 25 or decoded[0] != 0x00 or len(decoded[1:-4]) != 20:
                        logging.warning(f"Invalid address: {addr}")
                        continue

                    # Extract the hash160 part
                    h160 = decoded[1:-4]
                    targets.append((addr, h160))
                except Exception as e:
                    logging.warning(f"Error decoding {addr}: {e}")
                    continue

        if not targets:
            logging.error("No valid P2PKH addresses found")
            raise ValueError("No valid addresses loaded")

        logging.info(f"Loaded {len(targets)} valid P2PKH addresses")
        self.targets = targets
        return targets

    def load_keymaster_csv(self):
        """Load known keys from thekeymaster.csv if available."""
        if os.path.exists(Config.CSV_FILE):
            try:
                df = pd.read_csv(Config.CSV_FILE)
                logging.info(f"Loaded thekeymaster.csv with {len(df)} entries")
                return df
            except Exception as e:
                logging.warning(f"Error loading thekeymaster.csv: {e}")
        return None

    def init_target_stats(self):
        """Initialize statistics tracking for all targets."""
        for addr, _ in self.targets:
            self.target_stats[addr] = {
                "best": 160,  # Best Hamming distance (lower is better)
                "best_k": -1,  # Private key with best distance
                "scans": 0,    # Number of scans performed
                "hamming_history": [],  # History of Hamming distances
                "last_improvement": time.time()  # Timestamp of last improvement
            }

    def update_target_stat(self, addr, k, ham, compressed):
        """Update statistics for a target address."""
        if not isinstance(k, int):
            logging.error(f"update_target_stat received non-integer key: {k}")
            return

        # Keep history limited
        if len(self.target_stats[addr]["hamming_history"]) >= 1000:
            self.target_stats[addr]["hamming_history"].pop(0)

        self.target_stats[addr]["hamming_history"].append(ham)

        # If this is a new best result
        if ham < self.target_stats[addr]["best"]:
            self.target_stats[addr]["best"] = ham
            self.target_stats[addr]["best_k"] = k
            self.target_stats[addr]["last_improvement"] = time.time()

            # Log the improvement
            logging.info(f"New best for {addr}: Hamming {ham}, Key {hex(k)}, Compressed: {compressed}")

            # If exact match, record the recovered key
            if ham == 0:
                self.recovered_keys[addr] = {"key": hex(k), "compressed": compressed}

                # Save to disk
                with open(Config.RECOVERED_KEYS_PATH, 'w') as f:
                    json.dump(self.recovered_keys, f, indent=2)
                try:
                    with open(Config.DRIVE_RECOVERED_KEYS_PATH, 'w') as f:
                        json.dump(self.recovered_keys, f, indent=2)
                except:
                    pass

                logging.info(f"Recovered key for {addr}: {hex(k)}, Compressed: {compressed}")

        # Increment scan counter
        self.target_stats[addr]["scans"] += 1

    def get_priority_targets(self, top_n=None):
        """Get the most promising targets to focus on."""
        if top_n is None:
            top_n = self.priority_targets

        if not self.target_stats:
            # If no stats yet, return random sample
            return random.sample(self.targets, min(top_n, len(self.targets)))

        # Sort by best Hamming distance and recency of improvement
        sorted_targets = sorted(
            self.target_stats.items(),
            key=lambda x: (x[1]["best"], -x[1]["last_improvement"])
        )

        # Select addresses with promising Hamming distances
        top_addrs = set(addr for addr, stats in sorted_targets[:top_n]
                      if stats["best"] <= 50)  # Focus on promising targets

        return [(addr, h) for addr, h in self.targets if addr in top_addrs]

    def analyze_target_stats(self):
        """Analyze target statistics and produce a report."""
        from scipy import stats as scipy_stats

        if not self.target_stats:
            return {}

        analysis = {
            "global": {
                "best_hamming": min([s["best"] for s in self.target_stats.values()]),
                "avg_hamming": sum([s["best"] for s in self.target_stats.values()]) / len(self.target_stats),
                "total_scans": sum([s["scans"] for s in self.target_stats.values()]),
                "timestamp": time.time()
            },
            "targets": {}
        }

        for addr, stats in self.target_stats.items():
            if not stats["hamming_history"]:
                continue

            history = stats["hamming_history"]
            analysis["targets"][addr] = {
                "best": stats["best"],
                "mean": np.mean(history),
                "median": np.median(history),
                "std_dev": np.std(history),
                "scans": stats["scans"]
            }

            # Calculate trend if enough data points
            if len(history) > 10:
                x = np.arange(len(history))
                y = np.array(history)
                try:
                    slope, _, r_value, p_value, _ = scipy_stats.linregress(x, y)
                    analysis["targets"][addr].update({
                        "trend_slope": slope,
                        "trend_r_squared": r_value**2,
                        "trend_p_value": p_value
                    })
                except Exception as e:
                    analysis["targets"][addr].update({
                        "trend_slope": 0.0,
                        "trend_r_squared": 0.0,
                        "trend_p_value": 1.0
                    })

        # Save the analysis
        with open(os.path.join(Config.BASE_DIR, "reports", "target_analysis.json"), 'w') as f:
            json.dump(analysis, f, indent=2)

        try:
            with open(os.path.join(Config.DRIVE_DIR, "reports", "target_analysis.json"), 'w') as f:
                json.dump(analysis, f, indent=2)
        except:
            pass

        return analysis

    def load_recovered_keys(self):
        """Load any previously recovered keys."""
        if os.path.exists(Config.RECOVERED_KEYS_PATH):
            try:
                with open(Config.RECOVERED_KEYS_PATH, 'r') as f:
                    self.recovered_keys = json.load(f)
                logging.info(f"Loaded {len(self.recovered_keys)} previously recovered keys")
            except Exception as e:
                logging.warning(f"Error loading recovered keys: {e}")

    def export_best_matches(self, file_path=None):
        """Export the best matches to a CSV file."""
        if file_path is None:
            file_path = os.path.join(Config.BASE_DIR, "best_matches.csv")

        data = []
        for addr, stats in self.target_stats.items():
            if stats["best_k"] != -1:  # If we have a candidate
                # Calculate similarity percentage
                similarity = ((160 - stats["best"]) / 160) * 100

                # Estimate entropy level (simple model)
                entropy = 1.0  # Default full entropy
                if stats["best"] < 40:  # If suspiciously close
                    entropy = 0.5 + (stats["best"] / 80)  # Rough estimate

                data.append({
                    "Address": addr,
                    "Private Key": hex(stats["best_k"]),
                    "Hamming Distance": stats["best"],
                    "Similarity %": similarity,
                    "Entropy Level": entropy,
                    "Exact Match": stats["best"] == 0
                })

        if data:
            df = pd.DataFrame(data)
            df.to_csv(file_path, index=False)
            logging.info(f"Exported {len(data)} best matches to {file_path}")

            # Also save to Drive if possible
            try:
                drive_path = os.path.join(Config.DRIVE_DIR, "best_matches.csv")
                df.to_csv(drive_path, index=False)
            except:
                pass

            return df
        return None
        # [Existing code ... setup_environment(), imports, class/function definitions ...]

if __name__ == "__main__":
    # Initialize configuration
    Config.initialize() # [cite: 14, 85]

    # Setup logging
    log_file = setup_logging() # [cite: 21, 92]
    logging.info(f"Log file: {log_file}")

    # Prepare workspace (mount Drive, etc.)
    prepare_workspace() # [cite: 23, 94]

    # Handle file uploads (this is interactive, ensure it's what you want for Colab non-stop runs)
    # If you want to automate file paths, you might adjust handle_file_uploads
    # or set paths directly in Config
    # try:
    #     handle_file_uploads()
    # except FileNotFoundError as e:
    #     logging.error(f"Critical file not found: {e}")
    #     print(f"Critical file not found: {e}. Please upload required files. Exiting.")
    #     sys.exit(1)

    # Create the main Phantom instance and run it
    phantom_runner = Phantom() # [cite: 298]

    # You can adjust max_steps here or through the Config class if you add it there
    phantom_runner.run(max_steps=Config.MAX_STEPS) # [cite: 13, 302]
                                                 # (MAX_STEPS is defined in Config)

    print("RIPEMPHANTOM-T5 processing complete.")


██████╗ ██╗██████╗ ███████╗███╗   ███╗██████╗ ██╗  ██╗ █████╗ ███╗   ██╗████████╗ ██████╗ ███╗   ███╗
██╔══██╗██║██╔══██╗██╔════╝████╗ ████║██╔══██╗██║  ██║██╔══██╗████╗  ██║╚══██╔══╝██╔═══██╗████╗ ████║
██████╔╝██║██████╔╝█████╗  ██╔████╔██║██████╔╝███████║███████║██╔██╗ ██║   ██║   ██║   ██║██╔████╔██║
██╔══██╗██║██╔═══╝ ██╔══╝  ██║╚██╔╝██║██╔═══╝ ██╔══██║██╔══██║██║╚██╗██║   ██║   ██║   ██║██║╚██╔╝██║
██║  ██║██║██║     ███████╗██║ ╚═╝ ██║██║     ██║  ██║██║  ██║██║ ╚████║   ██║   ╚██████╔╝██║ ╚═╝ ██║
╚═╝  ╚═╝╚═╝╚═╝     ╚══════╝╚═╝     ╚═╝╚═╝     ╚═╝  ╚═╝╚═╝  ╚═╝╚═╝  ╚═══╝   ╚═╝    ╚═════╝ ╚═╝     ╚═╝
                    Bitcoin Private Key Recovery Tool - RIPEMD-160 Edition T5
                             *** For Educational Purposes Only ***

Setting up environment...
Could not determine CUDA version: [Errno 2] No such file or directory: 'nvidia-smi'. Will use CPU fallback.
Installing ecdsa==0.18.0...
Installing pycryptodome==3.18.0...
Installing base58==2.1.1...


KeyboardInterrupt: 

In [13]:
# Cell 1: Install Dependencies
!pip install -q base58 pycryptodome ecdsa tqdm torch matplotlib scipy pandas scikit-learn umap-learn transformers statsmodels

In [22]:
# -- Export script to file function --
def export_script():
    """Export the current script to a file for standalone execution"""
    script_path = f"{BASE_DIR}/ripemphantom-t5.py"
    with open(script_path, 'w') as f:
        f.write(open(__file__).read())
    print(f"Script exported to {script_path}")
    return script_pathdef run_pattern_correlation_analysis(sample_size=5000):
    """
    Analyze correlations between input patterns and output hash160 patterns
    """
    print(f"Performing pattern correlation analysis with {sample_size} samples...")

    # Generate keys with specific pattern types
    pattern_categories = {
        "random": [],           # Fully random private keys
        "low_entropy": [],      # Keys with many repeated bits
        "high_set_bits": [],    # Keys with many 1s
        "low_set_bits": [],     # Keys with many 0s
        "pattern_bits": []      # Keys with regular patterns
    }

    # Generate sample data
    for _ in tqdm(range(sample_size), desc="Generating patterns"):
        # Random key
        k_random = random.randint(1, MAX_K)
        h_random = privkey_to_hash160(k_random)
        if h_random:
            pattern_categories["random"].append((k_random, h_random))

        # Low entropy (repeating pattern)
        pattern = random.randint(0, 2**32-1)
        k_low_entropy = 0
        for i in range(8):
            k_low_entropy = (k_low_entropy << 32) | pattern
        h_low_entropy = privkey_to_hash160(k_low_entropy)
        if h_low_entropy:
            pattern_categories["low_entropy"].append((k_low_entropy, h_low_entropy))

        # High set bits
        k_high_bits = random.randint(1, MAX_K) | (MAX_K // 2)  # Force many high bits
        h_high_bits = privkey_to_hash160(k_high_bits)
        if h_high_bits:
            pattern_categories["high_set_bits"].append((k_high_bits, h_high_bits))

        # Low set bits
        k_low_bits = random.randint(1, MAX_K) & (MAX_K // 1000)  # Force many low bits
        h_low_bits = privkey_to_hash160(k_low_bits)
        if h_low_bits:
            pattern_categories["low_set_bits"].append((k_low_bits, h_low_bits))

        # Pattern bits (alternating)
        k_pattern = 0
        for i in range(256):
            if i % 2 == 0:
                k_pattern |= (1 << i)
        k_pattern = (k_pattern & random.randint(0, 2**64-1)) | random.randint(0, 2**32-1)
        h_pattern = privkey_to_hash160(k_pattern)
        if h_pattern:
            pattern_categories["pattern_bits"].append((k_pattern, h_pattern))

    # Analyze bit distributions for each category
    bit_stats = {}
    for category, samples in pattern_categories.items():
        if not samples:
            continue

        # Extract hash160 vectors
        hash_vecs = [hash160_to_vector(h).numpy() for _, h in samples]
        hash_matrix = np.stack(hash_vecs)

        # Calculate bit probabilities
        bit_probs = np.mean(hash_matrix, axis=0)

        # Calculate entropy and statistical tests
        bit_entropy = [-p*np.log2(p) - (1-p)*np.log2(1-p) if 0 < p < 1 else 0
                      for p in bit_probs]

        # Run chi-square tests
        chi_square_results = []
        for i, p in enumerate(bit_probs):
            observed = np.array([sum(hash_matrix[:, i]), len(hash_matrix) - sum(hash_matrix[:, i])])
            expected = np.array([len(hash_matrix) * 0.5, len(hash_matrix) * 0.5])

            chi2, p_value = stats.chisquare(observed, expected)
            chi_square_results.append({
                "bit": i,
                "chi2": float(chi2),
                "p_value": float(p_value),
                "significant": p_value < 0.01
            })

        bit_stats[category] = {
            "sample_size": len(samples),
            "bit_probabilities": bit_probs.tolist(),
            "bit_entropy": bit_entropy,
            "total_entropy": sum(bit_entropy),
            "chi_square_tests": chi_square_results,
            "significant_deviations": [r for r in chi_square_results if r["significant"]]
        }

    return bit_stats            # Run chi-square tests
            chi_square_results = []
            for i, p in enumerate(bit_probs):
                observed = np.array([sum(hash_matrix[:, i]), len(hash_matrix) - sum(hash_matrix[:, i])])
                expected = np.array([len(hash_matrix) * 0.5, len(hash_matrix) * 0.5])

                chi2, p_value = scipy_stats.chisquare(observed, expected)
                chi_square_results.append({
                    "bit": i,
                    "chi2": float(chi2),
                    "p_value": float(p_value),
                    "significant": p_value < 0.01
                })# Remove scipy_stats reference
# from scipy import stats as scipy_stats# -- Global variables --
# Configuration and paths
BASE_DIR = None  # Will be set in setup_environment
CHECKPOINT_PATH = None
LOG_PATH = None
REPORT_PATH = None

# Tracking variables
entropy_counter = np.zeros(160, dtype=int)
entropy_total = 0
entropy_samples = 0
step = 0  # Track training steps

# Mutation memory
bit_bias = defaultdict(float)   # Tracks effective bit flips
xor_chain = []                  # Stores successful XOR deltas
mutation_bank = []              # Stores effective mutation vectors
mutation_log = []               # Tracks mutation performance
target_stats = {}               # Address tracking statisticsif __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\nResearch interrupted by user.")
        print("Saving current state...")
        # Try to save current state if models initialized
        try:
            if 'models' in locals() or 'models' in globals():
                save_checkpoint(models, step)
                print("Checkpoint saved successfully.")
        except Exception as e:
            print(f"Could not save checkpoint: {e}")
        print("Exiting gracefully.")
    except Exception as e:
        print(f"\nAn unexpected error occurred: {e}")
        import traceback
        traceback.print_exc()
        print("\nExiting with error.")    # Check for addresses file
    if os.path.exists(ADDR_FILE):
        print(f"Found addresses file: {ADDR_FILE}")
    else:
        print("No addresses.txt found. Upload or create one.")
        if IN_COLAB:
            print("Waiting for file upload...")
            uploaded = files.upload()
            if not uploaded:
                print("No file uploaded. Exiting.")
                return
        else:
            return

    # Load addresses
    targets = load_addresses(ADDR_FILE)
    if not targets:
        print("No valid addresses found in file.")
        return

    print(f"Loaded {len(targets)} target addresses.")

    # Prompt for research mode
    print("\nSelect research mode:")
    print("1. Balanced - General research with equal focus on all aspects")
    print("2. Entropy - Focus on statistical entropy analysis and anomaly detection")
    print("3. Cluster - Focus on bit correlation and clustering patterns")
    print("4. Statistical - Run only statistical test suite")

    try:
        mode_choice = int(input("Enter choice (1-4) [default=1]: ") or "1")
        if mode_choice not in range(1, 5):
            mode_choice = 1
    except:
        mode_choice = 1

    # Map choice to mode
    mode_map = {
        1: "balanced",
        2: "entropy",
        3: "cluster",
        4: "statistical"
    }

    research_mode = mode_map[mode_choice]
    print(f"Selected mode: {research_mode}")

    # For statistical mode, run only statistical analysis
    if research_mode == "statistical":
        try:
            sample_size = int(input("Enter sample size for statistical analysis [default=10000]: ") or "10000")
        except:
            sample_size = 10000

        run_statistical_analysis(sample_size)
        return

    # For normal research modes, get steps
    try:
        max_steps = int(input("Enter maximum steps to run [default=100000]: ") or "100000")
    except:
        max_steps = 100000

    # Run main research
    print(f"\nStarting RIPEMPHANTOM-T5 with {len(targets)} targets, {max_steps} steps, mode: {research_mode}")
    results = run_phantom_t5(targets, max_steps=max_steps, research_mode=research_mode)

    # Print final results
    print("\nResearch complete!")
    print(f"Best Hamming distance: {results['best_hamming']}")
    print(f"Keys analyzed: {results['hash_samples']}")
    print(f"Anomalies found: {results['anomalies_found']}")

    # Export results
    with open(f"{BASE_DIR}/reports/final_results.json", 'w') as f:
        # Convert non-serializable objects to serializable form
        serializable_stats = {}
        for addr, stats in results['target_stats'].items():
            serializable_stats[addr] = {
                'best': stats['best'],
                'best_k': stats['best_k'],
                'scans': stats['scans'],
                'last_10_avg': stats['last_10_avg'],
                'history_length': len(stats['hamming_history'])
            }

        final_results = {
            'best_hamming': results['best_hamming'],
            'hash_samples': results['hash_samples'],
            'anomalies_found': results['anomalies_found'],
            'target_stats': serializable_stats,
            'timestamp': time.time(),
            'config': CONFIG
        }

        json.dump(final_results, f, indent=2)

    print(f"Results saved to {BASE_DIR}/reports/final_results.json")

    # Export script for reuse
    export_script()def print_banner():
    """Display program banner and version info"""
    banner = """
██████╗ ██╗██████╗ ███████╗███╗   ███╗██████╗ ██╗  ██╗ █████╗ ███╗   ██╗████████╗ ██████╗ ███╗   ███╗
██╔══██╗██║██╔══██╗██╔════╝████╗ ████║██╔══██╗██║  ██║██╔══██╗████╗  ██║╚══██╔══╝██╔═══██╗████╗ ████║
██████╔╝██║██████╔╝█████╗  ██╔████╔██║██████╔╝███████║███████║██╔██╗ ██║   ██║   ██║   ██║██╔████╔██║
██╔══██╗██║██╔═══╝ ██╔══╝  ██║╚██╔╝██║██╔═══╝ ██╔══██║██╔══██║██║╚██╗██║   ██║   ██║   ██║██║╚██╔╝██║
██║  ██║██║██║     ███████╗██║ ╚═╝ ██║██║     ██║  ██║██║  ██║██║ ╚████║   ██║   ╚██████╔╝██║ ╚═╝ ██║
╚═╝  ╚═╝╚═╝╚═╝     ╚══════╝╚═╝     ╚═╝╚═╝     ╚═╝  ╚═╝╚═╝  ╚═╝╚═╝  ╚═══╝   ╚═╝    ╚═════╝ ╚═╝     ╚═╝
╔════════════════════════════════════════════════════════════════════════════════════════════════════╗
║                  RIPEMD-160 ENTROPY ANALYSIS & NON-UNIFORMITY RESEARCH PLATFORM                    ║
║                       T5 EDITION - Nation-State Threat Level Research Tool                         ║
╚════════════════════════════════════════════════════════════════════════════════════════════════════╝
"""
    print(banner)
    print(f"Version: {CONFIG['version']} | Timestamp: {CONFIG['timestamp']}")
    print(f"{'='*89}")

def main():
    """Main entry point for RIPEMPHANTOM-T5"""
    # Setup environment
    setup_environment()

    # Handle command line arguments if provided
    import argparse
    parser = argparse.ArgumentParser(description='RIPEMPHANTOM-T5 Entropy Analysis Tool')
    parser.add_argument('--mode', type=int, choices=[1, 2, 3, 4], default=1,
                        help='Research mode: 1=Balanced, 2=Entropy, 3=Cluster, 4=Statistical')
    parser.add_argument('--steps', type=int, default=100000,
                        help='Maximum number of steps to run')
    parser.add_argument('--samples', type=int, default=10000,
                        help='Sample size for statistical analysis')
    parser.add_argument('--file', type=str, default=ADDR_FILE,
                        help='Path to addresses.txt file')

    # Parse args if available, otherwise use interactive mode
    if len(sys.argv) > 1:
        args = parser.parse_args()
        mode_choice = args.mode
        max_steps = args.steps
        sample_size = args.samples
        ADDR_FILE = args.file

        # Map choice to mode
        mode_map = {
            1: "balanced",
            2: "entropy",
            3: "cluster",
            4: "statistical"
        }

        research_mode = mode_map[mode_choice]
        print(f"Command-line mode: {research_mode}, Steps: {max_steps}, File: {ADDR_FILE}")

        # Load addresses
        targets = load_addresses(ADDR_FILE)
        if not targets:
            print("No valid addresses found in file.")
            return

        print(f"Loaded {len(targets)} target addresses.")

        # Run appropriate mode
        if research_mode == "statistical":
            run_statistical_analysis(sample_size)
        else:
            run_phantom_t5(targets, max_steps=max_steps, research_mode=research_mode)

        return

    # Interactive mode
    print_banner()            # Update progress bar
            pbar.set_postfix({
                'best': best_hamming,
                'p_loss': f'{predictor_loss:.4f}',
                't_loss': f'{transformer_loss:.4f}',
                'anomalies': anomalies_found
            })

            # Visualization and checkpointing with adaptive scheduling
            if step % 1000 == 0 or (step < 1000 and step % 100 == 0):
                # More frequent checkpoints in early stages
                save_checkpoint(models, step)
                analyze_target_statistics()

                # Visualize bit bias patterns
                save_bit_bias_heatmap()

                # Cluster visualization
                if len(hash_vectors) > 100:
                    # Alternate between different visualization methods
                    viz_method = 'tsne' if step % 2000 == 0 else 'umap'
                    visualize_hash_clusters(
                        hash_vectors[-500:],
                        save_path=f"{BASE_DIR}/figures/hash_clusters_{step}.png",
                        method=viz_method
                    )

            # Log progress to file
            if step % 100 == 0:
                log_event("progress", {
                    "step": step,
                    "best_hamming": best_hamming,
                    "anomalies": anomalies_found,
                    "transformer_loss": transformer_loss,
                    "predictor_loss": predictor_loss,
                    "timestamp": time.time()
                })

            step += 1
            pbar.update(1)

    # Final analysis and reporting
    final_stats = analyze_target_statistics()
    save_bit_bias_heatmap()

    if len(hash_vectors) > 100:
        visualize_hash_clusters(hash_vectors[-1000:],
                               save_path=f"{BASE_DIR}/figures/final_hash_clusters.png")

    # Save final models
    save_checkpoint(models, step)

    # Print summary
    print("\nRIPEMPHANTOM-T5 Research Complete!")
    print(f"{'='*50}")
    print(f"Best Hamming distance: {final_stats['global']['best_hamming']}")
    print(f"Total keys analyzed: {hash_samples}")
    print(f"Anomalies detected: {anomalies_found}")
    print(f"{'='*50}")

    # Display best results for each target
    print("\nBest results per target:")
    for addr, stats in target_stats.items():
        if stats["best"] < 160:
            addr_short = addr[:6] + "..." + addr[-6:]
            print(f"{addr_short}: Hamming={stats['best']}, Key={hex(stats['best_k'])}")

    return {
        'best_hamming': final_stats['global']['best_hamming'],
        'hash_samples': hash_samples,
        'anomalies_found': anomalies_found,
        'target_stats': target_stats
    }            # -- Train neural models --

            # 1. Hash predictor training
            predictor_loss = 0
            if step % 2 == 0 and len(batch_data) >= 4:  # Train every other batch
                keys_tensor = torch.stack([key_to_vector(k) for k, _, _ in batch_data])
                hash_tensor = torch.stack([h for _, _, h in batch_data])

                # Forward pass
                optimizers['predictor_opt'].zero_grad()
                pred = models['predictor_net'](keys_tensor)
                loss = F.binary_cross_entropy(pred, hash_tensor)

                # Backward pass
                loss.backward()
                optimizers['predictor_opt'].step()
                predictor_loss = loss.item()
                losses['predictor'].append(predictor_loss)

            # 2. Transformer predictor training
            transformer_loss = 0
            if step % 3 == 0 and len(batch_data) >= 4:  # Specialized training schedule
                keys_tensor = torch.stack([key_to_vector(k) for k, _, _ in batch_data])
                hash_tensor = torch.stack([h for _, _, h in batch_data])

                # Forward pass
                optimizers['transformer_opt'].zero_grad()
                pred = models['transformer_net'](keys_tensor)
                loss = F.binary_cross_entropy(pred, hash_tensor)

                # Add L2 regularization to prevent overfitting
                l2_lambda = 0.0001
                l2_reg = torch.tensor(0., requires_grad=True)
                for param in models['transformer_net'].parameters():
                    l2_reg = l2_reg + torch.norm(param, 2)
                loss = loss + l2_lambda * l2_reg

                # Backward pass
                loss.backward()

                # Gradient clipping for stability
                torch.nn.utils.clip_grad_norm_(models['transformer_net'].parameters(), max_norm=1.0)

                optimizers['transformer_opt'].step()
                transformer_loss = loss.item()
                losses['transformer'].append(transformer_loss)

            # 3. Entropy network training (autoencoder)
            entropy_loss = 0
            if step % 2 == 1 and len(batch_data) >= 4:  # Alternate with predictor
                hash_tensor = torch.stack([h for _, _, h in batch_data])

                # Forward pass
                optimizers['entropy_opt'].zero_grad()
                reconstructed, anomaly_score, encoded = models['entropy_net'](hash_tensor)

                # Reconstruction loss (encourages learning normal patterns)
                loss = F.mse_loss(reconstructed, hash_tensor)

                # Backward pass
                loss.backward()
                optimizers['entropy_opt'].step()
                entropy_loss = loss.item()
                losses['entropy'].append(entropy_loss)

            # 4. Distance network training
            # Skip for now as it's less critical but keep the model in the pipeline for future use

            # Update progress
            best_hamming = min([stats["best"] for stats in target_stats.values()])        while step < max_steps:
            # Get priority targets
            if step % 100 == 0 and step > 0:
                priority_targets = get_priority_targets(targets, top_n=min(10, len(targets)),
                                                      strategy=research_mode)

            # Generate batch of candidate keys
            if random.random() < 0.7:  # 70% of the time use intelligent generation
                # Generate intelligently
                seed_keys = []
                for _ in range(batch_size // 2):
                    # Select a random seed from previous successes when available
                    if any(target_stats[addr]["best_k"] > 0 for addr, _ in priority_targets):
                        addr, _ = random.choice(priority_targets)
                        if target_stats[addr]["best_k"] > 0:
                            seed_keys.append(target_stats[addr]["best_k"])
                            continue
                    # Otherwise generate random key
                    seed_keys.append(random.randint(1, MAX_K))

                # Generate keys using T5 mutation engine
                keys = []
                for seed_k in seed_keys:
                    keys.append(seed_k)  # Keep original
                    keys.append(generate_key(seed_k, models['transformer_net'], use_transformer=True))
            else:
                # Generate random keys
                keys = generate_batch_keys(batch_size)

            # Process keys to hash160
            results = process_key_batch(keys)
            batch_data = []

            # Evaluate against targets
            for k, h160 in results:
                hash_samples += 1
                h160_vec = hash160_to_vector(h160)

                # Add to batch data
                batch_data.append((k, h160, h160_vec))
                hash_vectors.append(h160_vec)

                # Limit memory usage by keeping only most recent samples
                if len(hash_vectors) > 10000:
                    hash_vectors.pop(0)

                # Check against targets - this is the core evaluation loop
                for addr, target_h160 in priority_targets:
                    hamming, flips = hamming_distance_and_flips(h160, target_h160)
                    update_target_stat(addr, k, hamming)

                    # Record improvements for learning
                    if hamming < target_stats[addr]["last_10_avg"]:
                        delta = target_stats[addr]["last_10_avg"] - hamming
                        update_bit_bias(flips, reward=delta)

                        # Log mutation event
                        log_mutation_event(
                            k_old=target_stats[addr]["best_k"] if target_stats[addr]["best_k"] > 0 else 0,
                            k_new=k,
                            mutation_type="improved",
                            flips=flips,
                            delta_hamming=delta
                        )

                        # Record XOR delta if significant improvement
                        if delta > 2 and target_stats[addr]["best_k"] > 0:
                            add_xor_delta(target_stats[addr]["best_k"], k, reward=delta)

                    # Advanced mutation memory for keys showing strong results
                    if hamming < 80:  # Strong result threshold
                        # Record successful mutations
                        if target_stats[addr]["best_k"] > 0:
                            mutation_vector = target_stats[addr]["best_k"] ^ k
                            record_mutation_result(mutation_vector, 80 - hamming)

                # Anomaly detection using entropy net
                with torch.no_grad():
                    _, anomaly_score, _ = models['entropy_net'](h160_vec.unsqueeze(0))
                    score = anomaly_score.item()
                    anomaly_scores.append(score)

                    # Detect and record anomalies
                    if score > 0.7:  # High anomaly threshold
                        anomalies_found += 1
                        log_event("anomaly_detected", {
                            "k": k,
                            "h160": h160.hex(),
                            "score": score,
                            "step": step
                        })def run_phantom_t5(targets, max_steps=100000, batch_size=32, research_mode="balanced"):
    """Main RIPEMPHANTOM-T5 research loop with neural learning & adaptive targeting"""
    print(f"\n{'='*70}")
    print(f"RIPEMPHANTOM-T5 RESEARCH ENGINE - MODE: {research_mode.upper()}")
    print(f"{'='*70}")
    print(f"Targets: {len(targets)} addresses")
    print(f"Steps: {max_steps}")
    print(f"Batch size: {batch_size}")

    # Initialize neural models
    models = {
        'dist_net': HashDistanceNet(),
        'predictor_net': HashPredictorNet(),
        'transformer_net': TransformerPredictor(),
        'entropy_net': EntropyAnalysisNet()
    }

    # Initialize optimizers with appropriate learning rates
    optimizers = {
        'dist_opt': torch.optim.Adam(models['dist_net'].parameters(), lr=0.001),
        'predictor_opt': torch.optim.Adam(models['predictor_net'].parameters(), lr=0.001),
        'transformer_opt': torch.optim.Adam(models['transformer_net'].parameters(), lr=0.0005),  # Lower LR for stability
        'entropy_opt': torch.optim.Adam(models['entropy_net'].parameters(), lr=0.001)
    }

    # Initialize target tracking
    init_target_stats(targets)

    # Load checkpoint if available
    step = load_checkpoint(models)

    # Training state
    losses = {
        'dist': [],
        'predictor': [],
        'transformer': [],
        'entropy': []
    }

    # Sample collection for analysis
    hash_vectors = []
    anomaly_scores = []

    # Neural learning loop
    priority_targets = targets
    anomalies_found = 0
    hash_samples = 0

    # Progress bar
    with tqdm(total=max_steps, initial=step) as pbar:def run_statistical_analysis(sample_size=10000):
    """Run full statistical test suite on random samples"""
    print(f"Running statistical analysis with {sample_size} samples...")

    # Generate random samples
    hash_samples = []
    bit_vectors = []

    for _ in tqdm(range(sample_size), desc="Collecting samples"):
        k = random.randint(1, MAX_K)
        h160 = privkey_to_hash160(k)
        if not h160:
            continue

        hash_samples.append(h160)
        bit_vectors.append(hash160_to_vector(h160).numpy())

    # Run statistical tests
    if len(bit_vectors) > 1000:
        print("Running NIST-based tests...")

        # Convert to bit array format for testing
        bit_array = np.stack(bit_vectors)

        # Initialize test suite
        nist_suite = NBISTTestSuite()

        # Test each bit position
        test_results = {}
        for bit_pos in tqdm(range(160), desc="Testing bit positions"):
            # Extract single bit position across all samples
            bits = bit_array[:, bit_pos].astype(int).tolist()
            test_results[bit_pos] = nist_suite.run_all_tests(bits)

        # Count failures
        failures = 0
        significant_failures = []
        for bit_pos, tests in test_results.items():
            for test_name, result in tests.items():
                if not result["passed"]:
                    failures += 1
                    significant_failures.append({
                        "bit_position": bit_pos,
                        "test": test_name,
                        "p_value": result["p_value"]
                    })

        # Report results
        print(f"Analysis complete. Found {failures} test failures across all bit positions.")

        # Save results
        with open(f"{BASE_DIR}/reports/statistical_tests.json", 'w') as f:
            json.dump({
                "timestamp": time.time(),
                "sample_size": len(bit_vectors),
                "failures": failures,
                "significant_failures": significant_failures,
                "results": {str(k): v for k, v in test_results.items()}
            }, f, indent=2)

        # Create visualization of bit position distribution
        plt.figure(figsize=(16, 6))
        bit_means = np.mean(bit_array, axis=0)
        plt.bar(range(160), bit_means, alpha=0.7)
        plt.axhline(y=0.5, color='r', linestyle='--', label='Expected (0.5)')

        # Mark significant positions
        for bit_pos in range(160):
            any_failure = any(not tests[test]["passed"] for test in test_results[bit_pos])
            if any_failure:
                plt.plot(bit_pos, bit_means[bit_pos], 'r*', markersize=10)

        plt.title("RIPEMD-160 Bit Distribution")
        plt.xlabel("Bit Position (0-159)")
        plt.ylabel("Probability of 1")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(f"{BASE_DIR}/figures/bit_distribution.png", dpi=300)
        plt.close()

        # Create correlation matrix for visual inspection
        if len(bit_vectors) > 2000:
            print("Generating bit correlation matrix...")
            corr_matrix = np.corrcoef(bit_array.T)

            plt.figure(figsize=(12, 10))
            mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
            cmap = sns.diverging_palette(230, 20, as_cmap=True)

            sns.heatmap(corr_matrix, mask=mask, cmap=cmap, vmax=0.3, vmin=-0.3,
                        center=0, square=True, linewidths=.5)

            plt.title('Bit Position Correlation Matrix')
            plt.tight_layout()
            plt.savefig(f"{BASE_DIR}/figures/bit_correlation_matrix.png", dpi=300)
            plt.close()

            # Also save as artifact for future analysis
            np.save(f"{BASE_DIR}/datasets/correlation_matrix.npy", corr_matrix)

    return test_results if 'test_results' in locals() else Noneclass NBISTTestSuite:
    """
    Advanced statistical test suite for cryptographic randomness assessment
    Based on NIST Statistical Test Suite framework but optimized for hash160
    """

    def __init__(self):
        self.tests = {
            "frequency": self.frequency_test,
            "runs": self.runs_test,
            "longest_run": self.longest_run_test,
            "serial": self.serial_test,
            "entropy": self.approximate_entropy_test
        }

    def frequency_test(self, bits):
        """Basic frequency (monobit) test"""
        n = len(bits)
        s = sum([(2*bit-1) for bit in bits])
        s_obs = abs(s) / np.sqrt(n)
        p_value = math.erfc(s_obs / np.sqrt(2))
        return p_value

    def runs_test(self, bits):
        """Runs test (frequency of runs)"""
        n = len(bits)
        if n < 100:
            return 1.0  # Not enough bits for meaningful test

        prop = sum(bits) / n
        tau = 2.0 / np.sqrt(n)

        if abs(prop - 0.5) >= tau:
            return 0.0  # Frequency test must be passed first

        # Count runs
        runs = 1
        for i in range(1, n):
            if bits[i] != bits[i-1]:
                runs += 1

        p_value = math.erfc(abs(runs - 2*n*prop*(1-prop)) /
                          (2*np.sqrt(2*n)*prop*(1-prop)))
        return p_value

    def longest_run_test(self, bits):
        """Test for longest run of ones"""
        n = len(bits)
        if n < 128:
            return 1.0  # Not enough bits

        # Simplified parameters for testing
        k, m = 3, 8
        v = [1, 2, 3, 4]
        pi = [0.2148, 0.3672, 0.2305, 0.1875]

        # Split into blocks and find longest run in each block
        num_blocks = n // m
        longest_runs = []

        for i in range(num_blocks):
            block = bits[i*m:(i+1)*m]

            # Find longest run of ones
            max_run = 0
            current_run = 0
            for bit in block:
                if bit == 1:
                    current_run += 1
                    max_run = max(max_run, current_run)
                else:
                    current_run = 0

            longest_runs.append(max_run)

        # Count frequencies
        frequencies = [0] * (k+1)
        for run in longest_runs:
            if run <= v[0]:
                frequencies[0] += 1
            elif run >= v[-1]:
                frequencies[-1] += 1
            else:
                for j in range(1, len(v)):
                    if v[j-1] < run <= v[j]:
                        frequencies[j] += 1
                        break

        # Calculate chi-square
        chi_squared = sum([(frequencies[i] - num_blocks*pi[i])**2 / (num_blocks*pi[i])
                          for i in range(len(frequencies))])

        p_value = stats.chi2.sf(chi_squared, k)
        return p_value

    def serial_test(self, bits, m=3):
        """Serial test (uniformity of m-bit patterns)"""
        n = len(bits)
        if n < 2**m:
            return 1.0  # Not enough bits

        # Extend bits for wrapping
        extended_bits = bits + bits[:m-1]

        # Count m-bit patterns
        pattern_counts1 = defaultdict(int)
        for i in range(n):
            pattern = tuple(extended_bits[i:i+m])
            pattern_counts1[pattern] += 1

        # Count (m-1)-bit patterns
        pattern_counts2 = defaultdict(int)
        for i in range(n):
            pattern = tuple(extended_bits[i:i+m-1])
            pattern_counts2[pattern] += 1

        # Calculate test statistics
        psi_sq_m = sum([count**2 for count in pattern_counts1.values()]) * 2**m / n - n
        psi_sq_m1 = sum([count**2 for count in pattern_counts2.values()]) * 2**(m-1) / n - n

        delta = psi_sq_m - psi_sq_m1
        p_value = stats.chi2.sf(delta, 2**(m-1))

        return p_value

    def approximate_entropy_test(self, bits, m=2):
        """Simplified approximate entropy test"""
        n = len(bits)
        if n < 100:
            return 1.0  # Not enough bits

        # Function to compute frequency of overlapping blocks
        def phi(m_val):
            # Count frequencies of m-bit patterns
            counts = defaultdict(int)
            for i in range(n):
                # Get the pattern using modulo to handle wrap-around
                pattern = tuple(bits[(i+j) % n] for j in range(m_val))
                counts[pattern] += 1

            # Compute C_m^i
            c_m = [count/n for count in counts.values()]
            return sum([p * np.log(p) for p in c_m if p > 0])

        # Compute ApEn
        apen = phi(m) - phi(m+1)
        chi_squared = 2 * n * (np.log(2) - apen)

        # Compute p-value (degrees of freedom = 2^m)
        p_value = stats.chi2.sf(chi_squared, 2**m)
        return p_value

    def run_all_tests(self, bits):
        """Run all tests and return results with confidence intervals"""
        results = {}
        for test_name, test_func in self.tests.items():
            p_value = test_func(bits)
            passed = p_value >= 0.01  # Standard threshold

            results[test_name] = {
                "p_value": p_value,
                "passed": passed,
            }

        return resultsdef analyze_target_statistics():
    """Generate statistical analysis of target performance"""
    if not target_stats:
        return {}

    analysis = {
        "global": {
            "best_hamming": min([s["best"] for s in target_stats.values()]),
            "avg_hamming": sum([s["best"] for s in target_stats.values()]) / len(target_stats),
            "total_scans": sum([s["scans"] for s in target_stats.values()]),
            "timestamp": time.time()
        },
        "targets": {}
    }

    for addr, stats in target_stats.items():
        if not stats["hamming_history"]:
            continue

        # Calculate statistical metrics
        history = stats["hamming_history"]

        analysis["targets"][addr] = {
            "best": stats["best"],
            "mean": float(np.mean(history)),
            "median": float(np.median(history)),
            "std_dev": float(np.std(history)),
            "improvement_rate": (160 - stats["best"]) / max(1, stats["scans"]),
            "scans": stats["scans"]
        }

        # Calculate trend (negative means improving)
        if len(history) > 10:
            try:
                x = np.arange(len(history))
                slope, _, r_value, p_value, _ = stats.linregress(x, history)
                analysis["targets"][addr].update({
                    "trend_slope": float(slope),
                    "trend_r_squared": float(r_value**2),
                    "trend_p_value": float(p_value)
                })
            except Exception as e:
                pass  # Skip trend analysis if statistical error

    # Save analysis
    with open(f"{BASE_DIR}/reports/target_analysis.json", 'w') as f:
        json.dump(analysis, f, indent=2)

    return analysis    # Plot results
    plt.figure(figsize=(12, 10))

    # Create scatter plot with cluster colors
    scatter = plt.scatter(reduced_data[:, 0], reduced_data[:, 1], c=clusters,
                        cmap='viridis', s=50, alpha=0.7)

    # Add legend for clusters
    legend1 = plt.legend(*scatter.legend_elements(), title="Clusters")
    plt.gca().add_artist(legend1)

    plt.title(f'Hash160 Clusters ({method.upper()}) - {len(hash_vectors)} samples')
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.close()
    else:
        plt.show()

    # Save reduced data for further analysis
    if save_path:
        np_save_path = save_path.replace('.png', '.npy')
        try:
            np.save(np_save_path, {
                'reduced_data': reduced_data,
                'clusters': clusters,
                'method': method
            })
        except:
            pass

    # Return results for further analysis
    return {
        'reduced_data': reduced_data,
        'clusters': clusters,
        'n_clusters': n_clusters if 'n_clusters' in locals() else 0
    }# === RIPEMPHANTOM-T5 ===
# Advanced RIPEMD-160 Entropy Analysis & Non-Uniformity Research Platform
# Tier 5 - Self-improving adversarial-level modeling with anomaly-aligned targeting

# -- Required Packages --

import os, sys, time, json, random, hashlib, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.mplot3d import Axes3D
from collections import defaultdict
from tqdm import tqdm
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from statsmodels.stats.proportion import proportion_confint
from sklearn.cluster import DBSCAN, KMeans
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

# -- Auto-install required packages if missing --
try:
    from ecdsa import SigningKey, SECP256k1
    from Crypto.Hash import RIPEMD160
    import base58
    import umap
except ImportError:
    print("Installing required packages...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                         "base58", "pycryptodome", "ecdsa", "tqdm",
                         "torch", "matplotlib", "scipy", "pandas",
                         "seaborn", "statsmodels", "scikit-learn", "umap-learn"])
    from ecdsa import SigningKey, SECP256k1
    from Crypto.Hash import RIPEMD160
    import base58
    import umap

# Try importing Google Colab-specific modules for Drive integration
try:
    from google.colab import drive, files
    IN_COLAB = True
    print("Running in Google Colab environment")
except ImportError:
    IN_COLAB = False
    print("Running in local environment")

# -- Constants --
MAX_K = 2**61
SECP256K1_ORDER = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141
ADDR_FILE = "addresses.txt"
EXPECTED_PROB = 0.5  # Expected probability for uniform distribution
CONFIG = {
    "timestamp": time.strftime("%Y-%m-%d_%H-%M-%S"),
    "rng_seed": int(time.time()),
    "max_k": MAX_K,
    "version": "RIPEMPHANTOM-T5"
}

# -- Setup environment and paths --
def setup_environment():
    """Set up workspace and paths"""
    global BASE_DIR, CHECKPOINT_PATH, LOG_PATH, REPORT_PATH

    if IN_COLAB:
        # Mount Google Drive if in Colab
        drive.mount('/content/drive', force_remount=True)
        BASE_DIR = "/content/drive/MyDrive/RIPEMPHANTOM-T5"
    else:
        BASE_DIR = "./RIPEMPHANTOM-T5"

    # Create necessary directories
    os.makedirs(BASE_DIR, exist_ok=True)
    for subdir in ["checkpoints", "logs", "figures", "reports", "datasets"]:
        os.makedirs(f"{BASE_DIR}/{subdir}", exist_ok=True)

    # Define paths
    CHECKPOINT_PATH = f"{BASE_DIR}/checkpoints"
    LOG_PATH = f"{BASE_DIR}/logs/telemetry.jsonl"
    REPORT_PATH = f"{BASE_DIR}/reports"

    # Save experiment configuration
    with open(f"{BASE_DIR}/experiment_config.json", 'w') as f:
        json.dump(CONFIG, f, indent=2)

    # Set random seeds for reproducibility
    torch.manual_seed(CONFIG["rng_seed"])
    np.random.seed(CONFIG["rng_seed"])
    random.seed(CONFIG["rng_seed"])

    # Enable CUDA if available
    if torch.cuda.is_available():
        print("CUDA available - GPU acceleration enabled")
        CONFIG["cuda"] = True
    else:
        print("CUDA not available - running on CPU")
        CONFIG["cuda"] = False

    print(f"Environment set up. Working directory: {BASE_DIR}")BASE_DIR, exist_ok=True)
    for subdir in ["checkpoints", "logs", "figures", "reports", "datasets"]:
        os.makedirs(f"{BASE_DIR}/{subdir}", exist_ok=True)

    # Define paths
    CHECKPOINT_PATH = f"{BASE_DIR}/checkpoints"
    LOG_PATH = f"{BASE_DIR}/logs/telemetry.jsonl"
    REPORT_PATH = f"{BASE_DIR}/reports"

    # Save experiment configuration
    with open(f"{BASE_DIR}/experiment_config.json", 'w') as f:
        json.dump(CONFIG, f, indent=2)

    # Set random seeds for reproducibility
    torch.manual_seed(CONFIG["rng_seed"])
    np.random.seed(CONFIG["rng_seed"])
    random.seed(CONFIG["rng_seed"])

    # Enable CUDA if available
    if torch.cuda.is_available():
        print("CUDA available - GPU acceleration enabled")
        CONFIG["cuda"] = True
    else:
        print("CUDA not available - running on CPU")
        CONFIG["cuda"] = False

    print(f"Environment set up. Working directory: {BASE_DIR}")BASE_DIR, exist_ok=True)
    for subdir in ["checkpoints", "logs", "figures", "reports", "datasets"]:
        os.makedirs(f"{BASE_DIR}/{subdir}", exist_ok=True)

    # Define paths
    CHECKPOINT_PATH = f"{BASE_DIR}/checkpoints"
    LOG_PATH = f"{BASE_DIR}/logs/telemetry.jsonl"
    REPORT_PATH = f"{BASE_DIR}/reports"

    # Save experiment configuration
    with open(f"{BASE_DIR}/experiment_config.json", 'w') as f:
        json.dump(CONFIG, f, indent=2)

    # Set random seeds for reproducibility
    torch.manual_seed(CONFIG["rng_seed"])
    np.random.seed(CONFIG["rng_seed"])
    random.seed(CONFIG["rng_seed"])

    print(f"Environment set up. Working directory: {BASE_DIR}")

# -- Core Crypto Functions --
def privkey_to_hash160(k: int, compressed: bool = True) -> bytes:
    """Generate RIPEMD160(SHA256(PubKey)) from a private key integer"""
    try:
        if not (1 <= k < SECP256K1_ORDER):
            return None
        sk = SigningKey.from_secret_exponent(k, curve=SECP256k1)
        vk = sk.get_verifying_key()
        prefix = b'\x02' if vk.to_string()[32] % 2 == 0 else b'\x03'
        pubkey = prefix + vk.to_string()[:32] if compressed else b'\x04' + vk.to_string()
        sha = hashlib.sha256(pubkey).digest()
        ripemd = RIPEMD160.new(); ripemd.update(sha)
        return ripemd.digest()
    except Exception:
        return None

def hamming_distance_and_flips(hash1: bytes, hash2: bytes) -> tuple:
    """Returns Hamming distance and list of differing bit positions"""
    distance = 0
    flips = []
    for i in range(len(hash1)):
        xor = hash1[i] ^ hash2[i]
        for j in range(8):
            if xor & (1 << j):
                bit_index = (i * 8) + (7 - j)
                flips.append(bit_index)
                distance += 1
    return distance, flips

def key_to_vector(k: int) -> torch.Tensor:
    """Converts private key to 256-bit binary tensor"""
    return torch.tensor([(k >> i) & 1 for i in range(255, -1, -1)], dtype=torch.float32)

def hash160_to_vector(h: bytes) -> torch.Tensor:
    """Converts hash160 (20 bytes) to 160-bit binary tensor"""
    return torch.tensor([(b >> i) & 1 for b in h for i in range(7, -1, -1)], dtype=torch.float32)

def vector_to_hash160(v: torch.Tensor) -> bytes:
    """Convert a 160-bit vector back to hash160 bytes"""
    hash_bytes = bytearray(20)
    for i in range(160):
        byte_idx = i // 8
        bit_idx = 7 - (i % 8)
        if v[i] > 0.5:  # Threshold for binary decision
            hash_bytes[byte_idx] |= (1 << bit_idx)
    return bytes(hash_bytes)

def generate_batch_keys(batch_size: int) -> list:
    """Generate a batch of private keys efficiently"""
    return [random.randint(1, MAX_K) for _ in range(batch_size)]

def process_key_batch(keys: list) -> list:
    """Process a batch of keys to hash160, filtering invalid ones"""
    results = []
    for k in keys:
        h160 = privkey_to_hash160(k)
        if h160:
            results.append((k, h160))
    return results

# -- Neural Models --
class HashDistanceNet(nn.Module):
    """Predicts the Hamming distance between predicted hash160 and a target"""
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1)  # Outputs predicted hamming
        )

    def forward(self, x):
        return self.model(x)

class HashPredictorNet(nn.Module):
    """Dense model that predicts hash160 bit pattern from private key"""
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 160),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

class TransformerPredictor(nn.Module):
    """Transformer model to infer likely hash160 patterns from keys"""
    def __init__(self, input_size=256, output_size=160, d_model=256, nhead=8, num_layers=2):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_proj = nn.Sequential(
            nn.Linear(d_model, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, output_size),
            nn.Sigmoid()
        )

    def forward(self, x):
        x_proj = self.input_proj(x).unsqueeze(1)  # (B, 1, D)
        x_encoded = self.transformer(x_proj)
        out = self.output_proj(x_encoded.squeeze(1))  # (B, output_size)
        return out

class EntropyAnalysisNet(nn.Module):
    """Specialized model for detecting entropy anomalies in hash160 outputs"""
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(160, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2)
        )
        self.decoder = nn.Sequential(
            nn.Linear(64, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 160),
            nn.Sigmoid()
        )
        self.anomaly_detector = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        anomaly_score = self.anomaly_detector(encoded)
        return decoded, anomaly_score, encoded

# -- Mutation & Learning Memory Functions --

def update_bit_bias(flips: list, reward: float = 1.0):
    """Reinforce bit positions that reduced Hamming"""
    for bit in flips:
        bit_bias[bit] += reward

def add_xor_delta(prev_k: int, new_k: int, reward: float = 1.0):
    """Register a successful XOR mutation"""
    delta = prev_k ^ new_k
    xor_chain.append((delta, reward))
    if len(xor_chain) > 100:
        xor_chain.pop(0)

def log_mutation_event(k_old, k_new, mutation_type, flips, delta_hamming):
    """Record a mutation's outcome for strategy evolution"""
    entry = {
        "time": time.time(),
        "type": mutation_type,
        "delta_hamming": delta_hamming,
        "flips": flips,
        "delta_k": k_old ^ k_new,
        "k_old": k_old,
        "k_new": k_new
    }
    mutation_log.append(entry)
    if len(mutation_log) > 5000:
        mutation_log.pop(0)

def record_mutation_result(mutation_vector: int, delta_hamming: int):
    """Track how effective a mutation was at improving entropy"""
    found = False
    for i, (vec, score) in enumerate(mutation_bank):
        if vec == mutation_vector:
            mutation_bank[i] = (vec, score + delta_hamming)
            found = True
            break
    if not found:
        mutation_bank.append((mutation_vector, delta_hamming))

    # Trim
    if len(mutation_bank) > 100:
        mutation_bank.sort(key=lambda x: -x[1])
        mutation_bank[:] = mutation_bank[:100]

def get_top_bit_bias(n=10):
    """Return highest scoring bit positions"""
    return sorted(bit_bias.items(), key=lambda x: -x[1])[:n]

def get_top_xor_deltas(n=10):
    """Return XOR deltas with highest reward"""
    return sorted(xor_chain, key=lambda x: -x[1])[:n]

def get_best_mutations(n=10):
    """Return top-performing mutation vectors"""
    return sorted(mutation_bank, key=lambda x: -x[1])[:n]

# -- Adaptation & Mutation Engine --
def mutate_key(k: int) -> int:
    """Create a new candidate key using all mutation intelligence"""
    new_k = k

    # Use best mutation vectors
    for vec, score in get_best_mutations(n=5):
        if random.random() < 0.3:
            new_k ^= vec

    # Flip strong bit bias positions
    for bit, score in get_top_bit_bias(n=5):
        if random.random() < 0.3:
            new_k ^= (1 << bit)

    # XOR with high-reward deltas
    for delta, reward in get_top_xor_deltas(n=5):
        if random.random() < 0.3:
            new_k ^= delta

    # Random exploratory mutation
    if random.random() < 0.2:
        new_k ^= random.getrandbits(8)

    return new_k

def generate_key(seed_k: int, transformer_model=None, use_transformer=True) -> int:
    """Generates a new candidate key intelligently using T5 blended approach"""
    k = seed_k

    # Start with evolved mutations
    k = mutate_key(k)

    # Apply transformer-based bit predictions if available
    if use_transformer and transformer_model is not None:
        try:
            with torch.no_grad():
                k_tensor = key_to_vector(k).unsqueeze(0)
                prediction = transformer_model(k_tensor).squeeze(0)
                # Flip predicted high-value bits
                for i in range(160):
                    if prediction[i].item() > 0.95 and random.random() < 0.5:
                        k ^= (1 << (i % 160))
        except Exception as e:
            pass  # Fail silently if model isn't ready

    # Ensure within valid bounds
    if k < 1 or k >= MAX_K:
        k = random.randint(1, MAX_K - 1)

    return k

# -- Checkpoint & State Management --
def log_event(event_type: str, payload: dict):
    """Appends a structured JSONL event to the telemetry log"""
    entry = {
        "time": time.time(),
        "event": event_type,
        "data": payload
    }
    with open(LOG_PATH, 'a') as f:
        f.write(json.dumps(entry) + '\n')

def save_checkpoint(models: dict, step_count: int):
    """Persist models, state, bit bias, mutation memory, and progress"""
    # Save models
    for name, model in models.items():
        torch.save(model.state_dict(), f"{CHECKPOINT_PATH}/{name}.pt")

    # Save research state
    state = {
        "bit_bias": dict(bit_bias),
        "xor_chain": xor_chain,
        "mutation_bank": [(hex(vec), score) for vec, score in mutation_bank],
        "mutation_log": mutation_log[-1000:], # Trim to last 1000 entries
        "step": step_count,
        "timestamp": time.time(),
        "config": CONFIG
    }

    # Save compressed state file
    torch.save(state, f"{CHECKPOINT_PATH}/research_state.pt")

    # Log checkpoint event
    log_event("checkpoint", {
        "step": step_count,
        "timestamp": time.time(),
        "models": list(models.keys()),
        "top_bit_bias": get_top_bit_bias(5)
    })

    # Also save bit position bias data in separate CSV for external analysis
    if bit_bias:
        try:
            with open(f"{BASE_DIR}/datasets/bit_bias_{step_count}.csv", 'w') as f:
                f.write("bit_position,bias_value\n")
                for bit, value in sorted(bit_bias.items()):
                    f.write(f"{bit},{value}\n")
        except:
            pass

def load_checkpoint(models: dict) -> int:
    """Load prior state if available, and return last step"""
    try:
        # Try loading consolidated state file
        state_path = f"{CHECKPOINT_PATH}/research_state.pt"
        if os.path.exists(state_path):
            state = torch.load(state_path)
            bit_bias.update(state["bit_bias"])
            xor_chain.extend(state["xor_chain"])
            mutation_bank.extend([(int(vec, 16), score) for vec, score in state["mutation_bank"]])

            # Load mutation log if available
            if "mutation_log" in state:
                mutation_log.extend(state["mutation_log"])

            step = state["step"]

            # Update config with loaded config
            if "config" in state:
                CONFIG.update(state["config"])

            print(f"Loaded consolidated state file from step {step}")

            # Load model states
            for name, model in models.items():
                model_path = f"{CHECKPOINT_PATH}/{name}.pt"
                if os.path.exists(model_path):
                    model.load_state_dict(torch.load(model_path))
                    print(f"Loaded model: {name}")

            log_event("checkpoint_loaded", {"step": step, "timestamp": time.time()})
            return step

    except Exception as e:
        print(f"No checkpoint loaded: {e}")

    return 0  # Start from 0 if no checkpoint

# -- Target & Address Handling Functions -- Handling --
def load_addresses(path: str) -> list:
    """Reads address list and returns tuples (address_str, hash160_bytes)"""
    targets = []
    with open(path, 'r') as f:
        for line in f:
            addr = line.strip()
            try:
                decoded = base58.b58decode(addr)
                h160 = decoded[1:-4]
                if len(h160) == 20:
                    targets.append((addr, h160))
            except:
                continue
    return targets

target_stats = {}  # addr_str: { best: int, best_k: int, scans: int, hamming_history: [] }

def init_target_stats(targets):
    """Initialize tracking statistics with enhanced monitoring"""
    for addr, _ in targets:
        target_stats[addr] = {
            "best": 160,
            "best_k": -1,
            "scans": 0,
            "hamming_history": [],
            "last_improvement": time.time(),
            "last_10_avg": 160
        }

def update_target_stat(addr: str, k: int, ham: int):
    """Update target statistics with enhanced metrics"""
    # Record history (limited to last 1000 points)
    if len(target_stats[addr]["hamming_history"]) >= 1000:
        target_stats[addr]["hamming_history"].pop(0)
    target_stats[addr]["hamming_history"].append(ham)

    # Update best score if improved
    if ham < target_stats[addr]["best"]:
        target_stats[addr]["best"] = ham
        target_stats[addr]["best_k"] = k
        target_stats[addr]["last_improvement"] = time.time()
        log_event("improvement", {
            "address": addr,
            "ham": ham,
            "k": k,
            "previous_best": target_stats[addr].get("best", 160)
        })

    # Update running statistics
    target_stats[addr]["scans"] += 1

    # Calculate moving average of last 10 attempts
    history = target_stats[addr]["hamming_history"]
    target_stats[addr]["last_10_avg"] = sum(history[-10:]) / min(10, len(history))

def get_priority_targets(targets, top_n=10, strategy="balanced"):
    """Return top-N most promising addresses based on advanced prioritization strategies"""
    if strategy == "balanced":
        # Balance between low Hamming and recent improvement
        sorted_targets = sorted(
            target_stats.items(),
            key=lambda x: (x[1]["best"], -x[1]["last_improvement"])
        )
    elif strategy == "momentum":
        # Prioritize addresses showing recent improvement trends
        sorted_targets = sorted(
            target_stats.items(),
            key=lambda x: (x[1]["best"] - x[1]["last_10_avg"], -x[1]["last_improvement"])
        )
    elif strategy == "exploration":
        # Prioritize less-scanned addresses
        sorted_targets = sorted(
            target_stats.items(),
            key=lambda x: (x[1]["scans"], x[1]["best"])
        )
    elif strategy == "exploitation":
        # Focus purely on lowest Hamming distances
        sorted_targets = sorted(
            target_stats.items(),
            key=lambda x: x[1]["best"]
        )
    else:  # Default to balanced
        sorted_targets = sorted(
            target_stats.items(),
            key=lambda x: (x[1]["best"], -x[1]["scans"])
        )

    # Get unique addresses from top targets
    top_addrs = set([addr for addr, _ in sorted_targets[:top_n]])
    return [(addr, h) for addr, h in targets if addr in top_addrs]

# -- Visualization & Analysis --
def save_bit_bias_heatmap():
    """Visualizes which bit positions are being reinforced with confidence intervals"""
    if not bit_bias:
        return

    # Calculate confidence intervals
    total_observations = sum(bit_bias.values()) or 1
    result = {}
    for bit, value in bit_bias.items():
        proportion = value / total_observations
        lower, upper = proportion_confint(value, total_observations, alpha=0.05, method='wilson')
        result[bit] = {
            "bias": value,
            "proportion": proportion,
            "ci_lower": lower,
            "ci_upper": upper,
            "significant": lower > EXPECTED_PROB or upper < EXPECTED_PROB
        }

    # Create visualization
    bits = sorted(result.keys())
    values = [result[bit]["bias"] for bit in bits]
    lower_ci = [result[bit]["ci_lower"] for bit in bits]

    plt.figure(figsize=(16, 6))
    plt.bar(bits, values, alpha=0.7)
    plt.errorbar(bits, values, yerr=[(v-l) for v, l in zip(values, lower_ci)],
                 fmt='none', ecolor='black', capsize=3)

    # Add significance indicators
    for bit in bits:
        if result[bit]["significant"]:
            plt.plot(bit, result[bit]["bias"], 'r*', markersize=10)

    plt.title("Bit Bias Heatmap with 95% Confidence Intervals")
    plt.xlabel("Bit Position (0–159)")
    plt.ylabel("Reinforcement Score")
    plt.grid(True, alpha=0.3)
    plt.axhline(y=sum(values)/len(values) if values else 0, color='r', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(f"{BASE_DIR}/figures/bit_bias_heatmap.png", dpi=300)
    plt.close()

def visualize_hash_clusters(hash_vectors, save_path=None, method='tsne'):
    """Visualize clusters in the hash space using dimensionality reduction"""
    if not hash_vectors or len(hash_vectors) < 10:
        return

    # Convert to numpy array
    if isinstance(hash_vectors[0], torch.Tensor):
        data = torch.stack(hash_vectors).numpy()
    else:
        data = np.array(hash_vectors)

    # Apply dimensionality reduction
    if method.lower() == 'tsne':
        print(f"Running t-SNE dimensionality reduction on {len(hash_vectors)} vectors...")
        reducer = TSNE(n_components=2, random_state=CONFIG["rng_seed"], perplexity=min(30, len(hash_vectors)-1))
    elif method.lower() == 'umap':
        print(f"Running UMAP dimensionality reduction on {len(hash_vectors)} vectors...")
        reducer = umap.UMAP(random_state=CONFIG["rng_seed"], min_dist=0.1, n_neighbors=min(15, len(hash_vectors)-1))
    elif method.lower() == 'pca':
        print(f"Running PCA dimensionality reduction on {len(hash_vectors)} vectors...")
        reducer = PCA(n_components=2, random_state=CONFIG["rng_seed"])
    else:
        print(f"Unknown dimensionality reduction method: {method}")
        return

    # Reduce dimensions
    try:
        reduced_data = reducer.fit_transform(data)
    except Exception as e:
        print(f"Error in dimensionality reduction: {e}")
        return

    # Try to find clusters - adjust parameters based on dataset size
    try:
        # For smaller datasets, use more lenient clustering parameters
        if len(hash_vectors) < 50:
            dbscan = DBSCAN(eps=0.8, min_samples=2)
        else:
            dbscan = DBSCAN(eps=0.5, min_samples=5)

        clusters = dbscan.fit_predict(reduced_data)

        # Count number of actual clusters (excluding noise points which are labeled -1)
        n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
        n_noise = list(clusters).count(-1)

        print(f"DBSCAN clustering found {n_clusters} clusters and {n_noise} noise points")
    except Exception as e:
        print(f"Error in clustering: {e}")
        clusters = np.zeros(len(reduced_data))


    # Plot results
    plt.figure(figsize=(12, 10))

    # Create scatter plot with cluster colors
    scatter = plt.scatter(reduced_data[:, 0], reduced_data[:, 1], c=clusters,
                        cmap='viridis', s=50, alpha=0.7)

    # Add legend for clusters
    legend1 = plt.legend(*scatter.legend_elements(), title="Clusters")
    plt.gca().add_artist(legend1)

    plt.title(f'Hash160 Clusters ({method.upper()}) - {len(hash_vectors)} samples')
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.close()
    else:
        plt.show()

    # Save reduced data for further analysis
    if save_path:
        np_save_path = save_path.replace('.png', '.npy')
        try:
            np.save(np_save_path, {
                'reduced_data': reduced_data,
                'clusters': clusters,
                'method': method
            })
        except:
            pass

    # Return results for further analysis
    return {
        'reduced_data': reduced_data,
        'clusters': clusters,
        'n_clusters': n_clusters if 'n_clusters' in locals() else 0
    }

def analyze_target_stats():
    """Generate statistical analysis of target performance"""
    if not target_stats:
        return {}

    analysis = {
        "global": {
            "best_hamming": min([s["best"] for s in target_stats.values()]),
            "avg_hamming": sum([s["best"] for s in target_stats.values()]) / len(target_stats),
            "total_scans": sum([s["scans"] for s in target_stats.values()]),
            "timestamp": time.time()
        },
        "targets": {}
    }

    for addr, stats in target_stats.items():
        if not stats["hamming_history"]:
            continue

        # Calculate statistical metrics
        history = stats["hamming_history"]

        analysis["targets"][addr] = {
            "best": stats["best"],
            "mean": float(np.mean(history)),
            "median": float(np.median(history)),
            "std_dev": float(np.std(history)),
            "improvement_rate": (160 - stats["best"]) / max(1, stats["scans"]),
            "scans": stats["scans"]
        }

        # Calculate trend (negative means improving)
        if len(history) > 10:
            try:
                x = np.arange(len(history))
                slope, _, r_value, p_value, _ = stats.linregress(x, history)
                analysis["targets"][addr].update({
                    "trend_slope": float(slope),
                    "trend_r_squared": float(r_value**2),
                    "trend_p_value": float(p_value)
                })
            except:
                pass

    # Save analysis
    with open(f"{BASE_DIR}/reports/target_analysis.json", 'w') as f:
        json.dump(analysis, f, indent=2)

    return analysis

# -- Main Research Engine --
def run_phantom_t5(targets, max_steps=100000, batch_size=32, research_mode="balanced"):
    """Main RIPEMPHANTOM-T5 research loop with neural learning & adaptive targeting"""
    print(f"\n{'='*70}")
    print(f"RIPEMPHANTOM-T5 RESEARCH ENGINE - MODE: {research_mode.upper()}")
    print(f"{'='*70}")
    print(f"Targets: {len(targets)} addresses")
    print(f"Steps: {max_steps}")
    print(f"Batch size: {batch_size}")

    # Initialize neural models
    models = {
        'dist_net': HashDistanceNet(),
        'predictor_net': HashPredictorNet(),
        'transformer_net': TransformerPredictor(),
        'entropy_net': EntropyAnalysisNet()
    }

    # Initialize optimizers with appropriate learning rates
    optimizers = {
        'dist_opt': torch.optim.Adam(models['dist_net'].parameters(), lr=0.001),
        'predictor_opt': torch.optim.Adam(models['predictor_net'].parameters(), lr=0.001),
        'transformer_opt': torch.optim.Adam(models['transformer_net'].parameters(), lr=0.0005),  # Lower LR for stability
        'entropy_opt': torch.optim.Adam(models['entropy_net'].parameters(), lr=0.001)
    }

    # Initialize target tracking
    init_target_stats(targets)

    # Load checkpoint if available
    step = load_checkpoint(models)

    # Training state
    losses = {
        'dist': [],
        'predictor': [],
        'transformer': [],
        'entropy': []
    }

    # Sample collection for analysis
    hash_vectors = []
    anomaly_scores = []

    # Neural learning loop
    priority_targets = targets
    anomalies_found = 0
    hash_samples = 0

    # Progress bar
    with tqdm(total=max_steps, initial=step) as pbar:
        while step < max_steps:
            # Get priority targets
            if step % 100 == 0 and step > 0:
                priority_targets = get_priority_targets(targets, top_n=min(10, len(targets)),
                                                       strategy=research_mode)

            # Generate batch of candidate keys
            if random.random() < 0.7:  # 70% of the time use intelligent generation
                # Generate intelligently
                seed_keys = []
                for _ in range(batch_size // 2):
                    # Select a random seed from previous successes when available
                    if any(target_stats[addr]["best_k"] > 0 for addr, _ in priority_targets):
                        addr, _ = random.choice(priority_targets)
                        if target_stats[addr]["best_k"] > 0:
                            seed_keys.append(target_stats[addr]["best_k"])
                            continue
                    # Otherwise generate random key
                    seed_keys.append(random.randint(1, MAX_K))

                # Generate keys using T5 mutation engine
                keys = []
                for seed_k in seed_keys:
                    keys.append(seed_k)  # Keep original
                    keys.append(generate_key(seed_k, models['transformer_net'], use_transformer=True))
            else:
                # Generate random keys
                keys = generate_batch_keys(batch_size)

            # Process keys to hash160
            results = process_key_batch(keys)
            batch_data = []

            # Evaluate against targets
            for k, h160 in results:
                hash_samples += 1
                h160_vec = hash160_to_vector(h160)

                # Add to batch data
                batch_data.append((k, h160, h160_vec))
                hash_vectors.append(h160_vec)

                # Limit memory usage by keeping only most recent samples
                if len(hash_vectors) > 10000:
                    hash_vectors.pop(0)

                # Check against targets
                for addr, target_h160 in priority_targets:
                    hamming, flips = hamming_distance_and_flips(h160, target_h160)
                    update_target_stat(addr, k, hamming)

                    # Record improvements for learning
                    if hamming < target_stats[addr]["last_10_avg"]:
                        delta = target_stats[addr]["last_10_avg"] - hamming
                        update_bit_bias(flips, reward=delta)

                    # Advanced mutation memory for keys showing strong results
                    if hamming < 80:  # Strong result threshold
                        # Record successful mutations
                        if target_stats[addr]["best_k"] > 0:
                            mutation_vector = target_stats[addr]["best_k"] ^ k
                            record_mutation_result(mutation_vector, 80 - hamming)

                # Anomaly detection using entropy net
                with torch.no_grad():
                    _, anomaly_score, _ = models['entropy_net'](h160_vec.unsqueeze(0))
                    score = anomaly_score.item()
                    anomaly_scores.append(score)

                    # Detect and record anomalies
                    if score > 0.7:  # High anomaly threshold
                        anomalies_found += 1
                        log_event("anomaly_detected", {
                            "k": k,
                            "h160": h160.hex(),
                            "score": score,
                            "step": step
                        })

            # -- Train neural models --

            # 1. Hash predictor training
            predictor_loss = 0
            if step % 2 == 0 and len(batch_data) >= 4:  # Train every other batch
                keys_tensor = torch.stack([key_to_vector(k) for k, _, _ in batch_data])
                hash_tensor = torch.stack([h for _, _, h in batch_data])

                # Forward pass
                optimizers['predictor_opt'].zero_grad()
                pred = models['predictor_net'](keys_tensor)
                loss = F.binary_cross_entropy(pred, hash_tensor)

                # Backward pass
                loss.backward()
                optimizers['predictor_opt'].step()
                predictor_loss = loss.item()
                losses['predictor'].append(predictor_loss)

            # 2. Transformer predictor training
            transformer_loss = 0
            if step % 3 == 0 and len(batch_data) >= 4:  # Specialized training schedule
                keys_tensor = torch.stack([key_to_vector(k) for k, _, _ in batch_data])
                hash_tensor = torch.stack([h for _, _, h in batch_data])

                # Forward pass
                optimizers['transformer_opt'].zero_grad()
                pred = models['transformer_net'](keys_tensor)
                loss = F.binary_cross_entropy(pred, hash_tensor)

                # Add L2 regularization to prevent overfitting
                l2_lambda = 0.0001
                l2_reg = torch.tensor(0., requires_grad=True)
                for param in models['transformer_net'].parameters():
                    l2_reg = l2_reg + torch.norm(param, 2)
                loss = loss + l2_lambda * l2_reg

                # Backward pass
                loss.backward()

                # Gradient clipping for stability
                torch.nn.utils.clip_grad_norm_(models['transformer_net'].parameters(), max_norm=1.0)

                optimizers['transformer_opt'].step()
                transformer_loss = loss.item()
                losses['transformer'].append(transformer_loss)

            # 3. Entropy network training (autoencoder)
            entropy_loss = 0
            if step % 2 == 1 and len(batch_data) >= 4:  # Alternate with predictor
                hash_tensor = torch.stack([h for _, _, h in batch_data])

                # Forward pass
                optimizers['entropy_opt'].zero_grad()
                reconstructed, anomaly_score, encoded = models['entropy_net'](hash_tensor)

                # Reconstruction loss (encourages learning normal patterns)
                loss = F.mse_loss(reconstructed, hash_tensor)

                # Backward pass
                loss.backward()
                optimizers['entropy_opt'].step()
                entropy_loss = loss.item()
                losses['entropy'].append(entropy_loss)

            # 4. Distance network training
            # Skip for now as it's less critical but keep the model in the pipeline for future use

            # Update progress
            best_hamming = min([stats["best"] for stats in target_stats.values()])

            # Update progress bar
            pbar.set_postfix({
                'best': best_hamming,
                'p_loss': f'{predictor_loss:.4f}',
                't_loss': f'{transformer_loss:.4f}',
                'anomalies': anomalies_found
            })

            # Visualization and checkpointing with adaptive scheduling
            if step % 1000 == 0 or (step < 1000 and step % 100 == 0):
                # More frequent checkpoints in early stages
                save_checkpoint(models, step)
                analyze_target_stats()

                # Visualize bit bias patterns
                save_bit_bias_heatmap()

                # Cluster visualization
                if len(hash_vectors) > 100:
                    # Alternate between different visualization methods
                    viz_method = 'tsne' if step % 2000 == 0 else 'umap'
                    visualize_hash_clusters(
                        hash_vectors[-500:],
                        save_path=f"{BASE_DIR}/figures/hash_clusters_{step}.png",
                        method=viz_method
                    )

            # Log progress to file
            if step % 100 == 0:
                log_event("progress", {
                    "step": step,
                    "best_hamming": best_hamming,
                    "anomalies": anomalies_found,
                    "transformer_loss": transformer_loss,
                    "predictor_loss": predictor_loss,
                    "timestamp": time.time()
                })

            step += 1
            pbar.update(1)

    # Final analysis and reporting
    final_stats = analyze_target_stats()
    save_bit_bias_heatmap()

    if len(hash_vectors) > 100:
        visualize_hash_clusters(hash_vectors[-1000:],
                             save_path=f"{BASE_DIR}/figures/final_hash_clusters.png")

    # Save final models
    save_checkpoint(models, step)

    # Print summary
    print("\nRIPEMPHANTOM-T5 Research Complete!")
    print(f"{'='*50}")
    print(f"Best Hamming distance: {final_stats['global']['best_hamming']}")
    print(f"Total keys analyzed: {hash_samples}")
    print(f"Anomalies detected: {anomalies_found}")
    print(f"{'='*50}")

    # Display best results for each target
    print("\nBest results per target:")
    for addr, stats in target_stats.items():
        if stats["best"] < 160:
            addr_short = addr[:6] + "..." + addr[-6:]
            print(f"{addr_short}: Hamming={stats['best']}, Key={hex(stats['best_k'])}")

    return {
        'best_hamming': final_stats['global']['best_hamming'],
        'hash_samples': hash_samples,
        'anomalies_found': anomalies_found,
        'target_stats': target_stats
    }

# -- Statistical Test Suite --
class NBISTTestSuite:
    """Advanced statistical test suite for cryptographic randomness assessment"""

    def __init__(self):
        self.tests = {
            "frequency": self.frequency_test,
            "runs": self.runs_test,
            "longest_run": self.longest_run_test,
            "serial": self.serial_test,
            "entropy": self.approximate_entropy_test
        }

    def frequency_test(self, bits):
        """Basic frequency (monobit) test"""
        n = len(bits)
        s = sum([(2*bit-1) for bit in bits])
        s_obs = abs(s) / np.sqrt(n)
        p_value = math.erfc(s_obs / np.sqrt(2))
        return p_value

    def runs_test(self, bits):
        """Runs test (frequency of runs)"""
        n = len(bits)
        if n < 100:
            return 1.0  # Not enough bits for meaningful test

        prop = sum(bits) / n
        tau = 2.0 / np.sqrt(n)

        if abs(prop - 0.5) >= tau:
            return 0.0  # Frequency test must be passed first

        # Count runs
        runs = 1
        for i in range(1, n):
            if bits[i] != bits[i-1]:
                runs += 1

        p_value = math.erfc(abs(runs - 2*n*prop*(1-prop)) /
                          (2*np.sqrt(2*n)*prop*(1-prop)))
        return p_value

    def longest_run_test(self, bits):
        """Test for longest run of ones"""
        n = len(bits)
        if n < 128:
            return 1.0  # Not enough bits

        # Simplified parameters for testing
        k, m = 3, 8
        v = [1, 2, 3, 4]
        pi = [0.2148, 0.3672, 0.2305, 0.1875]

        # Split into blocks and find longest run in each block
        num_blocks = n // m
        longest_runs = []

        for i in range(num_blocks):
            block = bits[i*m:(i+1)*m]

            # Find longest run of ones
            max_run = 0
            current_run = 0
            for bit in block:
                if bit == 1:
                    current_run += 1
                    max_run = max(max_run, current_run)
                else:
                    current_run = 0

            longest_runs.append(max_run)

        # Count frequencies
        frequencies = [0] * (k+1)
        for run in longest_runs:
            if run <= v[0]:
                frequencies[0] += 1
            elif run >= v[-1]:
                frequencies[-1] += 1
            else:
                for j in range(1, len(v)):
                    if v[j-1] < run <= v[j]:
                        frequencies[j] += 1
                        break

        # Calculate chi-square
        chi_squared = sum([(frequencies[i] - num_blocks*pi[i])**2 / (num_blocks*pi[i])
                          for i in range(len(frequencies))])

        p_value = stats.chi2.sf(chi_squared, k)
        return p_value

    def serial_test(self, bits, m=3):
        """Serial test (uniformity of m-bit patterns)"""
        n = len(bits)
        if n < 2**m:
            return 1.0  # Not enough bits

        # Extend bits for wrapping
        extended_bits = bits + bits[:m-1]

        # Count m-bit patterns
        pattern_counts1 = defaultdict(int)
        for i in range(n):
            pattern = tuple(extended_bits[i:i+m])
            pattern_counts1[pattern] += 1

        # Count (m-1)-bit patterns
        pattern_counts2 = defaultdict(int)
        for i in range(n):
            pattern = tuple(extended_bits[i:i+m-1])
            pattern_counts2[pattern] += 1

        # Calculate test statistics
        psi_sq_m = sum([count**2 for count in pattern_counts1.values()]) * 2**m / n - n
        psi_sq_m1 = sum([count**2 for count in pattern_counts2.values()]) * 2**(m-1) / n - n

        delta = psi_sq_m - psi_sq_m1
        p_value = stats.chi2.sf(delta, 2**(m-1))

        return p_value

    def approximate_entropy_test(self, bits, m=2):
        """Simplified approximate entropy test"""
        n = len(bits)
        if n < 100:
            return 1.0  # Not enough bits

        # Function to compute frequency of overlapping blocks
        def phi(m_val):
            # Count frequencies of m-bit patterns
            counts = defaultdict(int)
            for i in range(n):
                # Get the pattern using modulo to handle wrap-around
                pattern = tuple(bits[(i+j) % n] for j in range(m_val))
                counts[pattern] += 1

            # Compute C_m^i
            c_m = [count/n for count in counts.values()]
            return sum([p * np.log(p) for p in c_m if p > 0])

        # Compute ApEn
        apen = phi(m) - phi(m+1)
        chi_squared = 2 * n * (np.log(2) - apen)

        # Compute p-value (degrees of freedom = 2^m)
        p_value = stats.chi2.sf(chi_squared, 2**m)
        return p_value

    def run_all_tests(self, bits):
        """Run all tests and return results with confidence intervals"""
        results = {}
        for test_name, test_func in self.tests.items():
            p_value = test_func(bits)
            passed = p_value >= 0.01  # Standard threshold

            results[test_name] = {
                "p_value": p_value,
                "passed": passed,
            }

        return results

def run_statistical_analysis(sample_size=10000):
    """Run full statistical test suite on random samples"""
    print(f"Running statistical analysis with {sample_size} samples...")

    # Generate random samples
    hash_samples = []
    bit_vectors = []

    for _ in tqdm(range(sample_size), desc="Collecting samples"):
        k = random.randint(1, MAX_K)
        h160 = privkey_to_hash160(k)
        if not h160:
            continue

        hash_samples.append(h160)
        bit_vectors.append(hash160_to_vector(h160).numpy())

    # Run statistical tests
    if len(bit_vectors) > 1000:
        print("Running NIST-based tests...")

        # Convert to bit array format for testing
        bit_array = np.stack(bit_vectors)

        # Initialize test suite
        nist_suite = NBISTTestSuite()

        # Test each bit position
        test_results = {}
        for bit_pos in tqdm(range(160), desc="Testing bit positions"):
            # Extract single bit position across all samples
            bits = bit_array[:, bit_pos].astype(int).tolist()
            test_results[bit_pos] = nist_suite.run_all_tests(bits)

        # Count failures
        failures = 0
        significant_failures = []
        for bit_pos, tests in test_results.items():
            for test_name, result in tests.items():
                if not result["passed"]:
                    failures += 1
                    significant_failures.append({
                        "bit_position": bit_pos,
                        "test": test_name,
                        "p_value": result["p_value"]
                    })

        # Report results
        print(f"Analysis complete. Found {failures} test failures across all bit positions.")

        # Save results
        with open(f"{BASE_DIR}/reports/statistical_tests.json", 'w') as f:
            json.dump({
                "timestamp": time.time(),
                "sample_size": len(bit_vectors),
                "failures": failures,
                "significant_failures": significant_failures,
                "results": {str(k): v for k, v in test_results.items()}
            }, f, indent=2)

        # Create visualization of bit position distribution
        plt.figure(figsize=(16, 6))
        bit_means = np.mean(bit_array, axis=0)
        plt.bar(range(160), bit_means, alpha=0.7)
        plt.axhline(y=0.5, color='r', linestyle='--', label='Expected (0.5)')

        # Mark significant positions
        for bit_pos in range(160):
            any_failure = any(not tests[test]["passed"] for test in test_results[bit_pos])
            if any_failure:
                plt.plot(bit_pos, bit_means[bit_pos], 'r*', markersize=10)

        plt.title("RIPEMD-160 Bit Distribution")
        plt.xlabel("Bit Position (0-159)")
        plt.ylabel("Probability of 1")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(f"{BASE_DIR}/figures/bit_distribution.png", dpi=300)
        plt.close()

        # Create correlation matrix for visual inspection
        if len(bit_vectors) > 2000:
            print("Generating bit correlation matrix...")
            corr_matrix = np.corrcoef(bit_array.T)

            plt.figure(figsize=(12, 10))
            mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
            cmap = sns.diverging_palette(230, 20, as_cmap=True)

            sns.heatmap(corr_matrix, mask=mask, cmap=cmap, vmax=0.3, vmin=-0.3,
                        center=0, square=True, linewidths=.5)

            plt.title('Bit Position Correlation Matrix')
            plt.tight_layout()
            plt.savefig(f"{BASE_DIR}/figures/bit_correlation_matrix.png", dpi=300)
            plt.close()

            # Also save as artifact for future analysis
            np.save(f"{BASE_DIR}/datasets/correlation_matrix.npy", corr_matrix)

    return test_results if 'test_results' in locals() else None

# -- Main Entry Point --
def main():
    """Main entry point for RIPEMPHANTOM-T5"""
    # Handle command line arguments if provided
    import argparse
    parser = argparse.ArgumentParser(description='RIPEMPHANTOM-T5 Entropy Analysis Tool')
    parser.add_argument('--mode', type=int, choices=[1, 2, 3, 4], default=1,
                        help='Research mode: 1=Balanced, 2=Entropy, 3=Cluster, 4=Statistical')
    parser.add_argument('--steps', type=int, default=100000,
                        help='Maximum number of steps to run')
    parser.add_argument('--samples', type=int, default=10000,
                        help='Sample size for statistical analysis')
    parser.add_argument('--file', type=str, default=ADDR_FILE,
                        help='Path to addresses.txt file')

    # Parse args if available, otherwise use interactive mode
    if len(sys.argv) > 1:
        args = parser.parse_args()
        mode_choice = args.mode
        max_steps = args.steps
        sample_size = args.samples
        ADDR_FILE = args.file

        # Map choice to mode
        mode_map = {
            1: "balanced",
            2: "entropy",
            3: "cluster",
            4: "statistical"
        }

        research_mode = mode_map[mode_choice]
        print(f"Command-line mode: {research_mode}, Steps: {max_steps}, File: {ADDR_FILE}")

        # Load addresses
        targets = load_addresses(ADDR_FILE)
        if not targets:
            print("No valid addresses found in file.")
            return

        print(f"Loaded {len(targets)} target addresses.")

        # Run appropriate mode
        if research_mode == "statistical":
            run_statistical_analysis(sample_size)
        else:
            run_phantom_t5(targets, max_steps=max_steps, research_mode=research_mode)

        return

    # Interactive mode
    print_banner()

    # Check for addresses file
    if os.path.exists(ADDR_FILE):
        print(f"Found addresses file: {ADDR_FILE}")
    else:
        print("No addresses.txt found. Upload or create one.")
        if IN_COLAB:
            print("Waiting for file upload...")
            uploaded = files.upload()
            if not uploaded:
                print("No file uploaded. Exiting.")
                return
        else:
            return

    # Load addresses
    targets = load_addresses(ADDR_FILE)
    if not targets:
        print("No valid addresses found in file.")
        return

    print(f"Loaded {len(targets)} target addresses.")

    # Prompt for research mode
    print("\nSelect research mode:")
    print("1. Balanced - General research with equal focus on all aspects")
    print("2. Entropy - Focus on statistical entropy analysis and anomaly detection")
    print("3. Cluster - Focus on bit correlation and clustering patterns")
    print("4. Statistical - Run only statistical test suite")

    try:
        mode_choice = int(input("Enter choice (1-4) [default=1]: ") or "1")
        if mode_choice not in range(1, 5):
            mode_choice = 1
    except:
        mode_choice = 1

    # Map choice to mode
    mode_map = {
        1: "balanced",
        2: "entropy",
        3: "cluster",
        4: "statistical"
    }

    research_mode = mode_map[mode_choice]
    print(f"Selected mode: {research_mode}")

    # For statistical mode, run only statistical analysis
    if research_mode == "statistical":
        try:
            sample_size = int(input("Enter sample size for statistical analysis [default=10000]: ") or "10000")
        except:
            sample_size = 10000

        run_statistical_analysis(sample_size)
        return

    # For normal research modes, get steps
    try:
        max_steps = int(input("Enter maximum steps to run [default=100000]: ") or "100000")
    except:
        max_steps = 100000

    # Run main research
    print(f"\nStarting RIPEMPHANTOM-T5 with {len(targets)} targets, {max_steps} steps, mode: {research_mode}")
    results = run_phantom_t5(targets, max_steps=max_steps, research_mode=research_mode)

    # Print final results
    print("\nResearch complete!")
    print(f"Best Hamming distance: {results['best_hamming']}")
    print(f"Keys analyzed: {results['hash_samples']}")
    print(f"Anomalies found: {results['anomalies_found']}")

    # Export results
    with open(f"{BASE_DIR}/reports/final_results.json", 'w') as f:
        # Convert non-serializable objects to serializable form
        serializable_stats = {}
        for addr, stats in results['target_stats'].items():
            serializable_stats[addr] = {
                'best': stats['best'],
                'best_k': stats['best_k'],
                'scans': stats['scans'],
                'last_10_avg': stats['last_10_avg'],
                'history_length': len(stats['hamming_history'])
            }

        final_results = {
            'best_hamming': results['best_hamming'],
            'hash_samples': results['hash_samples'],
            'anomalies_found': results['anomalies_found'],
            'target_stats': serializable_stats,
            'timestamp': time.time(),
            'config': CONFIG
        }

        json.dump(final_results, f, indent=2)

    print(f"Results saved to {BASE_DIR}/reports/final_results.json")

def print_banner():
    """Display program banner and version info"""
    banner = """
██████╗ ██╗██████╗ ███████╗███╗   ███╗██████╗ ██╗  ██╗ █████╗ ███╗   ██╗████████╗ ██████╗ ███╗   ███╗
██╔══██╗██║██╔══██╗██╔════╝████╗ ████║██╔══██╗██║  ██║██╔══██╗████╗  ██║╚══██╔══╝██╔═══██╗████╗ ████║
██████╔╝██║██████╔╝█████╗  ██╔████╔██║██████╔╝███████║███████║██╔██╗ ██║   ██║   ██║   ██║██╔████╔██║
██╔══██╗██║██╔═══╝ ██╔══╝  ██║╚██╔╝██║██╔═══╝ ██╔══██║██╔══██║██║╚██╗██║   ██║   ██║   ██║██║╚██╔╝██║
██║  ██║██║██║     ███████╗██║ ╚═╝ ██║██║     ██║  ██║██║  ██║██║ ╚████║   ██║   ╚██████╔╝██║ ╚═╝ ██║
╚═╝  ╚═╝╚═╝╚═╝     ╚══════╝╚═╝     ╚═╝╚═╝     ╚═╝  ╚═╝╚═╝  ╚═╝╚═╝  ╚═══╝   ╚═╝    ╚═════╝ ╚═╝     ╚═╝
╔════════════════════════════════════════════════════════════════════════════════════════════════════╗
║                  RIPEMD-160 ENTROPY ANALYSIS & NON-UNIFORMITY RESEARCH PLATFORM                    ║
║                       T5 EDITION - Nation-State Threat Level Research Tool                         ║
╚════════════════════════════════════════════════════════════════════════════════════════════════════╝
    """
    print(banner)
    print(f"Version: {CONFIG['version']} | Timestamp: {CONFIG['timestamp']}")
    print(f"{'='*89}")╚██████╔╝██║ ╚═╝ ██║
╚═╝  ╚═╝╚═╝╚═╝     ╚══════╝╚═╝     ╚═╝╚═╝     ╚═╝  ╚═╝╚═╝  ╚═╝╚═╝  ╚═══╝   ╚═╝    ╚═════╝ ╚═╝     ╚═╝
╔════════════════════════════════════════════════════════════════════════════════════════════════════╗
║                  RIPEMD-160 ENTROPY ANALYSIS & NON-UNIFORMITY RESEARCH PLATFORM                    ║
║                       T5 EDITION - Nation-State Threat Level Research Tool                         ║
╚════════════════════════════════════════════════════════════════════════════════════════════════════╝
    """
    print(banner)
    print(f"Version: {CONFIG['version']} | Timestamp: {CONFIG['timestamp']}")
    print(f"{'='*89}")╚██████╔╝██║ ╚═╝ ██║
╚═╝  ╚═╝╚═╝╚═╝     ╚══════╝╚═╝     ╚═╝╚═╝     ╚═╝  ╚═╝╚═╝  ╚═╝╚═╝  ╚═══╝   ╚═╝    ╚═════╝ ╚═╝     ╚═╝
╔════════════════════════════════════════════════════════════════════════════════════════════════════╗
║                  RIPEMD-160 ENTROPY ANALYSIS & NON-UNIFORMITY RESEARCH PLATFORM                    ║
║                       T5 EDITION - Nation-State Threat Level Research Tool                         ║
╚════════════════════════════════════════════════════════════════════════════════════════════════════╝
    """
    print(banner)
    print(f"Version: {CONFIG['version']} | Timestamp: {CONFIG['timestamp']}")
    print(f"{'='*89}")

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\nResearch interrupted by user.")
        print("Saving current state...")
        # Try to save current state if models initialized
        try:
            if 'models' in locals():
                save_checkpoint(models, step)
                print("Checkpoint saved successfully.")
        except:
            print("Could not save checkpoint.")
        print("Exiting gracefully.")
    except Exception as e:
        print(f"\nAn unexpected error occurred: {e}")
        import traceback
        traceback.print_exc()
        print("\nExiting with error.")

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 1488)